# ASCII Flux v5 — usine D1 et baseline F1

Ce notebook contient ses sources Python et fonctionne sans dépôt externe.
La v4.2 reste gelée. Les clips créés par D1 sont des candidats à revoir ; aucun
corpus validé ni poids préentraîné v5 n'est fourni.
**Tous les jalons restent à valider.** Le convertisseur travaille hors ligne ;
l'inférence NCA lit et produit directement des jetons glyphe/couleur.

Exécuter l'installation au début d'une session vierge, laisser le noyau redémarrer,
puis reprendre à la cellule suivante. Les archives YouTube-VIS doivent être obtenues
selon les [conditions officielles](https://youtube-vos.org/dataset/term/) :
annotations CC BY 4.0, données pour recherche non commerciale uniquement.
Les étapes de revue humaine ne sont pas exécutables automatiquement par « Tout exécuter ».


In [ ]:
# Installation AVANT les imports scientifiques, puis redémarrage du noyau.
import hashlib, json, os, signal, subprocess, sys
from pathlib import Path
SOURCES = {'pyproject.toml': '[build-system]\nrequires = ["setuptools>=75"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "ascii-flux"\nversion = "5.0.0.dev1"\ndescription = "Offline ASCII corpus factory and native glyph-space flow research baseline"\nrequires-python = ">=3.11"\ndependencies = ["numpy>=2,<3", "pillow>=11,<13", "opencv-python-headless>=4.11,<5"]\n\n[project.optional-dependencies]\ntrain = ["torch>=2.6,<3", "transformers>=4.57,<5", "huggingface-hub>=0.34,<1", "sentencepiece>=0.2,<1"]\nvis = ["pycocotools>=2.0.10,<3", "gdown>=5.2,<6"]\nexport = ["onnx>=1.17,<2", "onnxruntime>=1.20,<2"]\ntest = ["pytest>=8,<10"]\n\n[project.scripts]\nascii-flux = "ascii_flux.cli:main"\n\n[tool.setuptools.packages.find]\ninclude = ["ascii_flux*"]\n\n[tool.setuptools.package-data]\nascii_flux = ["*.html"]\n\n\n[tool.pytest.ini_options]\ntestpaths = ["tests"]\n\n', 'ascii_flux/__init__.py': '"""ASCII Flux v5. The offline factory is never imported by inference."""\n\n__version__ = "5.0.0.dev1"\n\n', 'ascii_flux/__main__.py': 'from .cli import main\n\nmain()\n', 'ascii_flux/acquisition.py': '"""Acquire an authorized YouTube-VIS 2019 training archive, before conversion.\n\nThe organizer portal requires a login to reveal download links. We accept those\nlinks, never manufacture mirror URLs, and never request account credentials.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nimport tarfile\nimport tempfile\nimport time\nfrom urllib.parse import urlparse\nimport zipfile\nfrom .factory import file_sha256, validate_source\nfrom .storage import atomic_file, write_json\n\n\nPORTAL = "https://codalab.lisn.upsaclay.fr/competitions/6064#participate-get_data"\nDATASET_PAGE = "https://youtube-vos.org/dataset/vis/"\nTERMS = "https://youtube-vos.org/dataset/term/"\nPAPER = "https://arxiv.org/html/1905.04804v2#S4"\nDOWNLOAD_USER_AGENT = "ASCII-Flux/5.0 (+offline research dataset acquisition)"\n\n\ndef default_data_root(project_root, content_root="/content"):\n    """Use Drive only when it is actually mounted; local notebooks stay local."""\n    content = Path(content_root)\n    if content.is_dir():\n        drive = content/"drive/MyDrive"\n        return drive/"ascii_flux" if drive.is_dir() else content/"ascii_flux_data"\n    return Path(project_root).resolve()/"data"\n\n\ndef _http_url(url):\n    parsed = urlparse(str(url).strip())\n    if parsed.scheme not in ("http", "https") or not parsed.netloc or parsed.username or parsed.password:\n        raise ValueError("Fournir un lien HTTP(S) de fichier, sans identifiants de compte.")\n    if "/folders/" in parsed.path:\n        raise ValueError("Fournir le lien du fichier, pas celui d\'un dossier Google Drive.")\n    return str(url).strip()\n\n\ndef download_file(url, destination, kind, downloader=None, expected_sha256=None):\n    """Resume in a stable partial file; publish only after content validation."""\n    url = _http_url(url)\n    destination = Path(destination)\n    receipt_path = destination.with_suffix(destination.suffix+".receipt.json")\n    if destination.exists():\n        if not receipt_path.exists():\n            raise ValueError(f"Fichier existant sans reçu de provenance : {destination}. Choisir un autre dossier.")\n        receipt = json.loads(receipt_path.read_text(encoding="utf-8"))\n        if receipt["url"] != url or receipt["sha256"] != file_sha256(destination):\n            raise ValueError(f"Archive existante incompatible avec le lien demandé : {destination}")\n        if expected_sha256 and receipt["sha256"] != expected_sha256:\n            raise ValueError("SHA-256 différent de la révision attendue")\n        return receipt\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_suffix(destination.suffix+".partial")\n    partial_source = partial.with_suffix(partial.suffix+".source.json")\n    if partial_source.exists():\n        if json.loads(partial_source.read_text(encoding="utf-8"))["url"] != url:\n            raise ValueError("Téléchargement partiel d\'une autre URL ; choisir un nouveau dossier d\'acquisition.")\n    elif partial.exists():\n        raise ValueError("Téléchargement partiel sans provenance ; choisir un nouveau dossier d\'acquisition.")\n    write_json(partial_source, {"url": url, "kind": kind})\n    if downloader is None:\n        try:\n            import gdown\n        except ImportError as exc:\n            raise ImportError("gdown manque. Relancer l\'installation initiale puis redémarrer le noyau ; aucun paquet n\'est installé ici.") from exc\n        downloader = gdown.download\n    try:\n        # gdown\'s default Chrome/39 identity receives HTML from Kaggle\'s public\n        # API. Identify this API client consistently with the verified download.\n        result = downloader(url=url, output=str(partial), quiet=False, resume=True, fuzzy=True,\n                            use_cookies=False, user_agent=DOWNLOAD_USER_AGENT)\n    except Exception as exc:\n        raise OSError("Échec du téléchargement. Vérifier le lien et les droits d\'accès au fichier : " + str(exc)) from exc\n    if not result or not partial.is_file():\n        raise OSError("Téléchargement incomplet ou accès refusé. Vérifier le lien officiel ; ne pas coller de mot de passe.")\n    # A login page, Drive quota message or confirmation HTML is not an archive.\n    try:\n        if kind == "frames":\n            if not zipfile.is_zipfile(partial) and not tarfile.is_tarfile(partial):\n                raise ValueError("Le lien images n\'a retourné ni ZIP ni TAR. Vérifier l\'accès au fichier dans le navigateur.")\n        elif not zipfile.is_zipfile(partial):\n            try:\n                data = json.loads(partial.read_text(encoding="utf-8"))\n            except (ValueError, UnicodeError) as exc:\n                raise ValueError("Le lien annotations n\'a retourné ni un JSON ni un ZIP d\'annotations.") from exc\n            _validate_annotations(data)\n    except ValueError:\n        # A completed HTML/error response cannot be resumed as archive bytes.\n        partial.unlink(missing_ok=True)\n        partial_source.unlink(missing_ok=True)\n        raise\n    receipt = {"url": url, "kind": kind, "sha256": file_sha256(partial), "bytes": partial.stat().st_size}\n    if expected_sha256 and receipt["sha256"] != expected_sha256:\n        raise ValueError("SHA-256 du téléchargement différent de la révision attendue ; archive non publiée")\n    with partial.open("r+b") as stream:\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(partial, destination)\n    write_json(receipt_path, receipt)\n    partial_source.unlink(missing_ok=True)\n    return receipt\n\n\ndef _publish_directory(staging, destination):\n    # Antivirus/indexing handles can briefly prevent renaming a completed\n    # directory on Windows. Preserve the validated staging tree if it persists.\n    for attempt in range(15):\n        try:\n            os.replace(staging, destination)\n            return\n        except PermissionError:\n            if attempt == 14:\n                raise\n            time.sleep(1)\n\n\ndef extract_zip(archive, destination, durable_files=True):\n    """Atomic extraction. Reject path traversal, links and Windows path aliases."""\n    destination = Path(destination).resolve()\n    receipt = {"archive_sha256": file_sha256(archive)}\n    marker = destination/".extracted.json"\n    if destination.exists():\n        if marker.exists() and json.loads(marker.read_text(encoding="utf-8")) == receipt:\n            return destination\n        raise FileExistsError(f"Dossier d\'extraction existant sans reçu correspondant : {destination}")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    for previous in destination.parent.glob(destination.name+".*"):\n        previous_marker = previous/".extracted.json"\n        if previous.is_dir() and previous_marker.exists() and json.loads(previous_marker.read_text(encoding="utf-8")) == receipt:\n            _publish_directory(previous, destination)\n            return destination\n    staging = Path(tempfile.mkdtemp(prefix=destination.name+".", dir=destination.parent)).resolve()\n    try:\n        with zipfile.ZipFile(archive) as zipped:\n            members = zipped.infolist()\n            for member in members:\n                name = member.filename\n                path = PurePosixPath(name)\n                if path.is_absolute() or ".." in path.parts or "\\\\" in name or ":" in name:\n                    raise ValueError("Chemin dangereux dans le ZIP")\n                if any(part.rstrip(". ") != part for part in path.parts):\n                    raise ValueError("Chemin ambigu dans le ZIP")\n                if stat.S_ISLNK(member.external_attr >> 16):\n                    raise ValueError("Lien symbolique interdit dans le ZIP")\n                target = (staging/Path(*path.parts)).resolve()\n                if not target.is_relative_to(staging):\n                    raise ValueError("Chemin hors du dossier d\'extraction")\n            for member in members:\n                target = staging/Path(*PurePosixPath(member.filename).parts)\n                if member.is_dir():\n                    target.mkdir(parents=True, exist_ok=True)\n                    continue\n                if durable_files:\n                    with zipped.open(member) as source, atomic_file(target) as stream:\n                        shutil.copyfileobj(source, stream, length=1024*1024)\n                else:\n                    # Reconstructible media cache in an unpublished private\n                    # directory. ZIP checks each CRC; the downloaded archive is\n                    # already durable. Corpus shards always use fsync separately.\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with zipped.open(member) as source, target.open("wb") as stream:\n                        shutil.copyfileobj(source, stream, length=1024*1024)\n        write_json(staging/".extracted.json", receipt)\n        if destination.exists():\n            raise FileExistsError(f"Dossier apparu pendant l\'extraction : {destination}")\n        _publish_directory(staging, destination)\n        return destination\n    except BaseException:\n        if not (staging/".extracted.json").exists() and staging.parent == destination.parent and staging.name.startswith(destination.name+"."):\n            shutil.rmtree(staging)\n        raise\n\n\ndef _validate_annotations(data):\n    if not isinstance(data, dict) or not all(isinstance(data.get(k), list) and data[k] for k in ("videos", "annotations", "categories")):\n        raise ValueError("Choisir les annotations TRAIN YouTube-VIS : videos, annotations et categories doivent être présents et non vides.")\n    if any(not isinstance(v.get("file_names"), list) or not v["file_names"] for v in data["videos"]):\n        raise ValueError("Les annotations ne contiennent pas les noms des images vidéo.")\n    if any(not isinstance(a.get("segmentations"), list) for a in data["annotations"]):\n        raise ValueError("Masques d\'instances absents du fichier d\'annotations.")\n    return data\n\n\ndef find_annotations(root):\n    root = Path(root)\n    if root.is_file():\n        return root, _validate_annotations(json.loads(root.read_text(encoding="utf-8")))\n    found = []\n    for path in sorted(root.rglob("*.json")):\n        if path.name.startswith(".") or path.stat().st_size < 100:\n            continue\n        try:\n            data = _validate_annotations(json.loads(path.read_text(encoding="utf-8")))\n            found.append((path, data))\n        except (ValueError, UnicodeError, AttributeError):\n            continue\n    preferred = [item for item in found if "train" in str(item[0].relative_to(root)).lower()]\n    found = preferred or found\n    if len(found) != 1:\n        raise ValueError(f"Trouvé {len(found)} fichiers d\'annotations utilisables. Fournir le chemin exact des annotations d\'entraînement.")\n    return found[0]\n\n\ndef find_frames_root(root, data):\n    root = Path(root).resolve()\n    first = data["videos"][0]["file_names"][0]\n    relative = PurePosixPath(first)\n    if relative.is_absolute() or ".." in relative.parts or "\\\\" in first or ":" in first:\n        raise ValueError("Chemin d\'image invalide dans les annotations")\n    matches = []\n    # Find only the first named file, then validate every listed frame under its root.\n    for candidate in root.rglob(relative.name):\n        parent = candidate\n        for _ in relative.parts:\n            parent = parent.parent\n        if (parent/Path(*relative.parts)).resolve() != candidate.resolve():\n            continue\n        if parent not in matches:\n            matches.append(parent)\n    if len(matches) != 1:\n        raise ValueError(f"Impossible de choisir la racine des images ({len(matches)} correspondances). Vérifier l\'archive train.")\n    selected = matches[0].resolve()\n    missing = []\n    count = 0\n    for video in data["videos"]:\n        for name in video["file_names"]:\n            path = (selected/name).resolve()\n            if not path.is_relative_to(selected):\n                raise ValueError("Image en dehors du dossier source")\n            if not path.is_file() and len(missing) < 5:\n                missing.append(name)\n            count += 1\n    if missing:\n        raise ValueError(f"Archive d\'images incomplète ; exemples manquants : {missing}")\n    return selected, count\n\n\ndef prepare_ytvis2019(data_root, annotations_url, frames_url, source_template, source_fps=6., downloader=None):\n    """Download -> validate -> extract -> persist exact source, paths and cadence.\n\n    6 Hz is the listed annotated-frame cadence for the 2019 release described in\n    section 4 of the paper (30 fps sampled every 5 frames), not the raw-video FPS.\n    Only filenames listed by the instance annotations are read by the converter.\n    """\n    if not math.isfinite(source_fps) or source_fps <= 0:\n        raise ValueError("Cadence des images annotées invalide")\n    source = dict(source_template)\n    source.update(snapshot="YouTube-VIS-2019/train", local_media_provenance={\n        "annotations_url": _http_url(annotations_url), "frames_url": _http_url(frames_url), "portal": PORTAL})\n    validate_source(source)\n    root = Path(data_root).resolve()/"ytvis-2019"\n    print("Source : YouTube-VIS 2019 train. Annotations CC BY 4.0 ; données pour recherche non commerciale.", flush=True)\n    print("Conditions :", TERMS, flush=True)\n    print("Téléchargement des annotations…", flush=True)\n    ann_file = root/"downloads/annotations.download"\n    ann_receipt = download_file(annotations_url, ann_file, "annotations", downloader)\n    if zipfile.is_zipfile(ann_file):\n        ann_path, data = find_annotations(extract_zip(ann_file, root/"annotations"))\n    else:\n        ann_path, data = find_annotations(ann_file)\n    print("Téléchargement des images d\'entraînement (archive volumineuse)…", flush=True)\n    frames_file = root/"downloads/train.zip"\n    frames_receipt = download_file(frames_url, frames_file, "frames", downloader)\n    extracted = extract_zip(frames_file, root/"frames")\n    print("Vérification des images listées dans les annotations…", flush=True)\n    frames_root, count = find_frames_root(extracted, data)\n    source["local_media_provenance"].update(annotations_sha256=ann_receipt["sha256"], frames_sha256=frames_receipt["sha256"])\n    source["cadence_evidence"] = {"url": PAPER, "scope": "2019 listed annotated frames; original 30 fps / stride 5", "source_fps": source_fps}\n    prepared = {"annotations": str(ann_path.resolve()), "annotations_sha256": file_sha256(ann_path),\n                "frames_root": str(frames_root), "source_fps": source_fps, "source": source,\n                "videos": len(data["videos"]), "instances": len(data["annotations"]), "listed_frames": count}\n    write_json(root/"prepared.json", prepared)\n    print(f"Prêt pour D1 : {prepared[\'videos\']} vidéos, {prepared[\'instances\']} instances, {count} images listées.", flush=True)\n    return prepared\n\n\ndef load_prepared(data_root, profile="ytvis-2019"):\n    if profile not in ("ytvis-2019", "ytvis-2019-public"):\n        raise ValueError("Profil d\'acquisition inconnu")\n    path = Path(data_root)/profile/"prepared.json"\n    if not path.exists():\n        return None\n    prepared = json.loads(path.read_text(encoding="utf-8"))\n    annotations = Path(prepared["annotations"])\n    if not annotations.is_file() or file_sha256(annotations) != prepared["annotations_sha256"]:\n        raise ValueError("Annotations déplacées ou modifiées depuis l\'acquisition. Relancer la préparation.")\n    if not Path(prepared["frames_root"]).is_dir():\n        raise ValueError("Images déplacées depuis l\'acquisition. Relancer la préparation.")\n    validate_source(prepared["source"])\n    return prepared\n', 'ascii_flux/autoregressive.py': '"""Experimental causal transformer on native glyph/color patches.\n\nPatches group existing cells for attention; no learned image codec, VAE, raster\nintermediate or corpus retrieval is involved. Both outputs remain per-cell IDs.\n"""\nfrom dataclasses import asdict,dataclass\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass CausalConfig:\n    width: int = 192\n    heads: int = 6\n    layers: int = 6\n    patch_size: int = 4\n    patches_per_step: int = 4\n    text_dim: int = 768\n    text_dropout: float = .1\n    temperature: float = .8\n    guidance: float = 1.5\n\n    def __post_init__(self):\n        if min(self.width,self.heads,self.layers,self.patch_size,self.patches_per_step,self.text_dim) < 1 or self.width % self.heads:\n            raise ValueError("Invalid causal transformer dimensions")\n        if not 0 <= self.text_dropout < 1 or self.temperature <= 0 or self.guidance < 0:\n            raise ValueError("Invalid causal sampling settings")\n\n\nclass CausalGlyphFlow(nn.Module):\n    def __init__(self,contract,config=None):\n        super().__init__()\n        self.contract,self.config = contract,config or CausalConfig()\n        cfg = self.config\n        p,d = cfg.patch_size,cfg.width\n        if contract.rows % p or contract.cols % p:\n            raise ValueError("Grid must be divisible into complete native-cell patches")\n        self.patch_count = contract.rows*contract.cols//p**2\n        self.mask_glyph,self.mask_color = len(contract.glyphs),len(contract.palette)+1\n        self.glyph_embedding = nn.Embedding(self.mask_glyph+1,16)\n        self.color_embedding = nn.Embedding(self.mask_color+1,16)\n        self.patch_projection = nn.Linear(p*p*32,d)\n        self.previous_projection = nn.Linear(p*p*32,d)\n        self.text_projection = nn.Sequential(nn.Linear(cfg.text_dim,d),nn.SiLU(),nn.Linear(d,d))\n        self.position = nn.Parameter(torch.randn(1,self.patch_count,d)*.02)\n        self.bos = nn.Parameter(torch.randn(1,1,d)*.02)\n        self.mode_embedding = nn.Embedding(3,d)\n        self.decoder = nn.TransformerDecoder(nn.TransformerDecoderLayer(d,cfg.heads,d*4,dropout=0.,\n            activation="gelu",batch_first=True,norm_first=True),cfg.layers,norm=nn.LayerNorm(d))\n        # Explicitly initialize distinct layers (TransformerDecoder clones its\n        # template, whose identical initialization is an avoidable symmetry).\n        for module in self.decoder.modules():\n            if isinstance(module,nn.Linear):\n                nn.init.xavier_uniform_(module.weight)\n                if module.bias is not None:nn.init.zeros_(module.bias)\n        self.glyph_head = nn.Linear(d,p*p*len(contract.glyphs))\n        self.color_head = nn.Linear(d,p*p*len(contract.palette))\n        self.register_buffer("causal_mask",torch.triu(torch.ones(self.patch_count,self.patch_count,dtype=torch.bool),1))\n\n    def patches(self,value):\n        b,r,c = value.shape[:3]\n        p = self.config.patch_size\n        tail = value.shape[3:]\n        value = value.reshape(b,r//p,p,c//p,p,*tail)\n        return value.permute(0,1,3,2,4,*range(5,value.ndim)).reshape(b,self.patch_count,p*p,*tail)\n\n    def unpatch(self,value):\n        b,n,a = value.shape[:3]\n        p = self.config.patch_size\n        tail = value.shape[3:]\n        value = value.reshape(b,self.contract.rows//p,self.contract.cols//p,p,p,*tail)\n        return value.permute(0,1,3,2,4,*range(5,value.ndim)).reshape(b,self.contract.rows,self.contract.cols,*tail)\n\n    def embed(self,g,c):\n        g = g.long()\n        c = torch.where(g == self.mask_glyph,self.mask_color,\n                        torch.where(g == 0,len(self.contract.palette),c)).long()\n        return self.patches(torch.cat([self.glyph_embedding(g),self.color_embedding(c)],-1)).flatten(2)\n\n    def forward(self,g,c,pg,pc,text,mode,limit=None):\n        limit = self.patch_count if limit is None else limit\n        condition = self.text_projection(text*math.sqrt(self.config.text_dim))+self.mode_embedding(mode.long())\n        target = self.patch_projection(self.embed(g,c))[:,:max(0,limit-1)]\n        target = torch.cat([self.bos.expand(len(g),-1,-1),target],1)\n        target = target+self.position[:,:limit]+condition[:,None]\n        memory = self.previous_projection(self.embed(pg,pc))+self.position\n        memory = torch.cat([condition[:,None],memory],1)\n        value = self.decoder(target,memory,tgt_mask=self.causal_mask[:limit,:limit])\n        shape = (len(g),limit,self.config.patch_size**2)\n        return (self.glyph_head(value).reshape(*shape,len(self.contract.glyphs)),\n                self.color_head(value).reshape(*shape,len(self.contract.palette)))\n\n    def initial_state(self,batch=1,noisy=False):\n        state = self.position.new_zeros((batch,6,self.contract.rows,self.contract.cols))\n        state[:,1] = self.contract.empty_color\n        state[:,2] = self.mask_glyph\n        state[:,3] = self.mask_color\n        return state\n\n    def recondition(self,state):\n        result = state.clone()\n        result[:,2:4] = result[:,:2]\n        result[:,4] = 0\n        result[:,5] = 2\n        return result\n\n    def tokens(self,state):\n        g,c = state[:,0].long(),state[:,1].long()\n        return g,torch.where(g == 0,self.contract.empty_color,c)\n\n    def step(self,state,text):\n        state = state.clone()\n        cursor = int(state[0,4,0,0].item())\n        mode = int(state[0,5,0,0].item())\n        if cursor >= self.patch_count:\n            state[:,2:4] = state[:,:2]\n            state[:,4] = 0\n            state[:,5] = 1\n            cursor,mode = 0,1\n        modes = torch.full((len(state),),mode,dtype=torch.long,device=state.device)\n        for position in range(cursor,min(cursor+self.config.patches_per_step,self.patch_count)):\n            g,c,pg,pc = (state[:,i].long() for i in range(4))\n            if self.config.guidance != 1:\n                gl,cl = self(torch.cat([g,g]),torch.cat([c,c]),torch.cat([pg,pg]),torch.cat([pc,pc]),\n                    torch.cat([text,torch.zeros_like(text)]),torch.cat([modes,modes]),limit=position+1)\n                gl,ugl = gl[:,-1].chunk(2);cl,ucl = cl[:,-1].chunk(2)\n                gl = ugl+self.config.guidance*(gl-ugl);cl = ucl+self.config.guidance*(cl-ucl)\n            else:\n                gl,cl = self(g,c,pg,pc,text,modes,limit=position+1)\n                gl,cl = gl[:,-1],cl[:,-1]\n            temperature = self.config.temperature*(.5 if mode == 1 else 1.)\n            sample_g = (gl.float()+temperature*(-torch.log(-torch.log(torch.rand_like(gl).clamp_(1e-6,1-1e-6))))).argmax(-1)\n            sample_c = (cl.float()+temperature*(-torch.log(-torch.log(torch.rand_like(cl).clamp_(1e-6,1-1e-6))))).argmax(-1)\n            sample_c = torch.where(sample_g != 0,sample_c,self.contract.empty_color)\n            p = self.config.patch_size\n            row,col = divmod(position,self.contract.cols//p)\n            state[:,0,row*p:(row+1)*p,col*p:(col+1)*p] = sample_g.reshape(-1,p,p)\n            state[:,1,row*p:(row+1)*p,col*p:(col+1)*p] = sample_c.reshape(-1,p,p)\n        state[:,4] = min(cursor+self.config.patches_per_step,self.patch_count)\n        return state,gl,cl\n\n    def config_dict(self):return asdict(self.config)\n\n\ndef causal_loss(model,gl,cl,g,c):\n    pg,pc = model.patches(g),model.patches(c)\n    glyph = F.cross_entropy(gl.flatten(0,2).float(),pg.flatten().long())\n    active = pg != 0\n    color = F.cross_entropy(cl[active].float(),pc[active].long()) if active.any() else cl.sum()*0\n    return glyph+color\n', 'ascii_flux/cascade.py': '"""Two-stage masked token transformer over native glyph grids.\n\nDesign follows published masked generative transformers (MaskGIT, Chang et al.\n2022; Muse, Chang et al. 2023: cross-attention on text token sequences,\nclassifier-free guidance) and cascaded generation (Imagen, Saharia et al.\n2022): a silhouette is decided first, glyphs and colors are filled inside it.\nOne backbone is shared by both stages. Grids are cut into 2x2-cell patches.\nNothing here touches pixels: inputs and outputs are glyph/color token IDs.\n"""\nfrom dataclasses import asdict, dataclass\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n# Horizontal mirror of the structure vocabulary " .:+*#@-_|/\\()\'".\nMIRROR = {"/": "\\\\", "\\\\": "/", "(": ")", ")": "("}\n\n\n@dataclass(frozen=True)\nclass CascadeConfig:\n    dim: int = 256\n    depth: int = 8\n    heads: int = 8\n    text_dim: int = 768\n    text_tokens: int = 16\n    patch: int = 2\n    text_dropout: float = .1\n    silhouette_steps: int = 12\n    glyph_steps: int = 8\n    guidance: float = 3.\n    temperature: float = 1.\n\n    def __post_init__(self):\n        if self.dim % self.heads or self.dim % (2*self.patch**2) or min(self.depth, self.text_tokens) < 1:\n            raise ValueError("Invalid cascade dimensions")\n\n\nclass Attention(nn.Module):\n    """Fused projections and scaled_dot_product_attention, without the layout\n    copies of nn.MultiheadAttention (measured as the bottleneck on ROCm)."""\n    def __init__(self, dim, heads, cross=False):\n        super().__init__()\n        self.heads, self.cross = heads, cross\n        if cross:\n            self.q, self.kv = nn.Linear(dim, dim), nn.Linear(dim, 2*dim)\n        else:\n            self.qkv = nn.Linear(dim, 3*dim)\n        self.out = nn.Linear(dim, dim)\n\n    def forward(self, x, context=None):\n        b, n, d = x.shape\n        h = self.heads\n        if self.cross:\n            q = self.q(x).view(b, n, h, d//h).transpose(1, 2)\n            k, v = self.kv(context).view(context.shape[0], context.shape[1], 2, h, d//h).permute(2, 0, 3, 1, 4)\n            if k.shape[0] != b:\n                k, v = k.expand(b, -1, -1, -1), v.expand(b, -1, -1, -1)\n        else:\n            q, k, v = self.qkv(x).view(b, n, 3, h, d//h).permute(2, 0, 3, 1, 4)\n        y = F.scaled_dot_product_attention(q, k, v)\n        return self.out(y.transpose(1, 2).reshape(b, n, d))\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, heads):\n        super().__init__()\n        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)\n        self.attention = Attention(dim, heads)\n        self.cross = Attention(dim, heads, cross=True)\n        self.mlp = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Linear(4*dim, dim))\n\n    def forward(self, x, text):\n        x = x+self.attention(self.norm1(x))\n        x = x+self.cross(self.norm2(x), text)\n        return x+self.mlp(self.norm3(x))\n\n\nclass CascadeGlyphModel(nn.Module):\n    def __init__(self, contract, config=None):\n        super().__init__()\n        self.contract, self.config = contract, config or CascadeConfig()\n        c, p, d = self.config, self.config.patch, self.config.dim\n        if contract.rows % p or contract.cols % p:\n            raise ValueError("Grid must divide into patches")\n        self.ph, self.pw = contract.rows//p, contract.cols//p\n        self.cells = p*p\n        self.glyph_count, self.color_count = len(contract.glyphs), len(contract.palette)\n        self.mask_glyph = self.glyph_count            # input-only IDs\n        self.empty_color_id, self.mask_color = self.color_count, self.color_count+1\n        e = d//(2*self.cells)\n        self.glyph_embedding = nn.Embedding(self.glyph_count+1, e)\n        self.color_embedding = nn.Embedding(self.color_count+2, e)\n        self.silhouette_embedding = nn.Embedding(3, d)     # empty, occupied, masked\n        self.stage_embedding = nn.Embedding(2, d)\n        self.position = nn.Parameter(torch.randn(1, self.ph*self.pw, d)*.02)\n        self.text_projection = nn.Sequential(nn.LayerNorm(c.text_dim), nn.Linear(c.text_dim, d))\n        self.text_position = nn.Parameter(torch.randn(1, c.text_tokens, d)*.02)\n        self.null_text = nn.Parameter(torch.randn(1, c.text_tokens, d)*.02)\n        self.blocks = nn.ModuleList(Block(d, c.heads) for _ in range(c.depth))\n        self.norm = nn.LayerNorm(d)\n        self.silhouette_head = nn.Linear(d, 2)\n        self.glyph_head = nn.Linear(d, self.cells*self.glyph_count)\n        self.color_head = nn.Linear(d, self.cells*self.color_count)\n        mirror = [contract.glyphs.index(MIRROR.get(g, g)) for g in contract.glyphs]\n        self.register_buffer("mirror_glyph", torch.tensor(mirror), persistent=False)\n\n    # Layout helpers: grid [B,R,C] <-> patches [B,N,cells].\n    def patches(self, grid):\n        b, p = grid.shape[0], self.config.patch\n        return grid.view(b, self.ph, p, self.pw, p).permute(0, 1, 3, 2, 4).reshape(b, self.ph*self.pw, self.cells)\n\n    def grid(self, patches):\n        b, p = patches.shape[0], self.config.patch\n        return patches.view(b, self.ph, self.pw, p, p).permute(0, 1, 3, 2, 4).reshape(b, self.contract.rows, self.contract.cols)\n\n    def silhouette_of(self, glyphs):\n        return (self.patches(glyphs) != 0).any(-1).long()\n\n    def text_context(self, text, drop=None):\n        if text is None:\n            return self.null_text\n        context = self.text_projection(text.float())+self.text_position\n        if drop is not None:\n            context = torch.where(drop[:, None, None], self.null_text.expand_as(context), context)\n        return context\n\n    def backbone(self, x, stage, text):\n        x = x+self.position+self.stage_embedding.weight[stage]\n        text = text.expand(x.shape[0], -1, -1)\n        for block in self.blocks:\n            x = block(x, text)\n        return self.norm(x)\n\n    def silhouette_logits(self, silhouette, text):\n        """silhouette [B,N] in {0,1,2=masked}; returns [B,N,2]."""\n        return self.silhouette_head(self.backbone(self.silhouette_embedding(silhouette), 1, text))\n\n    def glyph_logits(self, glyph_patches, color_patches, silhouette, text):\n        """Patches [B,N,cells] with masked IDs; returns glyph [B,N,cells,G], color [B,N,cells,P]."""\n        b, n = glyph_patches.shape[:2]\n        colors = torch.where(glyph_patches == self.mask_glyph, self.mask_color,\n                             torch.where(glyph_patches == 0, self.empty_color_id, color_patches))\n        x = torch.cat([self.glyph_embedding(glyph_patches).flatten(2), self.color_embedding(colors).flatten(2)], -1)\n        x = x+self.silhouette_embedding(silhouette)\n        h = self.backbone(x, 0, text)\n        return (self.glyph_head(h).view(b, n, self.cells, self.glyph_count),\n                self.color_head(h).view(b, n, self.cells, self.color_count))\n\n    def mirror(self, glyphs, colors):\n        glyphs = self.mirror_glyph[glyphs.flip(-1).long()]\n        return glyphs, colors.flip(-1)\n\n    def config_dict(self):\n        return asdict(self.config)\n\n    # ----- sampling -----\n    def _guided(self, function, text, *inputs):\n        guidance = self.config.guidance\n        if guidance == 1:\n            return function(*inputs, self.text_context(text))\n        doubled = [torch.cat([i, i]) for i in inputs]\n        context = torch.cat([self.text_context(text), self.null_text.expand(len(text), -1, -1)])\n        outputs = function(*doubled, context)\n        if not isinstance(outputs, tuple):\n            outputs = (outputs,)\n        guided = tuple(u+guidance*(c-u) for c, u in (o.float().chunk(2) for o in outputs))\n        return guided if len(guided) > 1 else guided[0]\n\n    @staticmethod\n    def _keep_masked(confidence, masked, ratio, temperature):\n        """MaskGIT schedule: re-mask the least confident of the newly sampled."""\n        noise = -torch.log(-torch.log(torch.rand_like(confidence).clamp(1e-6, 1-1e-6)))\n        score = confidence+temperature*noise\n        score = torch.where(masked, score, torch.full_like(score, math.inf))\n        count = (masked.sum(-1).float()*ratio).floor().long()\n        order = score.argsort(-1)\n        rank = torch.empty_like(order)\n        rank.scatter_(-1, order, torch.arange(order.shape[-1], device=order.device).expand_as(order))\n        return rank < count[:, None]\n\n    @torch.no_grad()\n    def silhouette_step(self, text, silhouette, masked, step, total):\n        """One MaskGIT step of the silhouette stage; returns (silhouette, masked)."""\n        logits = self._guided(self.silhouette_logits, text, silhouette)\n        tau = self.config.temperature*(1-step/total)\n        noise = -torch.log(-torch.log(torch.rand(logits.shape, device=logits.device).clamp(1e-6, 1-1e-6)))\n        choice = (logits+tau*noise).argmax(-1)\n        confidence = F.log_softmax(logits, -1).gather(-1, choice[..., None])[..., 0]\n        candidate = torch.where(masked, choice, silhouette)\n        masked = self._keep_masked(confidence, masked, math.cos(math.pi/2*(step+1)/total), tau)\n        return torch.where(masked, 2, candidate), masked\n\n    def start_glyphs(self, silhouette):\n        """Glyph stage start: patches outside the silhouette are empty, inside masked."""\n        b, n = silhouette.shape\n        glyphs = torch.full((b, n, self.cells), self.mask_glyph, dtype=torch.long, device=silhouette.device)\n        colors = torch.full_like(glyphs, self.mask_color)\n        inside = silhouette == 1\n        glyphs[~inside] = 0\n        colors[~inside] = self.contract.empty_color\n        return glyphs, colors, inside.clone()\n\n    @torch.no_grad()\n    def glyph_step(self, text, glyphs, colors, silhouette, masked, step, total):\n        """One MaskGIT step of the glyph stage; returns (glyphs, colors, masked)."""\n        gl, cl = self._guided(self.glyph_logits, text, glyphs, colors, silhouette)\n        tau = self.config.temperature*(1-step/total)\n        noise = -torch.log(-torch.log(torch.rand(gl.shape, device=gl.device).clamp(1e-6, 1-1e-6)))\n        sampled_g = (gl+tau*noise).argmax(-1)\n        sampled_c = cl.argmax(-1)\n        confidence = F.log_softmax(gl, -1).gather(-1, sampled_g[..., None])[..., 0].mean(-1)\n        still = self._keep_masked(confidence, masked, math.cos(math.pi/2*(step+1)/total), tau)\n        fill = masked & ~still\n        return (torch.where(fill[..., None], sampled_g, glyphs),\n                torch.where(fill[..., None], sampled_c, colors), still)\n\n    @torch.no_grad()\n    def sample(self, text, steps=None, generator=None, return_trace=False):\n        """text [B,T,768]. Returns glyphs, colors [B,R,C] (and per-step grids)."""\n        c = self.config\n        b, n, device = text.shape[0], self.ph*self.pw, text.device\n        trace = []\n        s_steps, g_steps = steps or (c.silhouette_steps, c.glyph_steps)\n        silhouette = torch.full((b, n), 2, dtype=torch.long, device=device)\n        masked = torch.ones((b, n), dtype=torch.bool, device=device)\n        for step in range(s_steps):\n            silhouette, masked = self.silhouette_step(text, silhouette, masked, step, s_steps)\n            if return_trace:\n                trace.append(self._render(silhouette, None, None))\n        glyphs, colors, masked = self.start_glyphs(silhouette)\n        for step in range(g_steps):\n            glyphs, colors, masked = self.glyph_step(text, glyphs, colors, silhouette, masked, step, g_steps)\n            if return_trace:\n                trace.append(self._render(silhouette, glyphs, colors))\n        g, col = self._render(silhouette, glyphs, colors)\n        return (g, col, trace) if return_trace else (g, col)\n\n    def _render(self, silhouette, glyphs, colors):\n        """Displayable tokens at any stage: undecided silhouette patches stay empty."""\n        b = silhouette.shape[0]\n        if glyphs is None:\n            # During the silhouette stage, show occupied patches as a light fill.\n            fill = torch.full((b, silhouette.shape[1], self.cells), self.contract.glyphs.index(":"), device=silhouette.device)\n            g = torch.where((silhouette == 1)[..., None], fill, 0)\n            col = torch.full_like(g, self.contract.empty_color)\n            col[g != 0] = len(self.contract.palette)-1\n            return self.grid(g), self.grid(col)\n        g = torch.where(glyphs == self.mask_glyph, 0, glyphs)\n        col = torch.where(g == 0, self.contract.empty_color, colors.clamp(max=self.color_count-1))\n        return self.grid(g), self.grid(col)\n\n\ndef cascade_loss(model, glyphs, colors, text_context, generator=None):\n    """Both stages on one batch of real target grids [B,R,C]."""\n    device = glyphs.device\n    b = glyphs.shape[0]\n    silhouette = model.silhouette_of(glyphs)\n    # Stage A: masked silhouette with ratio from the cosine schedule.\n    ratio = torch.cos(torch.rand(b, device=device)*math.pi/2).clamp(min=1/silhouette.shape[1])\n    mask_a = torch.rand(silhouette.shape, device=device) < ratio[:, None]\n    logits_a = model.silhouette_logits(torch.where(mask_a, 2, silhouette), text_context)\n    loss_a = F.cross_entropy(logits_a[mask_a].float(), silhouette[mask_a])\n    # Stage B: glyphs/colors inside the true silhouette, masked by patch.\n    gp, cp = model.patches(glyphs), model.patches(colors)\n    inside = silhouette == 1\n    ratio = torch.cos(torch.rand(b, device=device)*math.pi/2).clamp(min=.05)\n    mask_b = (torch.rand(silhouette.shape, device=device) < ratio[:, None]) & inside\n    gin = torch.where(mask_b[..., None], model.mask_glyph, gp)\n    cin = torch.where(mask_b[..., None], model.mask_color, cp)\n    gl, cl = model.glyph_logits(gin, cin, silhouette, text_context)\n    target_g = gp[mask_b]\n    loss_g = F.cross_entropy(gl[mask_b].float().reshape(-1, model.glyph_count), target_g.reshape(-1)) if mask_b.any() else gl.sum()*0\n    occupied = mask_b[..., None] & (gp != 0)\n    loss_c = F.cross_entropy(cl[occupied].float(), cp[occupied].long()) if occupied.any() else cl.sum()*0\n    return loss_a+loss_g+loss_c, {"silhouette_ce": float(loss_a.detach()), "glyph_ce": float(loss_g.detach()), "color_ce": float(loss_c.detach())}\n', 'ascii_flux/cascade_training.py': '"""Training of the cascade glyph transformer on the teacher shape corpus.\n\nCorpus records are tagged ``synthetic: teacher-t2i``. Words are split by hash:\n``val`` and ``test_unseen`` words are never trained, so validation measures\ndrawing words the model has not been taught (an automatic proxy only).\n"""\nfrom dataclasses import asdict, dataclass\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom .cascade import CascadeConfig, CascadeGlyphModel, cascade_loss\nfrom .contract import canonical, digest\nfrom .storage import atomic_file, write_json\nfrom .structure import STRUCTURE\nfrom .training import restore_rng, rng_state\n\n\n@dataclass(frozen=True)\nclass CascadeTrainConfig:\n    steps: int = 40000\n    batch_size: int = 32\n    lr: float = 4e-4\n    warmup: int = 1000\n    weight_decay: float = .05\n    validate_every: int = 1000\n    val_batches: int = 16\n    flip: float = .5\n    ema: float = .999\n    seed: int = 20261031\n\n\ndef load_shape_corpus(root, shard_limit=None):\n    """Read the first ``shard_limit`` shards, a fixed snapshot while generation continues."""\n    root = Path(root)\n    words, glyphs, colors = [], [], []\n    shards = sorted(root.glob("shard-*.json"))[:shard_limit]\n    fingerprint = hashlib.sha256()\n    for sidecar in shards:\n        meta = json.loads(sidecar.read_text(encoding="utf-8"))\n        data = sidecar.with_suffix(".npz").read_bytes()\n        if hashlib.sha256(data).hexdigest() != meta["npz_sha256"] or meta["contract_hash"] != STRUCTURE.hash:\n            raise ValueError(f"Invalid shape shard {sidecar.name}")\n        fingerprint.update(meta["npz_sha256"].encode())\n        import io\n        with np.load(io.BytesIO(data), allow_pickle=False) as arrays:\n            offset = len(glyphs)\n            g, c = arrays["glyphs"], arrays["colors"]\n        glyphs.extend(g)\n        colors.extend(c)\n        for record in meta["words"]:\n            if record["samples"]:\n                words.append({"word": record["word"], "split": record["split"],\n                              "indices": [offset+s["index"] for s in record["samples"]]})\n    if not words:\n        raise ValueError("Empty shape corpus")\n    return words, np.asarray(glyphs, np.uint8), np.asarray(colors, np.uint8), fingerprint.hexdigest(), len(shards)\n\n\ndef phrases_for(word):\n    return [word, f"a {word}"]\n\n\ndef cache_text_tokens(words, output, device="cuda", shards=None):\n    from .text import TOKEN_SPEC, TextEncoder\n    encoder = TextEncoder(device)\n    names = sorted({w["word"] for w in words})\n    texts = [p for name in names for p in phrases_for(name)]\n    features = np.concatenate([encoder.encode_tokens(texts[i:i+128]) for i in range(0, len(texts), 128)])\n    features = features.reshape(len(names), len(phrases_for("x")), *features.shape[1:])\n    metadata = {"spec": TOKEN_SPEC, "words": len(names), "phrases": "word | a word", "corpus_shards": shards}\n    with atomic_file(output) as stream:\n        np.savez(stream, words=np.array(names), features=features, metadata=np.array(canonical(metadata)))\n    return metadata\n\n\nclass ShapeData:\n    def __init__(self, corpus_root, tokens_path, device):\n        with np.load(tokens_path, allow_pickle=False) as cache:\n            limit = json.loads(str(cache["metadata"])).get("corpus_shards")\n        self.words, self.glyphs, self.colors, self.fingerprint, self.shards = load_shape_corpus(corpus_root, limit)\n        with np.load(tokens_path, allow_pickle=False) as cache:\n            index = {w: i for i, w in enumerate(cache["words"].tolist())}\n            missing = {w["word"] for w in self.words}-set(index)\n            if missing:\n                raise ValueError(f"Text tokens missing for {len(missing)} words; rebuild the cache")\n            self.features = torch.tensor(cache["features"])\n            self.text_metadata = json.loads(str(cache["metadata"]))\n        self.index = index\n        self.device = device\n        self.split = {s: [w for w in self.words if w["split"] == s] for s in ("train", "val", "test_unseen")}\n        if not self.split["train"] or not self.split["val"]:\n            raise ValueError("Need train and never-trained validation words")\n\n    def batch(self, split, size, rng, flip=0.):\n        chosen = [rng.choice(self.split[split]) for _ in range(size)]\n        samples = [rng.choice(w["indices"]) for w in chosen]\n        phrase = [rng.randrange(self.features.shape[1]) if split == "train" else 0 for _ in chosen]\n        g = torch.tensor(self.glyphs[samples], dtype=torch.long, device=self.device)\n        c = torch.tensor(self.colors[samples], dtype=torch.long, device=self.device)\n        text = torch.stack([self.features[self.index[w["word"]], p] for w, p in zip(chosen, phrase)]).to(self.device)\n        flips = torch.tensor([rng.random() < flip for _ in chosen], device=self.device)\n        return g, c, text, flips\n\n\n@torch.inference_mode()\ndef validate(model, data, config, device, split="val"):\n    saved = rng_state()\n    model.eval()\n    try:\n        rng = random.Random(config.seed+1)\n        torch.manual_seed(config.seed+1)\n        rows = {"with_text": [], "without_text": []}\n        for i in range(config.val_batches):\n            g, c, text, _ = data.batch(split, config.batch_size, rng)\n            for key, context in (("with_text", model.text_context(text)), ("without_text", model.null_text)):\n                # Same masks on every device for both conditions.\n                torch.manual_seed(config.seed+100+i)\n                with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):\n                    loss, parts = cascade_loss(model, g, c, context)\n                rows[key].append(parts)\n        mean = {k: {p: float(np.mean([r[p] for r in v])) for p in v[0]} for k, v in rows.items()}\n        mean["silhouette_text_gain"] = mean["without_text"]["silhouette_ce"]-mean["with_text"]["silhouette_ce"]\n        mean["total"] = sum(mean["with_text"].values())\n        return mean\n    finally:\n        restore_rng(saved)\n        model.train()\n\n\ndef train_cascade(corpus_root, tokens_path, output, config=None, model_config=None, device="cuda", resume=False):\n    config, model_config = config or CascadeTrainConfig(), model_config or CascadeConfig()\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if (output/"latest.pt").exists() and not resume:\n        raise FileExistsError("Existing run; pass resume or choose a new output")\n    data = ShapeData(corpus_root, tokens_path, device)\n    random.seed(config.seed); np.random.seed(config.seed); torch.manual_seed(config.seed)\n    model = CascadeGlyphModel(STRUCTURE, model_config).to(device)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    decay, no_decay = [], []\n    for name, parameter in model.named_parameters():\n        (no_decay if parameter.ndim < 2 or "embedding" in name or "position" in name or "null" in name else decay).append(parameter)\n    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": config.weight_decay},\n                                   {"params": no_decay, "weight_decay": 0.}], lr=config.lr, betas=(.9, .98))\n    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"))\n    code = hashlib.sha256()\n    for name in ("cascade.py", "cascade_training.py", "structure.py"):\n        code.update(Path(__file__).with_name(name).read_bytes())\n    provenance = {"corpus_fingerprint": data.fingerprint, "corpus_shards": data.shards, "synthetic": "teacher-t2i",\n                  "text": data.text_metadata, "train_words": len(data.split["train"]),\n                  "never_trained_val_words": len(data.split["val"]), "training_code_sha256": code.hexdigest(),\n                  "recognition_validated": False, "development_only": True,\n                  "scope": "cascade silhouette->glyph transformer on teacher shapes; static shapes, no motion yet"}\n    start, best, history = 0, math.inf, []\n    if resume:\n        checkpoint = torch.load(output/"latest.pt", map_location="cpu", weights_only=False)\n        old, new = dict(checkpoint["train_config"]), asdict(config)\n        old.pop("steps"); new.pop("steps")\n        if checkpoint["provenance"] != provenance or old != new or checkpoint["model_config"] != asdict(model_config):\n            raise ValueError("Resume provenance/config mismatch")\n        model.load_state_dict(checkpoint["model"]); ema.load_state_dict(checkpoint["ema"])\n        optimizer.load_state_dict(checkpoint["optimizer"]); scaler.load_state_dict(checkpoint["scaler"])\n        start, best, history = checkpoint["step"], checkpoint["best"], checkpoint["history"]\n        restore_rng(checkpoint["rng"])\n    parameters = sum(p.numel() for p in model.parameters())\n    print(json.dumps({"parameters": parameters, "train_words": len(data.split["train"]),\n                      "val_words": len(data.split["val"]), "samples": len(data.glyphs)}), flush=True)\n    started = time.monotonic()\n    for step in range(start, config.steps):\n        lr = config.lr*min(1, (step+1)/config.warmup)*.5*(1+math.cos(math.pi*min(1, step/config.steps)))\n        for group in optimizer.param_groups:\n            group["lr"] = lr\n        g, c, text, flips = data.batch("train", config.batch_size, random, config.flip)\n        fg, fc = model.mirror(g, c)\n        g, c = torch.where(flips[:, None, None], fg, g), torch.where(flips[:, None, None], fc, c)\n        drop = torch.rand(len(g), device=device) < model_config.text_dropout\n        with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):\n            loss, parts = cascade_loss(model, g, c, model.text_context(text, drop))\n        if not torch.isfinite(loss):\n            raise FloatingPointError("Non-finite cascade loss")\n        optimizer.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)\n        scaler.step(optimizer)\n        scaler.update()\n        with torch.no_grad():\n            rate = min(config.ema, (1+step)/(10+step))\n            for a, b in zip(ema.parameters(), model.parameters()):\n                a.lerp_(b, 1-rate)\n        if (step+1) % 50 == 0:\n            write_json(output/"progress.json", {"step": step+1, "total_steps": config.steps, "loss": float(loss),\n                                                 **parts, "elapsed_seconds": time.monotonic()-started,\n                                                 "recognition_validated": False})\n        if (step+1) % config.validate_every == 0 or step+1 == config.steps:\n            val = validate(ema, data, config, device)\n            row = {"step": step+1, "train": parts, "val_never_trained_words": val}\n            if step+1 == config.steps and data.split["test_unseen"]:\n                row["test_unseen_final_report_only"] = validate(ema, data, config, device, "test_unseen")\n            history.append(row)\n            weights = {k: v.detach().cpu() for k, v in ema.state_dict().items()}\n            if val["total"] < best:\n                best = val["total"]\n                with atomic_file(output/"best.pt") as stream:\n                    torch.save({"format": "ascii-flux-v5-cascade-1", "contract": STRUCTURE.to_dict(),\n                                "model_config": asdict(model_config), "provenance": provenance,\n                                "model": weights, "step": step+1}, stream)\n            with atomic_file(output/"latest.pt") as stream:\n                torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "optimizer": optimizer.state_dict(),\n                            "scaler": scaler.state_dict(), "step": step+1, "best": best, "history": history,\n                            "rng": rng_state(), "train_config": asdict(config), "model_config": asdict(model_config),\n                            "provenance": provenance}, stream)\n            write_json(output/"history.json", history)\n            print(json.dumps(row), flush=True)\n    return model\n', 'ascii_flux/catalogue.py': '"""SQLite catalogue of immutable ASCII shards; frame counts are not epochs.\n\nPayloads remain uint8 NPZ in TAR files. SQLite keeps only provenance, clip\nlocations and hashes, so a million grids need not be loaded into memory.\n"""\nfrom collections import Counter\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nfrom .contract import Contract, DEFAULT, canonical, digest\nfrom .storage import atomic_file, read_clip, write_json\n\n\nclass Catalogue:\n    def __init__(self, root, contract=DEFAULT, target_grids=1_000_000):\n        self.root = Path(root).resolve()\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.contract = contract\n        self.db = sqlite3.connect(self.root / "catalogue.sqlite", timeout=60)\n        self.db.execute("PRAGMA foreign_keys=ON")\n        self.db.execute("PRAGMA journal_mode=WAL")\n        self.db.execute("PRAGMA synchronous=FULL")\n        self.db.executescript("""\n            CREATE TABLE IF NOT EXISTS settings(key TEXT PRIMARY KEY, value TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS runs(fingerprint TEXT PRIMARY KEY, manifest TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS videos(uid TEXT PRIMARY KEY, split TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS clips(\n                uid TEXT PRIMARY KEY, run TEXT NOT NULL REFERENCES runs(fingerprint),\n                video TEXT NOT NULL REFERENCES videos(uid), split TEXT NOT NULL,\n                label TEXT NOT NULL, record TEXT NOT NULL);\n            CREATE TABLE IF NOT EXISTS frames(\n                clip TEXT NOT NULL REFERENCES clips(uid), t INTEGER NOT NULL,\n                observation BLOB NOT NULL, tokens BLOB NOT NULL,\n                PRIMARY KEY(clip,t));\n            CREATE INDEX IF NOT EXISTS frame_observation ON frames(observation);\n            CREATE INDEX IF NOT EXISTS frame_tokens ON frames(tokens);\n            CREATE INDEX IF NOT EXISTS clip_split_label ON clips(split,label);\n        """)\n        with self.db:\n            old = self.db.execute("SELECT value FROM settings WHERE key=\'contract\'").fetchone()\n            if old and old[0] != canonical(contract.to_dict()):\n                raise ValueError("Catalogue representation mismatch; choose a new version")\n            self.db.execute("INSERT OR IGNORE INTO settings VALUES(\'contract\',?)", (canonical(contract.to_dict()),))\n            self.db.execute("INSERT OR IGNORE INTO settings VALUES(\'target_grids\',?)", (str(target_grids),))\n\n    def close(self):\n        self.db.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *args):\n        self.close()\n\n    def contains_run(self, fingerprint):\n        return bool(self.db.execute("SELECT 1 FROM runs WHERE fingerprint=?", (fingerprint,)).fetchone())\n\n    def ingest(self, manifest_path):\n        """Verify and commit a whole completed shard run, once, transactionally.\n\n        Files are copied before the transaction; a crash can leave harmless\n        unreferenced files. Repeating ingest then finishes the same transaction.\n        """\n        from .review import load_review_manifest\n        from .factory import file_sha256\n        path = Path(manifest_path).resolve()\n        manifest = load_review_manifest(path)\n        self.contract.require(manifest["contract"])\n        fingerprint = manifest["fingerprint"]\n        if self.contains_run(fingerprint):\n            return False\n        relative = Path("sets") / fingerprint\n        destination = self.root / relative\n        destination.mkdir(parents=True, exist_ok=True)\n        for name, expected in manifest["shards"].items():\n            source = (path.parent / name).resolve()\n            target = (destination / name).resolve()\n            if not target.is_relative_to(destination):\n                raise ValueError("Unsafe shard path")\n            if not target.exists():\n                with atomic_file(target) as stream, source.open("rb") as data:\n                    shutil.copyfileobj(data, stream, 1024*1024)\n            if file_sha256(target) != expected:\n                raise ValueError("Stored shard checksum mismatch")\n        with self.db:\n            self.db.execute("INSERT INTO runs VALUES(?,?)", (fingerprint, canonical(manifest)))\n            for record in manifest["clips"]:\n                g, c, _, meta = read_clip(path.parent, record, self.contract)\n                # Same source video stays in the same partition across snapshots.\n                video = str(meta.get("source_video_uid") or f"{record[\'source\']}:{record[\'video_id\']}")\n                old = self.db.execute("SELECT split FROM videos WHERE uid=?", (video,)).fetchone()\n                if old and old[0] != record["split"]:\n                    raise ValueError(f"Video split leakage: {video}")\n                self.db.execute("INSERT OR IGNORE INTO videos VALUES(?,?)", (video, record["split"]))\n                indices = meta.get("source_frame_indices", list(range(len(g))))\n                paths = meta.get("source_frame_paths", [str(i) for i in indices])\n                # Observation identity includes object: two subjects in a source\n                # frame are different samples, while upsampling is a repetition.\n                obj = str(meta.get("source_object_id") or record.get("instance_id", record["clip_id"]))\n                sample = digest([video, obj, indices, meta.get("augmentation"), self.contract.hash, manifest["converter"]])\n                if self.db.execute("SELECT 1 FROM clips WHERE uid=?", (sample,)).fetchone():\n                    continue\n                stored = dict(record, clip_id=record["clip_id"], shard=(relative/record["shard"]).as_posix())\n                stored["catalogue_id"] = sample\n                self.db.execute("INSERT INTO clips VALUES(?,?,?,?,?,?)", (\n                    sample, fingerprint, video, record["split"], record["normalized_label"], canonical(stored)))\n                frames = []\n                for t in range(len(g)):\n                    observation = hashlib.sha256(canonical([video, obj, paths[t]]).encode()).digest()\n                    tokens = hashlib.sha256(g[t].tobytes()+c[t].tobytes()).digest()\n                    frames.append((sample, t, observation, tokens))\n                self.db.executemany("INSERT INTO frames VALUES(?,?,?,?)", frames)\n        write_json(destination/"manifest.json", manifest)\n        rejections = path.parent/"rejections.json"\n        if rejections.exists():\n            write_json(destination/"rejections.json", json.loads(rejections.read_text(encoding="utf-8")))\n        return True\n\n    def report(self):\n        slots, observations, tokens = self.db.execute(\n            "SELECT COUNT(*),COUNT(DISTINCT observation),COUNT(DISTINCT tokens) FROM frames").fetchone()\n        target = int(self.db.execute("SELECT value FROM settings WHERE key=\'target_grids\'").fetchone()[0])\n        by_source = {}\n        original_clips = augmented_clips = 0\n        original_by_source = Counter()\n        for (raw,) in self.db.execute("SELECT record FROM clips"):\n            record = json.loads(raw)\n            if record.get("augmentation"):\n                augmented_clips += 1\n            else:\n                original_clips += 1\n                original_by_source[record["source"]] += 1\n        rejected = Counter()\n        for (raw,) in self.db.execute("SELECT manifest FROM runs"):\n            run = json.loads(raw)\n            source = run["source"]["source"]\n            rejected.update(run.get("report", {}).get("reasons", {}))\n            counts = by_source.setdefault(source, {"clips": 0, "grid_slots": 0, "attempted_original_clips": 0, "conditions": run["source"]})\n            actual = self.db.execute("SELECT COUNT(*) FROM clips WHERE run=?", (run["fingerprint"],)).fetchone()[0]\n            counts["clips"] += actual\n            counts["grid_slots"] += actual*self.contract.frames\n            counts["attempted_original_clips"] += run.get("report", {}).get("attempted_full_clips", 0)\n        for source, counts in by_source.items():\n            counts["accepted_original_clips"] = original_by_source[source]\n            counts["yield_original_clips"] = original_by_source[source]/max(1, counts["attempted_original_clips"])\n        return {"target_grids": target, "clips": self.db.execute("SELECT COUNT(*) FROM clips").fetchone()[0],\n                "grid_slots": slots, "distinct_source_object_observations": observations,\n                "original_clips": original_clips, "augmented_clips": augmented_clips,\n                "original_grid_slots": original_clips*self.contract.frames,\n                "augmentation_grid_slots": augmented_clips*self.contract.frames,\n                "distinct_token_grids": tokens, "repeated_observation_slots": slots-observations,\n                "target_slots_reached": slots >= target, "target_distinct_grids_reached": tokens >= target,\n                "split_clips": dict(self.db.execute("SELECT split,COUNT(*) FROM clips GROUP BY split")),\n                "by_source": by_source, "rejection_reasons": dict(rejected),\n                "status": "machine candidates; human quality validation required"}\n\n    def snapshot(self, name="manifest-d2.json"):\n        """Freeze a training view beside the DB; never change an existing view."""\n        if Path(name).name != name:\n            raise ValueError("Snapshot filename must stay in catalogue root")\n        runs = [json.loads(row[0]) for row in self.db.execute("SELECT manifest FROM runs ORDER BY fingerprint")]\n        if not runs:\n            raise ValueError("Empty catalogue")\n        converters = {canonical(run["converter"]) for run in runs}\n        heldout = {canonical(run["config"]["heldout_categories"]) for run in runs}\n        quality = {canonical(run["config"]["quality"]) for run in runs}\n        if len(converters) != 1 or len(heldout) != 1 or len(quality) != 1:\n            raise ValueError("A training snapshot requires one converter, held-out set and quality policy")\n        records = [json.loads(row[0]) for row in self.db.execute("SELECT record FROM clips ORDER BY uid")]\n        shards = {f"sets/{run[\'fingerprint\']}/{name}": sha for run in runs for name, sha in run["shards"].items()}\n        sources = {digest(r["source"]): r["source"] for r in runs}\n        manifest = {"dataset_version": "ascii-v5-d2-catalogue-1", "contract": self.contract.to_dict(),\n                    "contract_hash": self.contract.hash, "source": {"sources": [sources[key] for key in sorted(sources)]},\n                    "converter": runs[0]["converter"], "config": {"heldout_categories": json.loads(next(iter(heldout))),\n                    "quality": json.loads(next(iter(quality)))}, "clips": records, "shards": shards,\n                    "report": self.report(), "catalogue_runs": [r["fingerprint"] for r in runs]}\n        manifest["fingerprint"] = digest(manifest)\n        target = self.root/name\n        if target.exists() and json.loads(target.read_text(encoding="utf-8")) != manifest:\n            raise FileExistsError("Snapshot is immutable; use a new name")\n        write_json(target, manifest)\n        return target\n\n\n@contextmanager\ndef writer_lock(root):\n    """OS advisory lock automatically releases after a killed process."""\n    path = Path(root)/".writer.lock"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a+b") as stream:\n        stream.seek(0)\n        if os.name == "nt":\n            import msvcrt\n            if not stream.read(1):\n                stream.write(b"0"); stream.flush()\n            stream.seek(0)\n            msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)\n        else:\n            import fcntl\n            fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        yield\n\n', 'ascii_flux/causal_training.py': '"""Balanced real-data experiment for the native causal glyph transformer."""\nimport copy\nfrom dataclasses import asdict,dataclass\nimport hashlib\nimport math\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom .autoregressive import CausalConfig,CausalGlyphFlow,causal_loss\nfrom .contract import TEXT_SPEC\nfrom .masked_training import MaskedTrainConfig,subject_groups,token_batch\nfrom .storage import atomic_file,write_json\nfrom .training import Corpus,restore_rng,rng_state\n\n\n@dataclass(frozen=True)\nclass CausalTrainConfig(MaskedTrainConfig):\n    steps: int = 4000\n\n\ndef prepare(model,batch,validation=False):\n    g,c,pg,pc,text = batch\n    mode = torch.multinomial(torch.tensor([.65,.25,.10],device=g.device),len(g),replacement=True)\n    pg,pc = pg.clone(),pc.clone()\n    pg[mode == 0],pc[mode == 0] = model.mask_glyph,model.mask_color\n    for i in (mode == 2).nonzero().flatten().tolist():\n        others = ((text-text[i]).square().sum(-1) > 1e-8).nonzero().flatten()\n        if len(others):\n            j = int(others[torch.randint(len(others),(),device=g.device)])\n            pg[i],pc[i] = g[j],c[j]\n        else:\n            mode[i] = 0\n            pg[i],pc[i] = model.mask_glyph,model.mask_color\n    if not validation:\n        text = text*(torch.rand(len(g),1,device=g.device) >= model.config.text_dropout)\n    return (g,c,pg,pc,text,mode)\n\n\n@torch.inference_mode()\ndef validate(model,corpus,groups,config,device):\n    saved = rng_state()\n    model.eval()\n    try:\n        torch.manual_seed(config.seed+1)\n        rng = random.Random(config.seed+1)\n        values = []\n        for _ in range(config.val_batches):\n            batch = token_batch(corpus,groups,config,"val",device,rng)\n            inputs = prepare(model,batch,True)\n            with torch.autocast("cuda",dtype=torch.float16,enabled=str(device).startswith("cuda")):\n                gl,cl = model(*inputs)\n            values.append(float(causal_loss(model,gl,cl,batch[0],batch[1])))\n        return float(np.mean(values))\n    finally:\n        restore_rng(saved)\n        model.train()\n\n\ndef train_causal(manifest,embeddings,reviews,output,config=None,model_config=None,device="cuda",resume=None):\n    config,model_config = config or CausalTrainConfig(),model_config or CausalConfig()\n    if min(config.steps,config.batch_size,config.validate_every,config.val_batches) <= 0 or config.lr <= 0:\n        raise ValueError("Positive training configuration required")\n    corpus = Corpus(manifest,embeddings,reviews)\n    groups = subject_groups(corpus,config.subjects)\n    output = Path(output);output.mkdir(parents=True,exist_ok=True)\n    if (output/"latest.pt").exists() and not resume:\n        raise FileExistsError("Existing causal experiment; resume or choose a new run")\n    random.seed(config.seed);np.random.seed(config.seed);torch.manual_seed(config.seed)\n    model = CausalGlyphFlow(corpus.contract,model_config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(),lr=config.lr,weight_decay=.01)\n    scaler = torch.amp.GradScaler("cuda",enabled=str(device).startswith("cuda"),init_scale=128.)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    code = hashlib.sha256()\n    for name in ("autoregressive.py","causal_training.py","masked_training.py","text.py"):\n        code.update(Path(__file__).with_name(name).read_bytes())\n    provenance = {"dataset_fingerprint":corpus.manifest["fingerprint"],"text_spec":TEXT_SPEC,\n        "allow_fixture":False,"allow_unreviewed":False,"synthetic_encoder":False,\n        "development_only":True,"recognition_validated":False,\n        "scope":"real-data memorization diagnostic" if config.memorize_one_frame else "six-concept causal glyph experiment; recognition and motion unvalidated",\n        "training_concepts":list(config.subjects),"source_conditions":corpus.manifest["source"],\n        "review_scope":corpus.review_scope,"training_code_sha256":code.hexdigest(),"torch_version":str(torch.__version__)}\n    start,best,best_step,best_model,history,skipped = 0,math.inf,0,None,[],0\n    if resume:\n        ck = torch.load(resume,map_location="cpu",weights_only=False)\n        corpus.contract.require(ck["contract"])\n        old,new = dict(ck["train_config"]),asdict(config)\n        old.pop("steps");new.pop("steps")\n        if ck["provenance"] != provenance or old != new or ck["model_config"] != asdict(model_config):\n            raise ValueError("Causal resume provenance/config mismatch")\n        model.load_state_dict(ck["model"]);ema.load_state_dict(ck["ema"])\n        optimizer.load_state_dict(ck["optimizer"]);scaler.load_state_dict(ck["scaler"])\n        start,best,best_step,best_model,history,skipped = (ck[k] for k in ("step","best_val","best_step","best_model","history","skipped_amp_updates"))\n        restore_rng(ck["rng"])\n    started = time.monotonic()\n    print({"parameters":sum(p.numel() for p in model.parameters()),"scope":provenance["scope"]},flush=True)\n    for step in range(start,config.steps):\n        batch = token_batch(corpus,groups,config,"train",device,random)\n        inputs = prepare(model,batch)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast("cuda",dtype=torch.float16,enabled=str(device).startswith("cuda")):\n            gl,cl = model(*inputs)\n            loss = causal_loss(model,gl,cl,batch[0],batch[1])\n        if not torch.isfinite(loss):raise FloatingPointError("Non-finite causal loss")\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=not scaler.is_enabled())\n        scaler.step(optimizer);scaler.update()\n        if math.isfinite(float(norm)):\n            decay = min(.995,(1+step)/(10+step))\n            with torch.no_grad():\n                for a,b in zip(ema.parameters(),model.parameters()):a.lerp_(b,1-decay)\n        else:skipped += 1\n        if (step+1)%25 == 0:\n            write_json(output/"progress.json",{"step":step+1,"total_steps":config.steps,"loss":float(loss.detach()),\n                "elapsed_seconds":time.monotonic()-started,"optimizer_updates":step+1-skipped,"recognition_validated":False})\n        if (step+1)%config.validate_every == 0 or step+1 == config.steps:\n            value = validate(ema,corpus,groups,config,device)\n            if not math.isfinite(value):raise FloatingPointError("Non-finite causal validation")\n            if value < best:\n                best,best_step = value,step+1\n                best_model = copy.deepcopy({k:v.detach().cpu() for k,v in ema.state_dict().items()})\n            row = {"step":step+1,"train_ce":float(loss.detach()),"teacher_forced_val_ce":value,\n                   "best_val_ce":best,"optimizer_updates":step+1-skipped}\n            history.append(row)\n            ck = {"format":"ascii-flux-v5-causal-1","contract":corpus.contract.to_dict(),"model_config":asdict(model_config),\n                "train_config":asdict(config),"provenance":provenance,"model":model.state_dict(),"ema":ema.state_dict(),\n                "optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),"step":step+1,"best_val":best,"best_step":best_step,\n                "best_model":best_model,"rng":rng_state(),"history":history,"skipped_amp_updates":skipped}\n            with atomic_file(output/"latest.pt") as stream:torch.save(ck,stream)\n            with atomic_file(output/"best.pt") as stream:\n                torch.save({k:ck[k] for k in ("format","contract","model_config","provenance")} | {"model":best_model,"step":best_step},stream)\n            write_json(output/"history.json",history)\n            print(row,flush=True)\n    return model\n', 'ascii_flux/cli.py': '"""Small explicit CLI; optional training dependencies are imported on demand."""\nimport argparse\nimport json\nfrom pathlib import Path\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description="ASCII Flux v5 research pipeline")\n    sub = parser.add_subparsers(dest="command", required=True)\n    build = sub.add_parser("convert", help="Offline local YouTube-VIS -> ASCII shards")\n    for name in ("annotations", "frames-root", "output"):\n        build.add_argument("--"+name, required=True)\n    build.add_argument("--source", default="configs/ytvis_source.json")\n    build.add_argument("--config", default="configs/d1.json")\n    build.add_argument("--source-fps", required=True, type=float, help="Actual cadence of the extracted annotated frames")\n    review = sub.add_parser("review", help="Build standalone D1 review HTML")\n    review.add_argument("--manifest", required=True)\n    review.add_argument("--output", required=True)\n    scale = sub.add_parser("scale", help="Resume bounded D2 shard conversion into SQLite catalogue")\n    for name in ("annotations", "frames-root", "source", "output"):\n        scale.add_argument("--"+name, required=True)\n    scale.add_argument("--config", default="configs/d2.json")\n    scale.add_argument("--source-fps", type=float, required=True)\n    scale.add_argument("--workers", type=int, default=2)\n    scale.add_argument("--max-jobs", type=int)\n    catalog = sub.add_parser("catalogue", help="Index, count or freeze existing ASCII shard runs")\n    catalog.add_argument("--root", required=True)\n    catalog.add_argument("--ingest")\n    catalog.add_argument("--snapshot", help="Immutable manifest filename in catalogue root")\n    million = sub.add_parser("million", help="Acquire, convert, freeze and train one million grid slots on local GPU")\n    million.add_argument("--project-root", default=".")\n    million.add_argument("--data-root")\n    million.add_argument("--workers", type=int, default=4)\n    embed = sub.add_parser("embed", help="Cache pinned frozen SigLIP 2 text embeddings")\n    embed.add_argument("--manifest", required=True)\n    embed.add_argument("--output", required=True)\n    embed.add_argument("--device", default="cpu")\n    training = sub.add_parser("train", help="Train the native glyph NCA baseline")\n    for name in ("manifest", "embeddings", "output"):\n        training.add_argument("--"+name, required=True)\n    training.add_argument("--reviews")\n    training.add_argument("--config")\n    training.add_argument("--device", default="cpu")\n    training.add_argument("--resume")\n    training.add_argument("--allow-unreviewed", action="store_true")\n    training.add_argument("--allow-fixture", action="store_true")\n    for name in ("evaluate", "export", "serve"):\n        p = sub.add_parser(name)\n        p.add_argument("--checkpoint", required=True, help="Tensor-only best.pt inference artifact")\n        p.add_argument("--allow-development", action="store_true")\n        p.add_argument("--device", default="cpu")\n        if name != "serve":\n            p.add_argument("--output", required=True)\n        if name == "evaluate":\n            p.add_argument("--prompt", required=True)\n            p.add_argument("--switch-prompt")\n            p.add_argument("--steps", type=int, default=600)\n        if name == "serve":\n            p.add_argument("--port", type=int, default=8766)\n            p.add_argument("--fps", type=float, default=24.)\n            p.add_argument("--debounce-ms", type=float, default=0., help="Text delay; zero sends every input revision immediately")\n    args = parser.parse_args(argv)\n    if args.command == "convert":\n        from .factory import build_dataset\n        result = build_dataset(args.annotations, args.frames_root, read_json(args.source), read_json(args.config), args.output, args.source_fps)\n        print(json.dumps(result["report"], ensure_ascii=False, indent=2))\n    elif args.command == "scale":\n        from .scale import build_scale\n        result = build_scale(args.annotations, args.frames_root, read_json(args.source), read_json(args.config),\n                             args.output, args.source_fps, args.workers, args.max_jobs)\n        print(json.dumps(result, ensure_ascii=False, indent=2))\n    elif args.command == "catalogue":\n        from .catalogue import Catalogue\n        with Catalogue(args.root) as catalogue:\n            if args.ingest:\n                catalogue.ingest(args.ingest)\n            if args.snapshot:\n                print(catalogue.snapshot(args.snapshot))\n            print(json.dumps(catalogue.report(), ensure_ascii=False, indent=2))\n    elif args.command == "million":\n        from .pipeline import run_million\n        run_million(args.project_root, args.data_root, args.workers)\n    elif args.command == "review":\n        from .review import review_page\n        print(review_page(args.manifest, args.output))\n    elif args.command == "embed":\n        from .text import cache_embeddings\n        print(cache_embeddings(args.manifest, args.output, args.device))\n    elif args.command == "train":\n        from .training import TrainConfig, train\n        train(args.manifest, args.embeddings, args.output, TrainConfig(**read_json(args.config)) if args.config else None,\n              reviews_path=args.reviews, device=args.device, resume=args.resume,\n              allow_unreviewed=args.allow_unreviewed, allow_fixture=args.allow_fixture)\n    elif args.command == "serve":\n        from .runtime import serve\n        serve(args.checkpoint, args.port, args.fps, args.device, args.allow_development, args.debounce_ms/1000)\n    else:\n        from .inference import load_inference\n        from .evaluation import export_onnx, measure_rollout\n        from .storage import write_json\n        model, provenance = load_inference(args.checkpoint, args.device, args.allow_development)\n        if args.command == "export":\n            print(export_onnx(model, args.output))\n        else:\n            if provenance.get("synthetic_encoder"):\n                raise ValueError("Use fixture diagnostics for synthetic text vectors; do not apply real prompts")\n            from .text import TextEncoder\n            encoder = TextEncoder(args.device)\n            prompts = [args.prompt] + ([args.switch_prompt] if args.switch_prompt else [])\n            vectors = encoder.encode(prompts)\n            result = measure_rollout(model, vectors[0], args.steps, switch_embedding=vectors[1] if len(vectors)>1 else None)\n            result["text_encoding_scope"] = "performed once per prompt outside model timing"\n            result["checkpoint_provenance"] = provenance\n            write_json(args.output, result)\n            print(json.dumps(result, ensure_ascii=False, indent=2))\n\n', 'ascii_flux/contract.py': '"""The complete representation is serialized in datasets and checkpoints."""\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport json\nimport unicodedata\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()\n\n\ndef normalize_text(text):\n    return unicodedata.normalize("NFC", str(text)).lower().strip()\n\n\n@dataclass(frozen=True)\nclass Contract:\n    version: str = "ascii-v5-ramp-rgb64-1"\n    cols: int = 64\n    rows: int = 32\n    cell_height_width: float = 2.0\n    glyphs: str = " .:+*#@"\n    glyph_version: str = "density-1"\n    palette_version: str = "rgb-cube-4-1"\n    palette: tuple = tuple((r, g, b) for r in (0, 85, 170, 255)\n                           for g in (0, 85, 170, 255) for b in (0, 85, 170, 255))\n    empty_color: int = 255\n    fps: float = 12.0\n    frames: int = 48\n\n    def __post_init__(self):\n        if min(self.cols, self.rows, self.frames) <= 0 or self.fps <= 0 or self.cell_height_width <= 0:\n            raise ValueError("Invalid grid, frame count or cadence")\n        if not self.glyphs.startswith(" ") or len(set(self.glyphs)) != len(self.glyphs) or len(self.glyphs) > 256:\n            raise ValueError("Ordered unique vocabulary must start with space")\n        if not 1 <= len(self.palette) <= 255 or self.empty_color != 255:\n            raise ValueError("Palette uses 0..254; 255 means no color")\n        if any(len(c) != 3 or any(not 0 <= x <= 255 for x in c) for c in self.palette):\n            raise ValueError("Invalid RGB palette")\n\n    def to_dict(self):\n        return json.loads(canonical(asdict(self)))\n\n    @classmethod\n    def from_dict(cls, value):\n        v = dict(value)\n        v["palette"] = tuple(tuple(c) for c in v["palette"])\n        return cls(**v)\n\n    @property\n    def hash(self):\n        return digest(self.to_dict())\n\n    def require(self, other):\n        if self.hash != Contract.from_dict(other).hash:\n            raise ValueError("Incompatible dataset/checkpoint representation; create a new version and retrain")\n\n\nDEFAULT = Contract()\nTEXT_SPEC = {\n    "model": "google/siglip2-base-patch16-224",\n    "revision": "75de2d55ec2d0b4efc50b3e9ad70dba96a7b2fa2",\n    "normalization": "NFC/lower/strip",\n    "max_length": 64,\n    "embedding": "Siglip2TextModel.pooler_output/l2",\n    "dimension": 768,\n}\n\n', 'ascii_flux/deform.py': '"""Pure PyTorch deformable convolution for the offline segmenter.\n\nThe AMD ROCm index has no torchvision build matching the installed torch, and\nthe CPU torchvision kernels cannot take GPU tensors. This implementation of\n``torchvision.ops.deform_conv2d`` (Dai et al. 2017; Zhu et al. 2019 modulation)\nuses bilinear ``grid_sample`` and is checked against torchvision on CPU.\n"""\nimport torch\nfrom torch.nn import functional as F\n\n\ndef _pair(value):\n    return (value, value) if isinstance(value, int) else tuple(value)\n\n\ndef deform_conv2d(input, offset, weight, bias=None, stride=1, padding=0, dilation=1, mask=None):\n    b, c, h, w = input.shape\n    out_channels, c_per_group, kh, kw = weight.shape\n    groups = c//c_per_group\n    sh, sw = _pair(stride)\n    ph, pw = _pair(padding)\n    dh, dw = _pair(dilation)\n    k = kh*kw\n    ho, wo = offset.shape[-2:]\n    offset_groups = offset.shape[1]//(2*k)\n    dtype = input.dtype\n    offset = offset.float().view(b, offset_groups, k, 2, ho, wo)\n    ky, kx = torch.meshgrid(torch.arange(kh, device=input.device), torch.arange(kw, device=input.device), indexing="ij")\n    oy, ox = torch.meshgrid(torch.arange(ho, device=input.device), torch.arange(wo, device=input.device), indexing="ij")\n    base_y = (oy*sh-ph)[None]+(ky.reshape(-1)*dh)[:, None, None]\n    base_x = (ox*sw-pw)[None]+(kx.reshape(-1)*dw)[:, None, None]\n    y = base_y[None, None]+offset[:, :, :, 0]\n    x = base_x[None, None]+offset[:, :, :, 1]\n    # align_corners=True maps pixel centers exactly; zero padding outside.\n    grid = torch.stack((2*x/max(w-1, 1)-1, 2*y/max(h-1, 1)-1), -1)  # [B,G,K,Ho,Wo,2]\n    grid = grid.view(b*offset_groups, k*ho, wo, 2)\n    values = input.float().view(b*offset_groups, c//offset_groups, h, w)\n    sampled = F.grid_sample(values, grid, mode="bilinear", padding_mode="zeros", align_corners=True)\n    sampled = sampled.view(b, offset_groups, c//offset_groups, k, ho, wo)\n    if mask is not None:\n        sampled = sampled*mask.float().view(b, offset_groups, 1, k, ho, wo)\n    sampled = sampled.reshape(b, groups, c_per_group*k, ho*wo)\n    weights = weight.float().view(groups, out_channels//groups, c_per_group*k)\n    out = torch.einsum("bgip,goi->bgop", sampled, weights).reshape(b, out_channels, ho, wo)\n    if bias is not None:\n        out = out+bias.float()[None, :, None, None]\n    return out.to(dtype)\n\n\ndef install():\n    """Replace torchvision\'s operator before loading remote segmentation code."""\n    import torchvision.ops\n    torchvision.ops.deform_conv2d = deform_conv2d\n', 'ascii_flux/discrete.py': '"""Uniform categorical diffusion on glyph IDs; colors are a separate head.\n\nUses the discrete posterior parameterization of Austin et al. (D3PM, 2021,\nhttps://arxiv.org/abs/2107.03006), not their architecture or reported results.\nThe glyph CE is an x0 denoising objective; no pixel/codec loss is used.\n"""\nfrom dataclasses import dataclass\nimport math\nimport torch\nfrom torch.nn import functional as F\nfrom .masked import MaskedConfig,MaskedGlyphFlow\n\n\n@dataclass(frozen=True)\nclass DiscreteConfig(MaskedConfig):\n    emergence_steps: int = 32\n    motion_steps: int = 8\n    guidance: float = 2.\n    temperature: float = 1.\n\n\ndef replace_noise(glyphs,retention,vocabulary):\n    keep = torch.rand_like(glyphs.float()) < retention\n    return torch.where(keep,glyphs,torch.randint(vocabulary,glyphs.shape,device=glyphs.device))\n\n\ndef reverse_probabilities(p0,current,alpha_current,alpha_next):\n    """Exact mixture of q(x_s | x_t,x0) for uniform forward corruption.\n\n    p0 is the model distribution over clean tokens conditional on x_t.\n    alpha values are cumulative retention, not a per-step beta.\n    """\n    if not 0 <= alpha_current <= alpha_next <= 1 or alpha_current == 1:\n        raise ValueError("Reverse diffusion requires 0 <= alpha_t <= alpha_s <= 1, alpha_t < 1")\n    k = p0.shape[1]\n    observed = F.one_hot(current.long(),k).permute(0,3,1,2).float()\n    denominator = (1-alpha_current)/k + alpha_current*observed\n    weights = p0.float()/denominator.clamp_min(1e-12)\n    prior = alpha_next*weights+(1-alpha_next)/k*weights.sum(1,keepdim=True)\n    transition = alpha_current/alpha_next if alpha_next else 0.\n    likelihood = (1-transition)/k+transition*observed\n    posterior = (prior*likelihood).clamp_min(0.)\n    return posterior/posterior.sum(1,keepdim=True).clamp_min(1e-12)\n\n\nclass DiscreteGlyphFlow(MaskedGlyphFlow):\n    """Hidden noisy glyphs + displayed glyph/color grid + previous native grid."""\n    def __init__(self,contract,config=None):\n        super().__init__(contract,config or DiscreteConfig())\n\n    @staticmethod\n    def retention(mode,phase,total):\n        base = .65 if mode == 1 else .05 if mode == 2 else .0001\n        return base+(1-base)*math.sin(phase/total*math.pi/2)**2\n\n    def initial_state(self,batch=1,noisy=False):\n        state = self.coordinates.new_zeros((batch,7,self.contract.rows,self.contract.cols))\n        state[:,0] = torch.randint(len(self.contract.glyphs),state[:,0].shape,device=state.device)\n        state[:,2] = self.contract.empty_color\n        state[:,3] = self.mask_glyph;state[:,4] = self.mask_color\n        return state\n\n    def tokens(self,state):\n        g,c = state[:,1].long(),state[:,2].long()\n        return g,torch.where(g == 0,self.contract.empty_color,c)\n\n    def recondition(self,state):\n        result = state.clone()\n        result[:,3:5] = state[:,1:3]\n        result[:,0] = replace_noise(state[:,1].long(),self.retention(2,0,self.config.emergence_steps),len(self.contract.glyphs))\n        result[:,5] = 0;result[:,6] = 2\n        return result\n\n    def step(self,state,text):\n        state = state.clone()\n        phase,mode = int(state[0,5,0,0]),int(state[0,6,0,0])\n        total = self.config.motion_steps if mode == 1 else self.config.emergence_steps\n        if phase >= total:\n            state[:,3:5] = state[:,1:3]\n            state[:,0] = replace_noise(state[:,1].long(),self.retention(1,0,self.config.motion_steps),len(self.contract.glyphs))\n            state[:,5] = 0;state[:,6] = 1\n            phase,mode,total = 0,1,self.config.motion_steps\n        a,b = self.retention(mode,phase,total),self.retention(mode,phase+1,total)\n        g = state[:,0].long();c = torch.full_like(g,self.mask_color)\n        pg,pc = state[:,3].long(),state[:,4].long()\n        ratio = text.new_full((len(state),),1-a)\n        modes = torch.full((len(state),),mode,device=state.device,dtype=torch.long)\n        if self.config.guidance != 1:\n            gl,cl = self(torch.cat([g,g]),torch.cat([c,c]),torch.cat([pg,pg]),torch.cat([pc,pc]),\n                         torch.cat([text,torch.zeros_like(text)]),torch.cat([ratio,ratio]),torch.cat([modes,modes]))\n            gl,ugl = gl.chunk(2);cl,ucl = cl.chunk(2)\n            gl = ugl+self.config.guidance*(gl-ugl);cl = ucl+self.config.guidance*(cl-ucl)\n        else:gl,cl = self(g,c,pg,pc,text,ratio,modes)\n        p0 = F.softmax(gl.float()/self.config.temperature,1)\n        posterior = reverse_probabilities(p0,g,a,b)\n        random = torch.rand_like(posterior).clamp_(1e-6,1-1e-6)\n        state[:,0] = (posterior.clamp_min(1e-12).log()-torch.log(-torch.log(random))).argmax(1)\n        # The displayed grid is the model\'s clean-token readout, while noisy\n        # categorical tokens remain hidden. No raster or glyph converter.\n        display = gl.argmax(1)\n        state[:,1] = display\n        state[:,2] = torch.where(display != 0,cl.argmax(1),self.contract.empty_color)\n        state[:,5] = phase+1\n        return state,gl,cl\n', 'ascii_flux/discrete_training.py': '"""Real-data glyph diffusion experiment; auxiliary x0 CE, no pixel loss."""\nimport copy\nfrom dataclasses import asdict\nimport hashlib\nimport math\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom .causal_training import prepare\nfrom .contract import TEXT_SPEC\nfrom .discrete import DiscreteConfig,DiscreteGlyphFlow,replace_noise\nfrom .masked import masked_loss\nfrom .masked_training import MaskedTrainConfig,subject_groups,token_batch\nfrom .storage import atomic_file,write_json\nfrom .training import Corpus,restore_rng,rng_state\n\n\ndef inputs_for(model,batch,validation=False):\n    g,c,pg,pc,text,mode = prepare(model,batch,validation)\n    t = torch.rand(len(g),device=g.device)*.978+.02\n    alpha = torch.cos(t*math.pi/2).square()\n    noisy = replace_noise(g,alpha[:,None,None],len(model.contract.glyphs))\n    context = (torch.rand_like(g.float()) < .1) & (mode[:,None,None] == 1)\n    pg = pg.masked_fill(context,model.mask_glyph)\n    pc = pc.masked_fill(context,model.mask_color)\n    return noisy,torch.full_like(c,model.mask_color),pg,pc,text,1-alpha,mode\n\n\n@torch.inference_mode()\ndef validate(model,corpus,groups,config,device):\n    saved = rng_state();model.eval()\n    try:\n        torch.manual_seed(config.seed+1);rng = random.Random(config.seed+1)\n        losses = []\n        for _ in range(config.val_batches):\n            batch = token_batch(corpus,groups,config,\'val\',device,rng)\n            inputs = inputs_for(model,batch,True)\n            with torch.autocast(\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n                gl,cl = model(*inputs)\n            losses.append(float(masked_loss(gl,cl,batch[0],batch[1],torch.ones_like(batch[0],dtype=torch.bool))))\n        return float(np.mean(losses))\n    finally:\n        restore_rng(saved);model.train()\n\n\ndef train_discrete(manifest,embeddings,reviews,output,config=None,device=\'cuda\',resume=None,warm_start=None):\n    config = config or MaskedTrainConfig()\n    if min(config.steps,config.batch_size,config.validate_every,config.val_batches) <= 0 or config.lr <= 0 or config.memorize_one_frame:\n        raise ValueError(\'Positive real-data experiment configuration required\')\n    corpus = Corpus(manifest,embeddings,reviews)\n    groups = subject_groups(corpus,config.subjects)\n    output = Path(output);output.mkdir(parents=True,exist_ok=True)\n    if (output/\'latest.pt\').exists() and not resume:raise FileExistsError(\'Resume or choose another diffusion run\')\n    random.seed(config.seed);np.random.seed(config.seed);torch.manual_seed(config.seed)\n    model_config = DiscreteConfig()\n    model = DiscreteGlyphFlow(corpus.contract,model_config).to(device)\n    initialization = None\n    if warm_start:\n        data = Path(warm_start).read_bytes()\n        ck = torch.load(warm_start,map_location=\'cpu\',weights_only=True)\n        corpus.contract.require(ck[\'contract\'])\n        if (ck[\'format\'] != \'ascii-flux-v5-masked-1\' or ck[\'provenance\'][\'dataset_fingerprint\'] != corpus.manifest[\'fingerprint\']\n                or ck[\'provenance\'][\'training_concepts\'] != list(config.subjects) or ck[\'provenance\'][\'synthetic_encoder\']):\n            raise ValueError(\'Warm-start data/concept provenance mismatch\')\n        model.load_state_dict(ck[\'model\'])\n        initialization = {\'checkpoint_sha256\':hashlib.sha256(data).hexdigest(),\'step\':ck[\'step\'],\n                          \'format\':ck[\'format\'],\'optimizer_restarted\':True}\n    optimizer = torch.optim.AdamW(model.parameters(),lr=config.lr,weight_decay=.01)\n    scaler = torch.amp.GradScaler(\'cuda\',enabled=str(device).startswith(\'cuda\'),init_scale=128.)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    code = hashlib.sha256()\n    for name in (\'discrete.py\',\'discrete_training.py\',\'masked.py\',\'masked_training.py\',\'causal_training.py\',\'text.py\'):\n        code.update(Path(__file__).with_name(name).read_bytes())\n    provenance = {\'dataset_fingerprint\':corpus.manifest[\'fingerprint\'],\'text_spec\':TEXT_SPEC,\n        \'allow_fixture\':False,\'allow_unreviewed\':False,\'synthetic_encoder\':False,\'development_only\':True,\n        \'recognition_validated\':False,\'training_concepts\':list(config.subjects),\'source_conditions\':corpus.manifest[\'source\'],\n        \'review_scope\':corpus.review_scope,\'training_code_sha256\':code.hexdigest(),\'torch_version\':str(torch.__version__),\n        \'scope\':\'six-concept uniform glyph diffusion; recognition and motion unvalidated\',\'initialization\':initialization}\n    start,best,best_step,best_model,history,skipped = 0,math.inf,0,None,[],0\n    if resume:\n        ck = torch.load(resume,map_location=\'cpu\',weights_only=False);corpus.contract.require(ck[\'contract\'])\n        old,new = dict(ck[\'train_config\']),asdict(config);old.pop(\'steps\');new.pop(\'steps\')\n        if ck[\'provenance\'] != provenance or old != new or ck[\'model_config\'] != asdict(model_config):\n            raise ValueError(\'Diffusion resume provenance/config mismatch\')\n        model.load_state_dict(ck[\'model\']);ema.load_state_dict(ck[\'ema\'])\n        optimizer.load_state_dict(ck[\'optimizer\']);scaler.load_state_dict(ck[\'scaler\'])\n        start,best,best_step,best_model,history,skipped = (ck[k] for k in (\'step\',\'best_val\',\'best_step\',\'best_model\',\'history\',\'skipped_amp_updates\'))\n        restore_rng(ck[\'rng\'])\n    started = time.monotonic()\n    print({\'parameters\':sum(p.numel() for p in model.parameters()),\'scope\':provenance[\'scope\']},flush=True)\n    for step in range(start,config.steps):\n        batch = token_batch(corpus,groups,config,\'train\',device,random)\n        inputs = inputs_for(model,batch)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(\'cuda\',dtype=torch.float16,enabled=str(device).startswith(\'cuda\')):\n            gl,cl = model(*inputs)\n            loss = masked_loss(gl,cl,batch[0],batch[1],torch.ones_like(batch[0],dtype=torch.bool))\n        if not torch.isfinite(loss):raise FloatingPointError(\'Non-finite diffusion CE\')\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=not scaler.is_enabled())\n        scaler.step(optimizer);scaler.update()\n        if math.isfinite(float(norm)):\n            decay = min(.995,(1+step)/(10+step))\n            with torch.no_grad():\n                for a,b in zip(ema.parameters(),model.parameters()):a.lerp_(b,1-decay)\n        else:skipped += 1\n        if (step+1)%25 == 0:\n            write_json(output/\'progress.json\',{\'step\':step+1,\'total_steps\':config.steps,\'loss\':float(loss.detach()),\n                \'elapsed_seconds\':time.monotonic()-started,\'optimizer_updates\':step+1-skipped,\'recognition_validated\':False})\n        if (step+1)%config.validate_every == 0 or step+1 == config.steps:\n            value = validate(ema,corpus,groups,config,device)\n            if not math.isfinite(value):raise FloatingPointError(\'Non-finite diffusion validation\')\n            if value < best:\n                best,best_step = value,step+1\n                best_model = copy.deepcopy({k:v.detach().cpu() for k,v in ema.state_dict().items()})\n            row = {\'step\':step+1,\'train_x0_ce\':float(loss.detach()),\'val_x0_ce\':value,\'best_val_ce\':best,\n                   \'optimizer_updates\':step+1-skipped};history.append(row)\n            ck = {\'format\':\'ascii-flux-v5-discrete-1\',\'contract\':corpus.contract.to_dict(),\'model_config\':asdict(model_config),\n                \'train_config\':asdict(config),\'provenance\':provenance,\'model\':model.state_dict(),\'ema\':ema.state_dict(),\n                \'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'step\':step+1,\'best_val\':best,\'best_step\':best_step,\n                \'best_model\':best_model,\'rng\':rng_state(),\'history\':history,\'skipped_amp_updates\':skipped}\n            with atomic_file(output/\'latest.pt\') as stream:torch.save(ck,stream)\n            with atomic_file(output/\'best.pt\') as stream:\n                torch.save({k:ck[k] for k in (\'format\',\'contract\',\'model_config\',\'provenance\')} | {\'model\':best_model,\'step\':best_step},stream)\n            write_json(output/\'history.json\',history);print(row,flush=True)\n    return model\n', 'ascii_flux/evaluation.py': '"""Instrumental measurements; semantic recognition requires a human study."""\nimport time\nimport numpy as np\nimport torch\nfrom .storage import write_json\n\n\n@torch.inference_mode()\ndef measure_rollout(model, embedding, steps=600, warmup=10, switch_embedding=None, switch_at=None):\n    if steps < 500:\n        raise ValueError("Stability diagnostics require at least 500 steps")\n    device = next(model.parameters()).device\n    text = torch.as_tensor(embedding, dtype=torch.float32, device=device).reshape(1, -1)\n    other = None if switch_embedding is None else torch.as_tensor(switch_embedding, dtype=torch.float32, device=device).reshape(1, -1)\n    model.eval()\n    state = model.initial_state()\n    for _ in range(warmup):\n        state, _, _ = model.step(state, text)\n    state = model.initial_state()\n    latencies, changes, occupancy, norms, grids, transitions = [], [], [], [], set(), []\n    previous = None\n    finite = True\n    for i in range(steps):\n        condition = other if other is not None and i >= (switch_at or steps//2) else text\n        if other is not None and i == (switch_at or steps//2) and hasattr(model,"recondition"):\n            state = model.recondition(state)\n        if str(device).startswith("cuda"):\n            torch.cuda.synchronize()\n        begin = time.perf_counter()\n        state, _, _ = model.step(state, condition)\n        glyphs, _ = model.tokens(state)\n        if str(device).startswith("cuda"):\n            torch.cuda.synchronize()\n        latencies.append((time.perf_counter()-begin)*1000)\n        if not torch.isfinite(state).all():\n            finite = False\n            break\n        g = glyphs.cpu().numpy()[0]\n        grids.add(g.tobytes())\n        occupancy.append(float((g != 0).mean()))\n        norms.append(float(state.abs().max()))\n        if previous is not None:\n            change = int((g != previous).sum())\n            changes.append(change)\n            transitions.append(change > 0)\n        previous = g\n    result = {"scope": "instrumental rollout diagnostics; not human recognition", "steps_requested": steps,\n              "steps_measured": len(latencies), "device": str(device), "torch_threads": torch.get_num_threads(),\n              "step_and_readout_ms_median": float(np.median(latencies)), "step_and_readout_ms_p95": float(np.percentile(latencies, 95)),\n              "measured_max_steps_per_second": 1000/float(np.mean(latencies)),\n              "glyph_changes_per_step_mean": float(np.mean(changes)) if changes else 0.,\n              "distinct_updates_per_step": float(np.mean(transitions)) if transitions else 0.,\n              "unique_grid_count": len(grids), "occupancy_first": occupancy[0] if occupancy else None,\n              "occupancy_last": occupancy[-1] if occupancy else None,\n              "state_abs_max": max(norms) if norms else None, "finite": finite,\n              "frozen_last_100_steps": not any(changes[-100:]) if len(changes) >= 100 else None,\n              "human_recognition": "not measured", "time_to_readable_seconds": None,\n              "semantic_transition_seconds": None, "transition_continuity": "not measured",\n              "ten_minute_stability": "not established by this step-count diagnostic"}\n    return result\n\n\ndef export_onnx(model, output):\n    """One recurrent step, including both token heads; noise and mask are inputs."""\n    if not hasattr(model.config,"update_probability"):\n        raise ValueError("This experimental architecture has no validated recurrent ONNX export yet")\n    import onnxruntime as ort\n    from .storage import atomic_file\n    model = model.cpu().eval()\n    state = model.initial_state()\n    text = torch.zeros(1, model.config.text_dim)\n    noise = torch.randn(1, 1, model.contract.rows, model.contract.cols)\n    mask = (torch.rand_like(noise) < model.config.update_probability).float()\n    with torch.inference_mode():\n        expected = model(state, text, noise, mask)\n    # Fixed grid/batch initially. External inputs provide all stochasticity.\n    import io\n    buffer = io.BytesIO()\n    torch.onnx.export(model, (state, text, noise, mask), buffer, input_names=["state", "text", "noise", "update_mask"],\n                      output_names=["next_state", "glyph_logits", "color_logits"], opset_version=17, dynamo=False)\n    session = ort.InferenceSession(buffer.getvalue(), providers=["CPUExecutionProvider"])\n    actual = session.run(None, {k: v.numpy() for k, v in zip(["state", "text", "noise", "update_mask"], (state, text, noise, mask))})\n    errors = [float(np.max(np.abs(a-e.numpy()))) for a, e in zip(actual, expected)]\n    if any(not np.allclose(a, e.numpy(), rtol=1e-4, atol=1e-5) for a, e in zip(actual, expected)):\n        raise ValueError(f"ONNX numerical parity failed: {errors}")\n    with atomic_file(output) as stream:\n        stream.write(buffer.getvalue())\n    write_json(str(output)+".json", {"contract": model.contract.to_dict(), "model_config": model.config_dict(),\n                                   "onnx_max_abs_errors": errors, "scope": "export parity, not browser performance"})\n    return errors\n', 'ascii_flux/factory.py': '"""YouTube-VIS adapter and auditable D1 run. Reads local media only."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport tempfile\nimport numpy as np\nfrom PIL import Image\nfrom .contract import DEFAULT, digest, normalize_text\nfrom .offline import ConverterConfig, convert_clip, converter_identity, quality_rejections\nfrom .storage import write_json, write_shard\n\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for part in iter(lambda: stream.read(1024*1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef validate_source(source):\n    for key in ("source", "snapshot", "url", "terms_url", "terms_checked_on", "annotations_license", "media_conditions", "local_media_provenance"):\n        value = source.get(key, "")\n        if not value or "REPLACE_" in value:\n            raise ValueError(f"Complete source provenance before conversion: {key}")\n    if source.get("source") != "YouTube-VIS" or source.get("usage_scope") != "non-commercial-research":\n        raise ValueError("This adapter is restricted to the recorded YouTube-VIS research conditions")\n\n\ndef decode_mask(segmentation, height, width):\n    if not segmentation:\n        return np.zeros((height, width), np.uint8)\n    if isinstance(segmentation, dict) and isinstance(segmentation.get("counts"), list):\n        if segmentation.get("size", [height, width]) != [height, width]:\n            raise ValueError("invalid_mask_dimensions")\n        counts = segmentation["counts"]\n        if any(not isinstance(n, int) or n < 0 for n in counts) or sum(counts) != height*width:\n            raise ValueError("invalid_mask_rle")\n        flat = np.zeros(height*width, np.uint8)\n        pos, bit = 0, 0\n        for length in counts:\n            flat[pos:pos+length] = bit\n            pos += length\n            bit = 1-bit\n        return flat.reshape((height, width), order="F")\n    try:\n        from pycocotools import mask as mask_utils\n    except ImportError as exc:\n        raise ImportError("Compressed YouTube-VIS RLE requires pip install \'.[vis]\'") from exc\n    if isinstance(segmentation, list):\n        segmentation = mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))\n    result = mask_utils.decode(segmentation)\n    if result.shape != (height, width):\n        raise ValueError("invalid_mask_dimensions")\n    return result.astype(np.uint8)\n\n\ndef split_videos(videos, annotations, categories, heldout, seed):\n    missing = set(heldout) - set(categories.values())\n    if missing:\n        raise ValueError(f"Unknown held-out categories in this release: {sorted(missing)}")\n    reserved = {a["video_id"] for a in annotations if categories[a["category_id"]] in heldout}\n    result = {}\n    for v in videos:\n        if v in reserved:\n            result[v] = "test_unseen"\n        else:\n            number = int(hashlib.sha256(f"{seed}:YouTube-VIS:{v}".encode()).hexdigest()[:12], 16) / 16**12\n            result[v] = "train" if number < .8 else "val" if number < .9 else "test_seen"\n    return result\n\n\ndef build_dataset(annotations_path, frames_root, source, config, output, source_fps, contract=DEFAULT):\n    validate_source(source)\n    if not np.isfinite(source_fps) or source_fps <= 0:\n        raise ValueError("Supply the actual cadence of the extracted annotated frames, not the original video\'s FPS")\n    if config["max_clips"] <= 0 or config["shard_size"] <= 0:\n        raise ValueError("Invalid run limits")\n    output = Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f"Dataset runs are immutable: {output}")\n    root = Path(frames_root).resolve()\n    data = json.loads(Path(annotations_path).read_text(encoding="utf-8"))\n    videos = {v["id"]: v for v in data["videos"]}\n    categories = {c["id"]: c["name"] for c in data["categories"]}\n    partitions = split_videos(videos, data["annotations"], categories, config["heldout_categories"], config["seed"])\n    converter = ConverterConfig(**config["converter"])\n    identity = converter_identity(converter)\n    annotation_hash = file_sha256(annotations_path)\n    # Round-robin categories after seeded shuffling, rather than taking the first 200 annotations.\n    grouped = defaultdict(list)\n    for ann in data["annotations"]:\n        grouped[ann["category_id"]].append(ann)\n    rng = random.Random(config["seed"])\n    for items in grouped.values():\n        rng.shuffle(items)\n    ordered = []\n    while any(grouped.values()):\n        for category in sorted(grouped):\n            if grouped[category]:\n                ordered.append(grouped[category].pop())\n    output.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix=output.name + ".", dir=output.parent))\n    records, rejects, pending = [], [], []\n    shard_number = 0\n\n    def flush():\n        nonlocal shard_number\n        if pending:\n            write_shard(staging / f"shard-{shard_number:05d}.tar", pending, contract)\n            pending.clear()\n            shard_number += 1\n\n    try:\n        attempted = 0\n        for ann in ordered:\n            v = videos[ann["video_id"]]\n            n = len(v["file_names"])\n            # Only full real-duration clips. Never silently loop or pad a short video.\n            duration = n/source_fps\n            starts = np.arange(0, max(0., duration-contract.frames/contract.fps)+1e-8, contract.frames/contract.fps)\n            if duration + 1e-8 < contract.frames/contract.fps:\n                rejects.append({"video_id": str(v["id"]), "instance_id": str(ann["id"]), "source": source["source"], "reasons": ["too_short_for_48_frames"]})\n                continue\n            for start in starts:\n                if attempted >= config["max_clips"]:\n                    break\n                attempted += 1\n                clip_id = f"v{v[\'id\']}-i{ann[\'id\']}-t{round(start*1000)}"\n                base = {"clip_id": clip_id, "video_id": str(v["id"]), "instance_id": str(ann["id"]),\n                        "source": source["source"], "category": categories[ann["category_id"]], "split": partitions[v["id"]]}\n                indices = np.floor((start+np.arange(contract.frames)/contract.fps)*source_fps + 1e-7).astype(int)\n                try:\n                    frames, masks, hashes = [], [], {}\n                    for index in indices:\n                        path = (root / v["file_names"][index]).resolve()\n                        if not path.is_relative_to(root):\n                            raise ValueError("media_path_escapes_root")\n                        with Image.open(path) as image:\n                            frame = np.array(image.convert("RGB"))\n                        if frame.shape[:2] != (v["height"], v["width"]):\n                            raise ValueError("source_dimensions_mismatch")\n                        if index >= len(ann["segmentations"]):\n                            raise ValueError("missing_annotation_frame")\n                        frames.append(frame)\n                        masks.append(decode_mask(ann["segmentations"][index], v["height"], v["width"]))\n                        hashes[str(index)] = file_sha256(path)\n                    glyphs, colors, boxes, metrics = convert_clip(frames, masks, contract, converter)\n                    reasons = quality_rejections(metrics, config["quality"])\n                    if reasons:\n                        rejects.append({**base, "reasons": reasons, "quality": metrics})\n                        continue\n                    label = normalize_text(base["category"])\n                    metadata = {**base, "provenance": source, "annotation_sha256": annotation_hash,\n                                "source_frame_sha256": hashes, "source_frame_indices": indices.tolist(),\n                                "source_frame_fps": source_fps, "sampled_unique_source_frames": len(set(indices.tolist())),\n                                "source_frame_paths": [v["file_names"][index] for index in indices],\n                                "source_frame_indices_30fps": [v["source_frame_indices_30fps"][index] for index in indices] if "source_frame_indices_30fps" in v else None,\n                                "original_sequence_id": v.get("original_sequence_id"),\n                                "source_object_id": ann.get("source_object_id"),\n                                "source_url": v.get("url", source["url"]), "start_seconds": float(start),\n                                "contract": contract.to_dict(), "contract_hash": contract.hash,\n                                "converter": identity, "quality": metrics, "mask_method": "source-instance-mask",\n                                "original_label": base["category"], "original_caption": v.get("caption"),\n                                "normalized_label": label, "label_scope": "subject only; action not annotated by YouTube-VIS",\n                                "rewrite": None, "synthetic": bool(source.get("synthetic", False)), "human_review": "pending",\n                                "box_format": "xyxy source pixels; outside-image padding permitted"}\n                    pending.append((clip_id, glyphs, colors, boxes, metadata))\n                    records.append({**base, "shard": f"shard-{shard_number:05d}.tar", "normalized_label": label,\n                                    "synthetic": bool(source.get("synthetic", False)), "quality": metrics})\n                    if len(pending) >= config["shard_size"]:\n                        flush()\n                except (ValueError, OSError, IndexError) as exc:\n                    rejects.append({**base, "reasons": [type(exc).__name__ + ":" + str(exc)]})\n            if attempted >= config["max_clips"]:\n                break\n        flush()\n        counts = Counter(r["split"] for r in records)\n        report = {"source": source, "attempted_full_clips": attempted, "machine_accepted": len(records),\n                  "rejected_records": len(rejects), "yield_full_clips": len(records)/max(1, attempted),\n                  "yield_including_short_rejections": len(records)/max(1, len(records)+len(rejects)),\n                  "split_counts": dict(counts), "reasons": dict(Counter(reason for r in rejects for reason in r["reasons"])),\n                  "quality_thresholds_calibrated": config["quality"].get("calibrated", False),\n                  "human_review_completed": False, "D1_complete": False,\n                  "status": "candidates for human review; not a validated corpus"}\n        manifest = {"dataset_version": config["dataset_version"], "contract": contract.to_dict(),\n                    "contract_hash": contract.hash, "source": source, "converter": identity,\n                    "annotation_sha256": annotation_hash, "config": config, "clips": records,\n                    "shards": {p.name: file_sha256(p) for p in staging.glob("*.tar")},\n                    "video_splits": {str(k): v for k, v in partitions.items()}, "report": report}\n        manifest["fingerprint"] = digest(manifest)\n        write_json(staging/"manifest.json", manifest)\n        write_json(staging/"rejections.json", rejects)\n        write_json(staging/"report.json", report)\n        write_json(staging/"reviews.template.json", {"dataset_fingerprint": manifest["fingerprint"], "reviewer": "",\n                    "converter_hash": identity["code_sha256"], "decisions": {r["clip_id"]: {"decision": "pending", "reason": ""} for r in records}})\n        if output.exists():\n            raise FileExistsError(f"Output appeared while building this immutable run: {output}")\n        os.replace(staging, output)\n        return manifest\n    except BaseException:\n        resolved = staging.resolve()\n        if resolved.parent == output.parent and resolved.name.startswith(output.name + "."):\n            shutil.rmtree(resolved)\n        raise\n', 'ascii_flux/inference.py': '"""Load native model experiments without changing the NCA training identity."""\nimport torch\nfrom .contract import Contract,TEXT_SPEC\n\n\ndef load_inference(path,device="cpu",allow_development=False):\n    checkpoint = torch.load(path,map_location="cpu",weights_only=True)\n    if checkpoint["format"] == "ascii-flux-v5-1":\n        from .training import load_inference as load_nca\n        return load_nca(path,device,allow_development)\n    if checkpoint["format"] not in ("ascii-flux-v5-masked-1","ascii-flux-v5-causal-1","ascii-flux-v5-discrete-1"):\n        raise ValueError("Unknown checkpoint format")\n    provenance = checkpoint["provenance"]\n    if provenance["text_spec"] != TEXT_SPEC:\n        raise ValueError("Text representation mismatch")\n    if (provenance["allow_fixture"] or provenance["allow_unreviewed"] or provenance.get("development_only")) and not allow_development:\n        raise ValueError("Development checkpoint: explicitly enable development mode")\n    contract = Contract.from_dict(checkpoint["contract"])\n    if checkpoint["format"] == "ascii-flux-v5-discrete-1":\n        from .discrete import DiscreteConfig,DiscreteGlyphFlow\n        model = DiscreteGlyphFlow(contract,DiscreteConfig(**checkpoint["model_config"]))\n    elif checkpoint["format"] == "ascii-flux-v5-masked-1":\n        from .masked import MaskedConfig,MaskedGlyphFlow\n        model = MaskedGlyphFlow(contract,MaskedConfig(**checkpoint["model_config"]))\n    else:\n        from .autoregressive import CausalConfig,CausalGlyphFlow\n        model = CausalGlyphFlow(contract,CausalConfig(**checkpoint["model_config"]))\n    model.load_state_dict(checkpoint["model"])\n    return model.to(device).eval(),provenance\n', 'ascii_flux/live.py': '"""Live full-screen demo of the cascade model: one sampling step per frame.\n\nThe grid is the model state. A prompt revision is encoded once (separate\nthread, short debounce) and re-conditions the current grid without clearing\nit: part of the silhouette is re-masked and the schedule resumes, so the old\nshape turns into the new one. Once a shape is complete the grid holds still;\nno motion has been learned yet. Browser receives tokens over Server-Sent Events.\n"""\nimport base64\nimport json\nfrom pathlib import Path\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nimport torch\nfrom .contract import normalize_text\n\n\nclass LiveCascade:\n    def __init__(self, model, encoder, fps=24., debounce=.12):\n        self.model, self.encoder, self.fps, self.debounce = model.eval(), encoder, fps, debounce\n        self.device = next(model.parameters()).device\n        c, n = model.config, model.ph*model.pw\n        self.s_total, self.g_total = c.silhouette_steps, c.glyph_steps\n        self.colon = model.contract.glyphs.index(":")\n        self.dim_color = len(model.contract.palette)-1\n        self.lock = threading.Condition()\n        self.stopped = threading.Event()\n        self.prompt, self.revision, self.revised_at = "", 0, 0.\n        self.pending_text, self.text = None, None\n        self.version, self.frame = 0, None\n        self.silhouette = torch.zeros((1, n), dtype=torch.long, device=self.device)\n        self.display_g = torch.zeros((1, model.contract.rows, model.contract.cols), dtype=torch.long, device=self.device)\n        self.display_c = torch.full_like(self.display_g, model.contract.empty_color)\n        self.phase, self.step = "idle", 0\n        self._publish()\n\n    # ----- prompt handling -----\n    def set_prompt(self, prompt):\n        prompt = normalize_text(prompt)[:200]\n        with self.lock:\n            if prompt != self.prompt:\n                self.prompt, self.revised_at = prompt, time.monotonic()\n                self.revision += 1\n                self.lock.notify_all()\n\n    def _encode_loop(self):\n        done = 0\n        while not self.stopped.is_set():\n            with self.lock:\n                wait = self.debounce-(time.monotonic()-self.revised_at)\n                if done == self.revision or wait > 0:\n                    self.lock.wait(.05 if done == self.revision else wait)\n                    continue\n                revision, prompt = self.revision, self.prompt\n            done = revision\n            text = None if not prompt else torch.tensor(\n                self.encoder.encode_tokens([prompt], self.model.config.text_tokens), device=self.device).float()\n            with self.lock:\n                if revision == self.revision:\n                    self.pending_text = ("clear",) if text is None else ("text", text)\n\n    # ----- model clock -----\n    def _recondition(self, text):\n        """New concept: the model redraws its silhouette from scratch while the\n        displayed grid is kept; cells change only as the model decides them."""\n        self.text = text\n        n = self.silhouette.shape[1]\n        self.silhouette = torch.full((1, n), 2, dtype=torch.long, device=self.device)\n        self.s_masked = torch.ones((1, n), dtype=torch.bool, device=self.device)\n        self.step = 0\n        self.phase = "silhouette"\n\n    @torch.inference_mode()\n    def tick(self):\n        with self.lock:\n            pending, self.pending_text = self.pending_text, None\n        if pending is not None:\n            if pending[0] == "clear":\n                self.phase, self.text = "idle", None\n                self.silhouette.zero_()\n                self.display_g.zero_()\n                self.display_c.fill_(self.model.contract.empty_color)\n                self._publish()\n                return\n            self._recondition(pending[1])\n        m = self.model\n        if self.phase == "silhouette":\n            self.silhouette, self.s_masked = m.silhouette_step(self.text, self.silhouette, self.s_masked, self.step, self.s_total)\n            self.step += 1\n            cells = lambda value: m.grid((self.silhouette == value)[..., None].expand(-1, -1, m.cells))\n            occupied, vacated = cells(1), cells(0)\n            # Undecided cells keep what is shown; cells decided empty vanish;\n            # cells decided occupied keep their glyph or appear as a faint fill.\n            self.display_g = torch.where(vacated, 0, torch.where(occupied & (self.display_g == 0), self.colon, self.display_g))\n            self.display_c = torch.where(self.display_g == 0, m.contract.empty_color,\n                                         torch.where(self.display_c == m.contract.empty_color, self.dim_color, self.display_c))\n            if self.step >= self.s_total:\n                self.glyphs, self.colors, self.g_masked = m.start_glyphs(self.silhouette)\n                self.phase, self.step = "glyphs", 0\n        elif self.phase == "glyphs":\n            self.glyphs, self.colors, self.g_masked = m.glyph_step(self.text, self.glyphs, self.colors, self.silhouette,\n                                                                   self.g_masked, self.step, self.g_total)\n            self.step += 1\n            g, c = m._render(self.silhouette, self.glyphs, self.colors)\n            decided = m.grid((~self.g_masked)[..., None].expand(-1, -1, m.cells)) & (m.grid((self.silhouette == 1)[..., None].expand(-1, -1, m.cells)))\n            self.display_g = torch.where(decided, g, self.display_g)\n            self.display_c = torch.where(decided, c, self.display_c)\n            if self.step >= self.g_total:\n                self.display_g, self.display_c = g, c\n                self.phase = "idle"\n        else:\n            return\n        self._publish()\n\n    def _publish(self):\n        glyphs = "".join(self.model.contract.glyphs[i] for i in self.display_g[0].flatten().tolist())\n        colors = base64.b64encode(bytes(self.display_c[0].flatten().to(torch.uint8).tolist())).decode()\n        with self.lock:\n            self.version += 1\n            self.frame = json.dumps({"v": self.version, "g": glyphs, "c": colors})\n            self.lock.notify_all()\n\n    def _clock(self):\n        deadline = time.monotonic()\n        while not self.stopped.is_set():\n            self.tick()\n            deadline = max(deadline+1/self.fps, time.monotonic())\n            self.stopped.wait(max(0., deadline-time.monotonic()))\n\n    def start(self):\n        for target in (self._encode_loop, self._clock):\n            threading.Thread(target=target, daemon=True).start()\n\n    def wait_frame(self, after, timeout=1.):\n        with self.lock:\n            if self.version == after:\n                self.lock.wait(timeout)\n            return self.version, self.frame\n\n\ndef make_server(live, port):\n    page = Path(__file__).with_name("live.html").read_bytes()\n    contract = live.model.contract\n    config = json.dumps({"rows": contract.rows, "cols": contract.cols, "cell": contract.cell_height_width,\n                         "palette": [list(c) for c in contract.palette], "empty": contract.empty_color}).encode()\n\n    class Handler(BaseHTTPRequestHandler):\n        protocol_version = "HTTP/1.1"\n\n        def log_message(self, *_):\n            pass\n\n        def _send(self, status, body, kind):\n            self.send_response(status)\n            self.send_header("Content-Type", kind)\n            self.send_header("Content-Length", str(len(body)))\n            self.send_header("Cache-Control", "no-store")\n            self.end_headers()\n            self.wfile.write(body)\n\n        def do_GET(self):\n            if self.path == "/":\n                self._send(200, page, "text/html; charset=utf-8")\n            elif self.path == "/config":\n                self._send(200, config, "application/json")\n            elif self.path == "/stream":\n                self.send_response(200)\n                self.send_header("Content-Type", "text/event-stream")\n                self.send_header("Cache-Control", "no-store")\n                self.send_header("Connection", "close")\n                self.end_headers()\n                self.close_connection = True\n                seen = -1\n                try:\n                    while not live.stopped.is_set():\n                        version, frame = live.wait_frame(seen)\n                        if version != seen:\n                            seen = version\n                            self.wfile.write(f"data: {frame}\\n\\n".encode())\n                            self.wfile.flush()\n                except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):\n                    pass\n            else:\n                self._send(404, b"{}", "application/json")\n\n        def do_POST(self):\n            origin = self.headers.get("Origin")\n            if self.path != "/prompt" or (origin and origin != f"http://{self.headers.get(\'Host\')}"):\n                self._send(403, b"{}", "application/json")\n                return\n            length = int(self.headers.get("Content-Length", "0"))\n            try:\n                value = json.loads(self.rfile.read(min(length, 4000)))\n                live.set_prompt(str(value.get("prompt", "")))\n                self._send(200, b"{}", "application/json")\n            except ValueError:\n                self._send(400, b"{}", "application/json")\n\n    return ThreadingHTTPServer(("127.0.0.1", port), Handler)\n\n\ndef serve(checkpoint, port=8767, device="cuda", fps=24., on_ready=None):\n    from .cascade import CascadeConfig, CascadeGlyphModel\n    from .contract import Contract\n    from .text import TextEncoder\n    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    model = CascadeGlyphModel(Contract.from_dict(saved["contract"]), CascadeConfig(**saved["model_config"]))\n    model.load_state_dict(saved["model"])\n    encoder = TextEncoder(device)\n    model = model.to(device).eval()\n    # Warm-up: compile GPU kernels before the first prompt, and time a step.\n    begin = time.perf_counter()\n    with torch.inference_mode():\n        model.sample(torch.tensor(encoder.encode_tokens(["warm up"], model.config.text_tokens), device=device).float())\n    print(f"warm-up {time.perf_counter()-begin:.1f}s", flush=True)\n    live = LiveCascade(model, encoder, fps)\n    server = make_server(live, port)\n    live.start()\n    print(f"ASCII Flux live: http://127.0.0.1:{port}", flush=True)\n    if on_ready:\n        on_ready(f"http://127.0.0.1:{port}")\n    try:\n        server.serve_forever()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        live.stopped.set()\n        server.server_close()\n', 'ascii_flux/lvvis.py': '"""Official LV-VIS masked-video acquisition. Offline data source only."""\nimport copy\nimport json\nfrom pathlib import Path\nfrom .acquisition import download_file, extract_zip\nfrom .factory import file_sha256\nfrom .storage import write_json\n\n\ndef prepare_lvvis(data_root, profile):\n    source = copy.deepcopy(profile["source"])\n    if source["annotations_license"] != "CC-BY-NC-SA-4.0" or not source.get("terms_checked_on"):\n        raise ValueError("Verify and record the original LV-VIS conditions before download")\n    root = Path(data_root)/"lvvis"\n    annotations = root/"downloads/train_instances.json"\n    media = root/"downloads/train.zip"\n    a = download_file(profile["annotations_url"], annotations, "annotations", expected_sha256=profile["annotations_sha256"])\n    f = download_file(profile["frames_url"], media, "frames", expected_sha256=profile["frames_sha256"])\n    frames = extract_zip(media, root/"frames", durable_files=False)\n    normalized = root/"annotations.train.normalized.json"\n    if not normalized.exists():\n        data = json.loads(annotations.read_text(encoding="utf-8"))\n        for video in data["videos"]:\n            video["original_sequence_id"] = video["id"]\n            video["id"] = "train-"+str(video["id"])\n        for ann in data["annotations"]:\n            ann["source_object_id"] = str(ann["id"])\n            ann["id"] = "train-"+str(ann["id"])\n            ann["video_id"] = "train-"+str(ann["video_id"])\n        write_json(normalized, data)\n    source["local_media_provenance"] = {"annotations": a, "frames": f}\n    result = {"annotations": str(normalized.resolve()), "frames_root": str((frames/"train/JPEGImages").resolve()),\n              "source_fps": profile["source_fps"], "source": source,\n              "normalized_annotation_sha256": file_sha256(normalized)}\n    write_json(root/"prepared.json", result)\n    return result\n', 'ascii_flux/masked.py': '"""Experimental denoising directly on glyph/color tokens, with no image codec.\n\nMasked-token learning and iterative confidence sampling are inspired by MaskGIT\n(Chang et al., CVPR 2022). This convolutional network, temporal context and native\nglyph vocabulary are our own baseline, not the published MaskGIT architecture.\n"""\nfrom dataclasses import asdict, dataclass\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass MaskedConfig:\n    width: int = 64\n    text_dim: int = 768\n    condition_dim: int = 256\n    text_dropout: float = .15\n    emergence_steps: int = 8\n    motion_steps: int = 4\n    guidance: float = 2.\n    temperature: float = .7\n\n    def __post_init__(self):\n        if self.width % 8 or min(self.width, self.text_dim, self.condition_dim, self.emergence_steps, self.motion_steps) < 1:\n            raise ValueError("Invalid masked model dimensions")\n        if not 0 <= self.text_dropout < 1 or self.temperature <= 0 or self.guidance < 0:\n            raise ValueError("Invalid masked model sampling settings")\n\n\nclass ConditionedBlock(nn.Module):\n    def __init__(self, channels, condition):\n        super().__init__()\n        self.norm1 = nn.GroupNorm(8, channels)\n        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)\n        self.norm2 = nn.GroupNorm(8, channels)\n        self.film = nn.Linear(condition, channels*2)\n        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)\n\n    def forward(self, value, condition):\n        hidden = self.conv1(F.silu(self.norm1(value)))\n        scale, shift = self.film(condition).chunk(2, -1)\n        hidden = self.norm2(hidden)*(1+scale[:, :, None, None]) + shift[:, :, None, None]\n        return (value+self.conv2(F.silu(hidden)))/math.sqrt(2)\n\n\nclass MaskedGlyphFlow(nn.Module):\n    """State is bounded native tokens, masks, previous tokens and sampling phase.\n\n    Channels: current glyph/color, unresolved mask, previous glyph/color,\n    phase, mode. Internal mask IDs never appear in the rendered output.\n    """\n    def __init__(self, contract, config=None):\n        super().__init__()\n        self.contract, self.config = contract, config or MaskedConfig()\n        w, d = self.config.width, self.config.condition_dim\n        self.mask_glyph, self.mask_color = len(contract.glyphs), len(contract.palette)+1\n        self.glyph_embedding = nn.Embedding(self.mask_glyph+1, 24)\n        self.color_embedding = nn.Embedding(self.mask_color+1, 24)\n        self.text_condition = nn.Sequential(nn.Linear(self.config.text_dim, d), nn.SiLU(), nn.Linear(d, d))\n        self.ratio_condition = nn.Sequential(nn.Linear(1, d), nn.SiLU(), nn.Linear(d, d))\n        self.mode_embedding = nn.Embedding(3, d)\n        self.input = nn.Conv2d(99, w, 3, padding=1)\n        self.block1 = ConditionedBlock(w, d)\n        self.down1 = nn.Conv2d(w, w*2, 3, stride=2, padding=1)\n        self.block2 = ConditionedBlock(w*2, d)\n        self.down2 = nn.Conv2d(w*2, w*3, 3, stride=2, padding=1)\n        self.block3 = ConditionedBlock(w*3, d)\n        self.global_norm = nn.LayerNorm(w*3)\n        self.global_attention = nn.MultiheadAttention(w*3, 4, batch_first=True)\n        self.block4 = ConditionedBlock(w*3, d)\n        self.up1 = nn.Conv2d(w*5, w*2, 3, padding=1)\n        self.block5 = ConditionedBlock(w*2, d)\n        self.up2 = nn.Conv2d(w*3, w, 3, padding=1)\n        self.block6 = ConditionedBlock(w, d)\n        self.glyph_head = nn.Conv2d(w, len(contract.glyphs), 1)\n        self.color_head = nn.Conv2d(w, len(contract.palette), 1)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, contract.rows), torch.linspace(-1, 1, contract.cols), indexing="ij")\n        self.register_buffer("coordinates", torch.stack((xx, yy))[None])\n\n    def _color_ids(self, glyphs, colors):\n        return torch.where(glyphs == self.mask_glyph, self.mask_color,\n                           torch.where(glyphs == 0, len(self.contract.palette), colors)).long()\n\n    def forward(self, glyphs, colors, previous_glyphs, previous_colors, text, ratio, mode):\n        glyphs, previous_glyphs = glyphs.long(), previous_glyphs.long()\n        colors = self._color_ids(glyphs, colors)\n        previous_colors = self._color_ids(previous_glyphs, previous_colors)\n        embed = torch.cat([self.glyph_embedding(glyphs), self.color_embedding(colors),\n                           self.glyph_embedding(previous_glyphs), self.color_embedding(previous_colors)], -1).permute(0, 3, 1, 2)\n        known = (glyphs != self.mask_glyph).float()[:, None]\n        value = self.input(torch.cat([embed, known, self.coordinates.expand(glyphs.shape[0], -1, -1, -1)], 1))\n        condition = self.text_condition(text*math.sqrt(self.config.text_dim)) + self.ratio_condition(ratio[:, None]) + self.mode_embedding(mode.long())\n        a = self.block1(value, condition)\n        b = self.block2(self.down1(a), condition)\n        c = self.block3(self.down2(b), condition)\n        tokens = c.flatten(2).transpose(1, 2)\n        normalized = self.global_norm(tokens)\n        attended, _ = self.global_attention(normalized, normalized, normalized, need_weights=False)\n        c = self.block4((tokens+attended).transpose(1, 2).reshape_as(c), condition)\n        b = self.block5(self.up1(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1)), condition)\n        a = self.block6(self.up2(torch.cat([F.interpolate(b, size=a.shape[-2:], mode="nearest"), a], 1)), condition)\n        return self.glyph_head(a), self.color_head(a)\n\n    def initial_state(self, batch=1, noisy=False):\n        state = self.coordinates.new_zeros((batch, 7, self.contract.rows, self.contract.cols))\n        state[:, 1] = self.contract.empty_color\n        state[:, 2] = 1\n        state[:, 3] = self.mask_glyph\n        state[:, 4] = self.mask_color\n        return state\n\n    def recondition(self, state):\n        # Preserve the displayed grid; only restart the refinement schedule.\n        result = state.clone()\n        result[:, 3:5] = state[:, :2]\n        result[:, 2] = (torch.rand_like(state[:, 2]) < .85).float()\n        result[:, 5] = 0\n        result[:, 6] = 2\n        return result\n\n    def tokens(self, state):\n        glyphs = state[:, 0].long()\n        colors = state[:, 1].long()\n        return glyphs, torch.where(glyphs != 0, colors, self.contract.empty_color)\n\n    def step(self, state, text):\n        state = state.clone()\n        phase = int(state[0, 5, 0, 0].item())\n        mode = int(state[0, 6, 0, 0].item())\n        total = self.config.motion_steps if mode == 1 else self.config.emergence_steps\n        if phase >= total:\n            state[:, 3:5] = state[:, :2]\n            state[:, 2] = (torch.rand_like(state[:, 2]) < .65).float()\n            state[:, 5] = 0\n            state[:, 6] = 1\n            phase, mode, total = 0, 1, self.config.motion_steps\n        masked = state[:, 2].bool()\n        g = state[:, 0].long().masked_fill(masked, self.mask_glyph)\n        c = state[:, 1].long().masked_fill(masked, self.mask_color)\n        pg, pc = state[:, 3].long(), state[:, 4].long()\n        ratio = masked.float().mean((1, 2))\n        modes = torch.full((len(state),), mode, device=state.device, dtype=torch.long)\n        if self.config.guidance != 1:\n            gl, cl = self(torch.cat([g,g]), torch.cat([c,c]), torch.cat([pg,pg]), torch.cat([pc,pc]),\n                          torch.cat([text,torch.zeros_like(text)]), torch.cat([ratio,ratio]), torch.cat([modes,modes]))\n            gl, ungl = gl.chunk(2); cl, uncl = cl.chunk(2)\n            gl = ungl+self.config.guidance*(gl-ungl)\n            cl = uncl+self.config.guidance*(cl-uncl)\n        else:\n            gl, cl = self(g,c,pg,pc,text,ratio,modes)\n        temperature = self.config.temperature*(.25 if mode == 1 else max(.25, 1-phase/total))\n        gp = F.softmax(gl.float(), 1)\n        # Gumbel-max samples native tokens, rather than independently taking the\n        # marginal argmax of every cell for every pose of a concept.\n        randg = -torch.log(-torch.log(torch.rand_like(gl).clamp_(1e-6, 1-1e-6)))\n        randc = -torch.log(-torch.log(torch.rand_like(cl).clamp_(1e-6, 1-1e-6)))\n        sampled_g = (gl.float()+temperature*randg).argmax(1)\n        sampled_c = (cl.float()+temperature*randc).argmax(1)\n        sampled_c = torch.where(sampled_g != 0, sampled_c, self.contract.empty_color)\n        state[:, 0] = torch.where(masked, sampled_g, state[:, 0].long())\n        state[:, 1] = torch.where(masked, sampled_c, state[:, 1].long())\n        confidence = gp.gather(1, sampled_g[:, None]).squeeze(1)\n        next_ratio = math.cos((phase+1)/total*math.pi/2)\n        # Keep positive anchors as well as background anchors: a vocabulary\n        # dominated by spaces otherwise accepts only background in early steps.\n        remaining = torch.zeros_like(masked)\n        for i in range(len(state)):\n            for foreground in (False, True):\n                group = masked[i] & ((sampled_g[i] != 0) == foreground)\n                indices = group.flatten().nonzero().flatten()\n                count = min(len(indices), round(len(indices)*next_ratio)) if phase+1 < total else 0\n                if count:\n                    scores = confidence[i].flatten()[indices]\n                    keep_masked = indices[scores.topk(count, largest=False).indices]\n                    remaining[i].view(-1)[keep_masked] = True\n        state[:, 2] = remaining.float()\n        state[:, 5] = phase+1\n        return state, gl, cl\n\n    def config_dict(self):\n        return asdict(self.config)\n\n\ndef masked_loss(gl, cl, glyphs, colors, mask):\n    glyph = F.cross_entropy(gl.float(), glyphs.long(), reduction="none")\n    glyph_loss = glyph[mask].mean()\n    active = mask & (glyphs != 0)\n    color_loss = F.cross_entropy(cl.permute(0,2,3,1)[active].float(), colors[active].long()) if active.any() else cl.sum()*0\n    return glyph_loss+color_loss\n', 'ascii_flux/masked_training.py': '"""Controlled masked-token experiment on the existing real ASCII corpus."""\nfrom collections import defaultdict\nfrom dataclasses import asdict, dataclass\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport time\nimport numpy as np\nimport torch\nfrom .contract import TEXT_SPEC\nfrom .masked import MaskedConfig, MaskedGlyphFlow, masked_loss\nfrom .storage import atomic_file, write_json\nfrom .training import Corpus, restore_rng, rng_state\n\n\n@dataclass(frozen=True)\nclass MaskedTrainConfig:\n    steps: int = 3000\n    batch_size: int = 12\n    lr: float = .0003\n    subjects: tuple = ("cat", "dog", "horse", "bird", "fish", "person")\n    validate_every: int = 250\n    val_batches: int = 8\n    seed: int = 20261031\n    memorize_one_frame: bool = False\n    # Exponent of the label-frequency sampling: 1 follows the corpus, 0 is\n    # uniform over labels. Uniform over-samples labels with a single clip.\n    label_balance: float = .5\n\n    def __post_init__(self):\n        object.__setattr__(self,"subjects",tuple(self.subjects))\n        if len(set(self.subjects)) != len(self.subjects) or len(self.subjects) == 1:\n            raise ValueError("Choose at least two distinct real-data subjects, or none for every training label")\n        if not 0 <= self.label_balance <= 1:\n            raise ValueError("label_balance must lie in [0, 1]")\n\n    @property\n    def open_vocabulary(self):\n        return not self.subjects\n\n\ndef _by_label(records):\n    labels = defaultdict(lambda: defaultdict(list))\n    for record in records:\n        labels[record["normalized_label"]][(record["source"], record["video_id"])].append(record)\n    return {label: dict(videos) for label,videos in labels.items()}\n\n\ndef subject_groups(corpus, subjects):\n    """Train/val groups; with no subjects, every label plus never-trained labels.\n\n    ``unseen_val`` holds validation videos whose label never occurs in training:\n    the only direct measure of drawing a word the model was not taught.\n    ``unseen_test`` comes from the reserved test partition and is reported\n    once at the end, never used to choose a checkpoint.\n    """\n    if not subjects:\n        result = {split: _by_label(corpus.groups[split]) for split in ("train", "val")}\n        trained = set(result["train"])\n        result["seen_val"] = {l: v for l,v in result["val"].items() if l in trained}\n        result["unseen_val"] = {l: v for l,v in result["val"].items() if l not in trained}\n        result["unseen_test"] = {l: v for l,v in _by_label(corpus.groups["test_unseen"]).items() if l not in trained}\n        if not result["seen_val"]:\n            raise ValueError("No validation video shares a label with training")\n        return result\n    result = {}\n    for split in ("train", "val"):\n        labels = _by_label(r for r in corpus.groups[split] if r["normalized_label"] in subjects)\n        if set(subjects)-set(labels):\n            raise ValueError(f"Missing {split} subjects: {sorted(set(subjects)-set(labels))}")\n        result[split] = labels\n    result["seen_val"] = result["val"]\n    return result\n\n\ndef label_sampler(groups, config):\n    """Per-group label lists and weights; frequency is counted in source videos."""\n    samplers = {}\n    for name, labels in groups.items():\n        names = sorted(labels)\n        if name == "train":\n            weights = [len(labels[l])**config.label_balance for l in names]\n        else:\n            weights = [1.]*len(names)\n        samplers[name] = (names, weights)\n    return samplers\n\n\ndef token_batch(corpus, groups, config, split, device, rng, samplers=None, phrases=None):\n    if samplers and split in samplers:\n        names, weights = samplers[split]\n        labels = rng.choices(names, weights, k=config.batch_size)\n    else:\n        labels = [rng.choice(config.subjects) for _ in range(config.batch_size)]\n    targets_g, targets_c, previous_g, previous_c, vectors = [], [], [], [], []\n    for label in labels:\n        videos = groups[split][label]\n        if config.memorize_one_frame and split == "train":\n            candidates = [r for rs in videos.values() for r in rs if not r.get("augmentation")]\n            record = min(candidates, key=lambda r: abs(r["quality"]["occupancy"]-.2))\n            t, delta = 12, 1\n        else:\n            record = rng.choice(videos[rng.choice(list(videos))])\n            t, delta = rng.randrange(3, corpus.contract.frames), rng.choice((1,2,3))\n        g,c,_,_ = corpus.read(record)\n        targets_g.append(g[t]);targets_c.append(c[t])\n        previous_g.append(g[t-delta]);previous_c.append(c[t-delta])\n        if phrases is not None:\n            # Training sees every phrasing; evaluation uses the canonical one,\n            # the closest to what a person types.\n            options = phrases[label]\n            vectors.append(options[rng.randrange(len(options))] if split == "train" else options[0])\n        else:\n            vectors.append(corpus.embeddings[label])\n    return (torch.tensor(np.stack(targets_g), dtype=torch.long, device=device),\n            torch.tensor(np.stack(targets_c), dtype=torch.long, device=device),\n            torch.tensor(np.stack(previous_g), dtype=torch.long, device=device),\n            torch.tensor(np.stack(previous_c), dtype=torch.long, device=device),\n            torch.tensor(np.stack(vectors), device=device))\n\n\ndef corrupt(model, batch, validation=False):\n    g,c,pg,pc,text = batch\n    count = len(g)\n    mode = torch.multinomial(torch.tensor([.65,.25,.10], device=g.device), count, replacement=True)\n    # Broad mask ratios include the entirely unknown initial state.\n    ratio = torch.rand(count, device=g.device)*.9+.05\n    ratio = torch.where(torch.rand(count, device=g.device) < .20, 1., ratio)\n    ratio = torch.where(mode != 0, torch.maximum(ratio, ratio.new_tensor(.7)), ratio)\n    mask = torch.rand_like(g.float()) < ratio[:,None,None]\n    xg = g.masked_fill(mask, model.mask_glyph)\n    xc = c.masked_fill(mask, model.mask_color)\n    pg,pc = pg.clone(),pc.clone()\n    pg[mode == 0], pc[mode == 0] = model.mask_glyph, model.mask_color\n    # A different real subject supplies transition context; no generated media.\n    pg[mode == 2],pc[mode == 2] = g.roll(1,0)[mode == 2],c.roll(1,0)[mode == 2]\n    # Slight context corruption exposes the temporal model to imperfect tokens.\n    context_mask = (torch.rand_like(g.float()) < .04) & (mode[:,None,None] == 1)\n    pg[context_mask],pc[context_mask] = model.mask_glyph,model.mask_color\n    if not validation:\n        text = text * (torch.rand(count,1,device=g.device) >= model.config.text_dropout)\n    return (xg,xc,pg,pc,text,mask.float().mean((1,2)),mode),mask\n\n\n@torch.inference_mode()\ndef validate(model, corpus, groups, config, device, split="seen_val", samplers=None, phrases=None, with_gain=False):\n    """Masked CE on fixed draws. With ``with_gain``, also the CE without text.\n\n    The text gain (CE without text minus CE with text) is positive only if the\n    prompt actually informs the grid. On labels never trained, it measures\n    transfer through the encoder; it is an automatic proxy, not recognition.\n    """\n    saved = rng_state()\n    model.eval()\n    try:\n        rng = random.Random(config.seed+1)\n        torch.manual_seed(config.seed+1)\n        values, unconditioned = [], []\n        for _ in range(config.val_batches):\n            batch = token_batch(corpus, groups, config, split, device, rng, samplers, phrases)\n            inputs, mask = corrupt(model, batch, validation=True)\n            with torch.autocast("cuda",dtype=torch.float16,enabled=str(device).startswith("cuda")):\n                gl,cl = model(*inputs)\n                values.append(float(masked_loss(gl,cl,batch[0],batch[1],mask)))\n                if with_gain:\n                    blind = inputs[:4]+(torch.zeros_like(inputs[4]),)+inputs[5:]\n                    gl,cl = model(*blind)\n                    unconditioned.append(float(masked_loss(gl,cl,batch[0],batch[1],mask)))\n        if not with_gain:\n            return float(np.mean(values))\n        return {"ce": float(np.mean(values)), "ce_without_text": float(np.mean(unconditioned)),\n                "text_gain": float(np.mean(unconditioned)-np.mean(values)), "labels": len(groups[split])}\n    finally:\n        restore_rng(saved)\n        model.train()\n\n\ndef train_masked(manifest_path, embeddings_path, reviews_path, output, config=None, model_config=None, device="cuda", resume=None,\n                 phrases_path=None):\n    config, model_config = config or MaskedTrainConfig(), model_config or MaskedConfig()\n    if min(config.steps,config.batch_size,config.validate_every,config.val_batches) <= 0 or config.lr <= 0:\n        raise ValueError("Positive training configuration required")\n    corpus = Corpus(manifest_path, embeddings_path, reviews_path)\n    groups = subject_groups(corpus, config.subjects)\n    samplers = label_sampler(groups, config) if config.open_vocabulary else None\n    phrases, phrase_metadata = (None, None)\n    if phrases_path:\n        from .text import load_phrase_embeddings\n        phrases, phrase_metadata = load_phrase_embeddings(phrases_path, corpus.manifest["fingerprint"])\n        if set(corpus.embeddings)-set(phrases):\n            raise ValueError("Phrase cache misses dataset labels")\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if (output/"latest.pt").exists() and not resume:\n        raise FileExistsError("Existing masked experiment; resume or choose a new run")\n    random.seed(config.seed);np.random.seed(config.seed);torch.manual_seed(config.seed)\n    model = MaskedGlyphFlow(corpus.contract,model_config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(),lr=config.lr,weight_decay=.01)\n    scaler = torch.amp.GradScaler("cuda", enabled=str(device).startswith("cuda"), init_scale=256.)\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    code = hashlib.sha256()\n    for name in ("masked.py","masked_training.py","text.py"):\n        code.update(Path(__file__).with_name(name).read_bytes())\n    provenance = {"dataset_fingerprint":corpus.manifest["fingerprint"],"text_spec":TEXT_SPEC,\n                  "allow_fixture":False,"allow_unreviewed":False,"synthetic_encoder":False,\n                  "recognition_validated":False,"development_only":True,\n                  "scope":("real-data memorization diagnostic" if config.memorize_one_frame else\n                           "open-vocabulary masked glyph experiment over every training label; recognition and motion unvalidated"\n                           if config.open_vocabulary else "six-concept masked glyph experiment; recognition and motion unvalidated"),\n                  "training_concepts":sorted(groups["train"]) if config.open_vocabulary else list(config.subjects),\n                  "never_trained_validation_labels":sorted(groups.get("unseen_val",{})),\n                  "phrases":phrase_metadata,"source_conditions":corpus.manifest["source"],\n                  "review_scope":corpus.review_scope,"training_code_sha256":code.hexdigest(),\n                  "torch_version":str(torch.__version__)}\n    best,best_model,best_step,start,history = math.inf,None,0,0,[]\n    if resume:\n        checkpoint = torch.load(resume,map_location="cpu",weights_only=False)\n        corpus.contract.require(checkpoint["contract"])\n        old,new = dict(checkpoint["train_config"]),asdict(config)\n        old.pop("steps");new.pop("steps")\n        if checkpoint["provenance"] != provenance or old != new or checkpoint["model_config"] != asdict(model_config):\n            raise ValueError("Masked resume provenance/config mismatch")\n        model.load_state_dict(checkpoint["model"]);ema.load_state_dict(checkpoint["ema"])\n        optimizer.load_state_dict(checkpoint["optimizer"]);scaler.load_state_dict(checkpoint["scaler"])\n        start,best,best_model,history = checkpoint["step"],checkpoint["best_val"],checkpoint["best_model"],checkpoint["history"]\n        best_step = checkpoint["best_step"]\n        restore_rng(checkpoint["rng"])\n    started = time.monotonic()\n    print(json.dumps({"parameters":sum(p.numel() for p in model.parameters()),"training_labels":len(groups["train"]),\n                      "never_trained_val_labels":len(groups.get("unseen_val",{})),"scope":provenance["scope"]}),flush=True)\n    for step in range(start,config.steps):\n        batch = token_batch(corpus,groups,config,"train",device,random,samplers,phrases)\n        inputs,mask = corrupt(model,batch)\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast("cuda",dtype=torch.float16,enabled=str(device).startswith("cuda")):\n            gl,cl = model(*inputs)\n            loss = masked_loss(gl,cl,batch[0],batch[1],mask)\n        if not torch.isfinite(loss):\n            raise FloatingPointError("Non-finite masked-token loss")\n        scaler.scale(loss).backward();scaler.unscale_(optimizer)\n        norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=not scaler.is_enabled())\n        scaler.step(optimizer);scaler.update()\n        if math.isfinite(float(norm)):\n            decay = min(.999,(1+step)/(10+step))\n            with torch.no_grad():\n                for a,b in zip(ema.parameters(),model.parameters()):\n                    a.lerp_(b,1-decay)\n        if (step+1) % 25 == 0:\n            write_json(output/"progress.json",{"step":step+1,"total_steps":config.steps,"loss":float(loss.detach()),\n                         "elapsed_seconds":time.monotonic()-started,"recognition_validated":False})\n        if (step+1) % config.validate_every == 0 or step+1 == config.steps:\n            if config.open_vocabulary:\n                seen = validate(ema,corpus,groups,config,device,"seen_val",samplers,phrases,with_gain=True)\n                value = seen["ce"]\n            else:\n                value = validate(ema,corpus,groups,config,device)\n            if not math.isfinite(value):\n                raise FloatingPointError("Non-finite masked validation")\n            # The checkpoint is chosen on labels seen in training only.\n            if value < best:\n                best,best_model = value,copy.deepcopy({k:v.detach().cpu() for k,v in ema.state_dict().items()})\n                best_step = step+1\n            row = {"step":step+1,"masked_train_ce":float(loss.detach()),"masked_val_ce":value,"best_val_ce":best}\n            if config.open_vocabulary:\n                row["seen_val"] = seen\n                if groups["unseen_val"]:\n                    row["unseen_val"] = validate(ema,corpus,groups,config,device,"unseen_val",samplers,phrases,with_gain=True)\n                if step+1 == config.steps and groups["unseen_test"]:\n                    row["unseen_test_final_report_only"] = validate(ema,corpus,groups,config,device,"unseen_test",samplers,phrases,with_gain=True)\n            history.append(row)\n            checkpoint = {"format":"ascii-flux-v5-masked-1","contract":corpus.contract.to_dict(),"model_config":asdict(model_config),\n                          "train_config":asdict(config),"provenance":provenance,"model":model.state_dict(),"ema":ema.state_dict(),\n                          "optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),"step":step+1,"best_val":best,"best_step":best_step,\n                          "best_model":best_model,"rng":rng_state(),"history":history}\n            with atomic_file(output/"latest.pt") as stream:torch.save(checkpoint,stream)\n            with atomic_file(output/"best.pt") as stream:\n                torch.save({k:checkpoint[k] for k in ("format","contract","model_config","provenance")} | {"model":best_model,"step":best_step},stream)\n            write_json(output/"history.json",history)\n            print(json.dumps(row),flush=True)\n    return model\n', 'ascii_flux/model.py': '"""Glyph-space NCA baseline. No image representation exists in this module."""\nfrom dataclasses import asdict, dataclass\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass ModelConfig:\n    state_channels: int = 48\n    width: int = 256\n    text_dim: int = 768\n    update_probability: float = 0.5\n    global_stride: int = 4\n    text_dropout: float = 0.1\n\n    def __post_init__(self):\n        if min(self.state_channels, self.width, self.text_dim, self.global_stride) <= 0:\n            raise ValueError("Invalid model dimensions")\n        if not 0 < self.update_probability <= 1 or not 0 <= self.text_dropout < 1:\n            raise ValueError("Invalid stochastic probabilities")\n\n\nclass GlyphNCA(nn.Module):\n    def __init__(self, contract, config=None):\n        super().__init__()\n        self.contract = contract\n        self.config = config or ModelConfig()\n        c, w = self.config.state_channels, self.config.width\n        self.glyph_embedding = nn.Embedding(len(contract.glyphs), c)\n        self.color_embedding = nn.Embedding(len(contract.palette)+1, c, padding_idx=len(contract.palette))\n        self.local = nn.Conv2d(c, c*2, 3, padding=1)\n        self.global_context = nn.Sequential(nn.Conv2d(c, c*2, 3, padding=1), nn.SiLU(), nn.Conv2d(c*2, c*2, 3, padding=1))\n        self.project = nn.Conv2d(c*4+3, w, 1)\n        self.film = nn.Sequential(nn.Linear(self.config.text_dim, 768), nn.SiLU(), nn.Linear(768, w*2))\n        self.update = nn.Sequential(nn.SiLU(), nn.Conv2d(w, w, 1), nn.SiLU(), nn.Conv2d(w, c, 1))\n        nn.init.zeros_(self.update[-1].weight)\n        nn.init.zeros_(self.update[-1].bias)\n        self.glyph_head = nn.Conv2d(c, len(contract.glyphs), 1)\n        self.color_head = nn.Conv2d(c, len(contract.palette), 1)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, contract.rows), torch.linspace(-1, 1, contract.cols), indexing="ij")\n        self.register_buffer("coordinates", torch.stack((xx, yy))[None])\n\n    def initial_state(self, batch=1, noisy=False):\n        state = self.coordinates.new_zeros((batch, self.config.state_channels, self.contract.rows, self.contract.cols))\n        return state + torch.randn_like(state)*.01 if noisy else state\n\n    def lift(self, glyphs, colors):\n        # Teacher forcing only. The deployed recurrent step does not read targets.\n        color = torch.where(glyphs != 0, colors, len(self.contract.palette))\n        return (self.glyph_embedding(glyphs) + self.color_embedding(color)).permute(0, 3, 1, 2).contiguous()\n\n    def forward(self, state, text, noise, update_mask):\n        """Explicit noise/mask inputs permit deterministic ONNX parity and RNG control."""\n        cfg = self.config\n        local = self.local(state)\n        pooled = F.avg_pool2d(state, cfg.global_stride)\n        context = F.interpolate(self.global_context(pooled), size=(self.contract.rows, self.contract.cols), mode="nearest")\n        coordinates = self.coordinates.expand(state.shape[0], -1, -1, -1)\n        features = self.project(torch.cat((local, context, coordinates, noise), dim=1))\n        scale, bias = self.film(text).chunk(2, dim=1)\n        features = features * (1 + scale[:, :, None, None]) + bias[:, :, None, None]\n        next_state = state + .1*torch.tanh(self.update(features))*update_mask\n        return next_state, self.glyph_head(next_state), self.color_head(next_state)\n\n    def step(self, state, text):\n        shape = (state.shape[0], 1, self.contract.rows, self.contract.cols)\n        noise = torch.randn(shape, device=state.device, dtype=state.dtype)\n        mask = (torch.rand(shape, device=state.device) < self.config.update_probability).to(state.dtype)\n        return self(state, text, noise, mask)\n\n    def tokens(self, state):\n        glyphs = self.glyph_head(state).argmax(1)\n        colors = self.color_head(state).argmax(1)\n        return glyphs, torch.where(glyphs != 0, colors, self.contract.empty_color)\n\n    def config_dict(self):\n        return asdict(self.config)\n\n\ndef token_loss(glyph_logits, color_logits, glyphs, colors, foreground_weight=1.):\n    # Color on empty cells has neither a target nor a gradient. Separate heads.\n    if foreground_weight == 1.:\n        glyph_loss = F.cross_entropy(glyph_logits, glyphs)\n    else:\n        weights = torch.where(glyphs != 0, foreground_weight, 1.)\n        glyph_loss = (F.cross_entropy(glyph_logits, glyphs, reduction="none")*weights).sum()/weights.sum()\n    valid = glyphs != 0\n    if valid.any():\n        color_loss = F.cross_entropy(color_logits.permute(0, 2, 3, 1)[valid], colors[valid])\n    else:\n        color_loss = color_logits.sum()*0\n    return glyph_loss + color_loss\n\n', 'ascii_flux/offline.py': '"""Offline video/mask -> glyphs. Never used by the inference runtime.\n\nTone, coverage, density gamma, hysteresis and HSV saturation are adapted from\nthe frozen v4.2 notebook. No procedural halo or synthetic volume is baked in.\n"""\nfrom dataclasses import asdict, dataclass\nimport hashlib\nimport platform\nfrom importlib.metadata import version\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nfrom .contract import digest\n\n\n@dataclass(frozen=True)\nclass ConverterConfig:\n    margin: float = 0.15\n    smoothing: float = 0.25\n    gamma: float = 0.8\n    hysteresis: float = 0.7\n    saturation: float = 1.15\n    min_coverage: float = 0.12\n\n    def __post_init__(self):\n        if self.margin < 0 or not 0 < self.smoothing <= 1 or self.gamma <= 0 or not 0 <= self.min_coverage < 1:\n            raise ValueError("Invalid converter configuration")\n        if self.hysteresis < 0 or self.saturation < 0:\n            raise ValueError("Invalid tone settings")\n\n\ndef converter_identity(config):\n    # Includes adapter, schema and metrics: implementation changes invalidate provenance.\n    h = hashlib.sha256()\n    for name in ("offline.py", "factory.py", "storage.py", "contract.py"):\n        h.update(name.encode())\n        h.update(Path(__file__).with_name(name).read_bytes())\n    return {"version": "offline-v5-1", "code_sha256": h.hexdigest(),\n            "config": asdict(config), "config_sha256": digest(asdict(config)),\n            "environment": {"python": platform.python_version(), "numpy": np.__version__,\n                            "opencv": cv2.__version__, "pillow": version("Pillow")}}\n\n\ndef tracked_boxes(masks, contract, config):\n    measurements = []\n    for mask in masks:\n        y, x = np.nonzero(mask)\n        measurements.append(None if len(x) == 0 else\n                            np.array([x.min(), y.min(), x.max() + 1, y.max() + 1], np.float32))\n    valid = [i for i, b in enumerate(measurements) if b is not None]\n    if not valid:\n        raise ValueError("no_subject_mask")\n    measurements = np.stack([measurements[min(valid, key=lambda j: abs(j-i))] for i in range(len(masks))])\n    centers = (measurements[:, :2] + measurements[:, 2:]) / 2\n    # One clip-wide scale avoids per-frame crop breathing. Padding may lie outside\n    # the image and is rendered black, preserving the actual grid aspect ratio.\n    aspect = contract.cols / (contract.rows * contract.cell_height_width)\n    extents = (measurements[:, 2:] - measurements[:, :2]).max(0) * (1 + 2 * config.margin)\n    width = max(float(extents[0]), float(extents[1]) * aspect, 1)\n    height = width / aspect\n    for i in range(1, len(centers)):\n        centers[i] = config.smoothing * centers[i] + (1-config.smoothing) * centers[i-1]\n    # Include the subject even when smoothing lags a fast translation.\n    centers = np.maximum(centers, measurements[:, 2:] - [width/2, height/2])\n    centers = np.minimum(centers, measurements[:, :2] + [width/2, height/2])\n    return np.concatenate([centers - [width/2, height/2], centers + [width/2, height/2]], 1).astype(np.float32)\n\n\ndef _crop(array, box, width, height, nearest=False):\n    xs = box[0] + (np.arange(width, dtype=np.float32) + .5) * (box[2]-box[0])/width - .5\n    ys = box[1] + (np.arange(height, dtype=np.float32) + .5) * (box[3]-box[1])/height - .5\n    mx, my = np.meshgrid(xs, ys)\n    return cv2.remap(array, mx, my, cv2.INTER_NEAREST if nearest else cv2.INTER_LINEAR,\n                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n\n\ndef _luminance(rgb):\n    rgb = rgb.astype(np.float32) / 255\n    return .55 * (rgb @ np.array([.2126, .7152, .0722], np.float32)) + .45 * rgb.max(-1)\n\n\ndef _change(current, previous, valid=None):\n    active = (current != 0) | (previous != 0)\n    if valid is not None:\n        active &= valid\n    return float(((current != previous) & active).sum() / max(1, active.sum()))\n\n\ndef convert_clip(frames, masks, contract, config=None):\n    config = config or ConverterConfig()\n    if len(frames) != contract.frames or len(masks) != contract.frames:\n        raise ValueError("wrong_clip_length")\n    if any(f.shape[:2] != m.shape or f.shape[-1] != 3 for f, m in zip(frames, masks)):\n        raise ValueError("frame_mask_shape_mismatch")\n    boxes = tracked_boxes(masks, contract, config)\n    width, height = contract.cols * 4, contract.rows * 4\n    rgbs, alphas, grays = [], [], []\n    for frame, mask, box in zip(frames, masks, boxes):\n        alpha = _crop(mask.astype(np.float32), box, width, height)\n        premultiplied = _crop(frame.astype(np.float32) * mask[..., None], box, width, height)\n        rgb = premultiplied / np.maximum(alpha[..., None], 1e-6)\n        rgbs.append(rgb)\n        alphas.append(alpha)\n        grays.append(cv2.cvtColor(np.uint8(np.clip(premultiplied, 0, 255)), cv2.COLOR_RGB2GRAY))\n    # Fixed clip-wide percentiles and CDF avoid pumping the contrast every frame.\n    luminances = [_luminance(rgb) for rgb in rgbs]\n    samples = np.concatenate([l[a > .5] for l, a in zip(luminances, alphas)])\n    if len(samples) == 0:\n        raise ValueError("subject_lost_in_crop")\n    lo, hi = np.percentile(samples, [2, 98])\n    reference = np.sort(np.clip((samples-lo)/max(hi-lo, .001), 0, 1))\n    top = len(contract.glyphs)-1\n    glyphs, colors, raw, compensated, camera, subject_sizes = [], [], [], [], [], []\n    previous_level = None\n    grid_y, grid_x = np.mgrid[:contract.rows, :contract.cols].astype(np.float32)\n    palette = np.array(contract.palette, dtype=np.float32)\n    for i, (rgb, alpha, lum) in enumerate(zip(rgbs, alphas, luminances)):\n        coverage = cv2.resize(alpha, (contract.cols, contract.rows), interpolation=cv2.INTER_AREA)\n        safe = np.maximum(coverage, 1e-6)\n        base = np.clip((lum-lo)/max(hi-lo, .001), 0, 1)\n        tone = .5*base + .5*np.searchsorted(reference, base)/len(reference)\n        tone_cell = cv2.resize((tone*alpha).astype(np.float32), (contract.cols, contract.rows), interpolation=cv2.INTER_AREA)/safe\n        target = np.clip(tone_cell, 0, 1)**config.gamma * np.sqrt(np.minimum(1, coverage/.6)) * top\n        level = np.rint(target).clip(0, top).astype(np.uint8)\n        transported, valid = None, None\n        if previous_level is not None:\n            # Backward optical flow on the already tracked crops, in pixel units.\n            flow = cv2.calcOpticalFlowFarneback(grays[i], grays[i-1], None, .5, 3, 15, 3, 5, 1.2, 0)\n            small = cv2.resize(flow, (contract.cols, contract.rows), interpolation=cv2.INTER_AREA)/4\n            mx, my = grid_x + small[..., 0], grid_y + small[..., 1]\n            valid = (mx >= 0) & (my >= 0) & (mx <= contract.cols-1) & (my <= contract.rows-1)\n            transported = cv2.remap(previous_level, mx, my, cv2.INTER_NEAREST,\n                                    borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n            level = np.where(valid & (np.abs(target-transported) < config.hysteresis), transported, level).astype(np.uint8)\n        level[coverage >= .35] = np.maximum(level[coverage >= .35], 1)\n        level[coverage < config.min_coverage] = 0\n        mean_rgb = cv2.resize((rgb*alpha[..., None]).astype(np.float32), (contract.cols, contract.rows), interpolation=cv2.INTER_AREA)/safe[..., None]\n        hsv = cv2.cvtColor(np.clip(mean_rgb/255, 0, 1), cv2.COLOR_RGB2HSV)\n        hsv[..., 1] = np.clip(hsv[..., 1]*config.saturation, 0, 1)\n        hsv[..., 2] = .42 + .58*level/top\n        shown = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)*255\n        indices = ((shown[..., None, :]-palette)**2).sum(-1).argmin(-1).astype(np.uint8)\n        indices[level == 0] = contract.empty_color\n        if previous_level is not None:\n            raw.append(_change(level, previous_level))\n            compensated.append(_change(level, transported, valid))\n            # Background translation is a camera-motion proxy, not subject motion.\n            prev_gray = cv2.cvtColor(frames[i-1], cv2.COLOR_RGB2GRAY)\n            curr_gray = cv2.cvtColor(frames[i], cv2.COLOR_RGB2GRAY)\n            scale = min(1., 320/max(prev_gray.shape))\n            size = (max(8, round(prev_gray.shape[1]*scale)), max(8, round(prev_gray.shape[0]*scale)))\n            bg = cv2.resize((~(masks[i-1].astype(bool) | masks[i].astype(bool))).astype(np.uint8), size, interpolation=cv2.INTER_NEAREST) > 0\n            flow = cv2.calcOpticalFlowFarneback(cv2.resize(prev_gray, size), cv2.resize(curr_gray, size), None, .5, 3, 15, 3, 5, 1.2, 0)\n            texture = cv2.resize(prev_gray, size)\n            bg &= (np.abs(cv2.Laplacian(texture, cv2.CV_32F)) > 2)\n            if bg.sum() >= 32:\n                translation = np.median(flow[bg], axis=0)/scale\n                crop_translation = (boxes[i, :2]+boxes[i, 2:]-boxes[i-1, :2]-boxes[i-1, 2:])/2\n                camera.append(float(np.linalg.norm(translation-crop_translation)))\n        subject_sizes.append(int((coverage >= config.min_coverage).sum()))\n        glyphs.append(level)\n        colors.append(indices)\n        previous_level = level\n    glyphs, colors = np.stack(glyphs), np.stack(colors)\n    metrics = {\n        "occupancy": float((glyphs != 0).mean()),\n        "subject_cells_mean": float(np.mean(subject_sizes)),\n        "subject_cells_min": min(subject_sizes),\n        "glyph_change_raw_active": float(np.mean(raw)) if raw else 0.,\n        "glyph_change_compensated_active": float(np.mean(compensated)) if compensated else 0.,\n        "glyph_changes_per_second": float(np.count_nonzero(glyphs[1:] != glyphs[:-1])/((len(glyphs)-1)/contract.fps)),\n        "mask_fraction": float(np.mean([np.any(m) for m in masks])),\n        "residual_camera_motion_pixels_proxy": float(np.mean(camera)) if camera else None,\n        "camera_proxy_measured_pairs": len(camera),\n        "unique_grid_count": len({g.tobytes() for g in glyphs}),\n        "distinct_grid_updates_per_second": float(np.any(glyphs[1:] != glyphs[:-1], axis=(1, 2)).sum()/((len(glyphs)-1)/contract.fps)),\n        "seam_change_active": _change(glyphs[0], glyphs[-1]),\n        "seam_scope": "diagnostic only: natural clips are not periodic",\n    }\n    return glyphs, colors, boxes, metrics\n\n\ndef quality_rejections(metrics, thresholds):\n    reasons = []\n    for setting, metric, minimum in (\n        ("min_occupancy", "occupancy", True), ("max_occupancy", "occupancy", False),\n        ("min_subject_cells", "subject_cells_min", True),\n        ("max_compensated_change", "glyph_change_compensated_active", False),\n        ("min_mask_fraction", "mask_fraction", True),\n        ("max_residual_camera_motion", "residual_camera_motion_pixels_proxy", False),\n    ):\n        bound = thresholds.get(setting)\n        if bound is not None:\n            value = metrics[metric]\n            if value is None:\n                reasons.append(setting + ":unmeasured")\n            elif (value < bound if minimum else value > bound):\n                reasons.append(setting)\n    return reasons\n\n', 'ascii_flux/pipeline.py': '"""Local D2 -> frozen corpus -> native NCA training, with resumable phases."""\nfrom dataclasses import asdict\nfrom contextlib import ExitStack\nimport json\nimport os\nfrom pathlib import Path\nimport time\nfrom .catalogue import Catalogue, writer_lock\nfrom .storage import write_json\n\n\ndef run_million(project_root, data_root=None, workers=4):\n    project = Path(project_root).resolve()\n    data = Path(data_root).resolve() if data_root else project/"data"\n    output = data/"v5/million-001"\n    output.mkdir(parents=True, exist_ok=True)\n    status_file = output/"pipeline-status.json"\n    started = time.time()\n    state = {"pid": os.getpid(), "started_unix": started, "target_grid_slots": 1_000_000}\n\n    def status(phase, **extra):\n        state.update(phase=phase, updated_unix=time.time(), **extra)\n        write_json(status_file, state)\n        print(json.dumps({"phase": phase, **extra}), flush=True)\n\n    def read(name):\n        return json.loads((project/name).read_text(encoding="utf-8"))\n\n    # The long-lived pipeline lock is separate from short converter DB locks.\n    with ExitStack() as stack:\n        try:\n            stack.enter_context(writer_lock(output/"pipeline"))\n        except (BlockingIOError, PermissionError):\n            existing = json.loads(status_file.read_text(encoding="utf-8")) if status_file.exists() else {"phase": "starting"}\n            print(json.dumps({"already_running": True, "status": existing, "status_file": str(status_file)}), flush=True)\n            return existing\n        try:\n            import torch\n            if not torch.cuda.is_available():\n                raise RuntimeError("Select the ASCII Flux - Radeon GPU kernel/interpreter; CPU fallback is disabled for this run")\n            sample = data/"v5/d1-public-001/manifest.json"\n            evidence = read("configs/d1_visual_review.json")\n            if not sample.exists() or json.loads(sample.read_text(encoding="utf-8"))["fingerprint"] != evidence["sample_dataset_fingerprint"]:\n                raise ValueError("Use the actual D1 sample reviewed by the user before launching scale conversion")\n            status("acquisition", gpu=torch.cuda.get_device_name(0), torch=str(torch.__version__))\n            from .acquisition import load_prepared\n            from .public_source import prepare_public_ytvis2019\n            from .lvvis import prepare_lvvis\n            ytvis = load_prepared(data, profile="ytvis-2019-public")\n            if ytvis is None:\n                ytvis = prepare_public_ytvis2019(data, read("configs/ytvis_source.json"), read("configs/ytvis2019_public.json"))\n            lvvis = prepare_lvvis(data, read("configs/lvvis_public.json"))\n            config = read("configs/d2.json")\n            from .scale import build_scale\n            for prepared in (ytvis, lvvis):\n                status("conversion", source=prepared["source"]["source"])\n                # A converter started interactively may still be finishing. Its\n                # OS lock is released even after a crash; completed jobs are reused.\n                while True:\n                    try:\n                        build_scale(prepared["annotations"], prepared["frames_root"], prepared["source"], config,\n                                    output, prepared["source_fps"], workers=workers)\n                        break\n                    except (BlockingIOError, PermissionError) as error:\n                        # Only lock acquisition errors have this precise marker.\n                        # A real media/SQL error must be surfaced, not retried forever.\n                        from .catalogue import writer_lock as lock\n                        try:\n                            with lock(output):\n                                pass\n                        except (BlockingIOError, PermissionError):\n                            status("waiting_for_active_converter", source=prepared["source"]["source"])\n                            time.sleep(20)\n                            continue\n                        raise error\n            with Catalogue(output) as catalogue:\n                report = catalogue.report()\n                status("conversion_finished", grid_slots=report["grid_slots"], distinct_token_grids=report["distinct_token_grids"],\n                       source_observations=report["distinct_source_object_observations"])\n                # Never fill a shortfall by looping clips or silently relaxing quality.\n                if report["grid_slots"] < config["target_grids"]:\n                    raise ValueError("Quality-filtered source yield is below one million; add verified videos or explicitly revise the dataset plan")\n                manifest = catalogue.snapshot("manifest-million-001.json")\n            from .quality import bind_sample_review\n            reviews = bind_sample_review(manifest, sample, output/"reviews-million-001.json",\n                                         evidence["reviewer"], evidence["human_statement"])\n            from .text import cache_embeddings\n            embeddings = output/"text-million-001.npz"\n            if not embeddings.exists():\n                status("text_embeddings")\n                previous = sample.with_name("text.real.npz")\n                cache_embeddings(manifest, embeddings, "cuda", previous if previous.exists() else None)\n            from .training import TrainConfig, train\n            training = TrainConfig(**read("configs/train_local_8gb.json"))\n            run = data/"runs/native-million-001"\n            resume = run/"latest.pt"\n            status("training", manifest=str(manifest), run=str(run), configuration=asdict(training))\n            train(manifest, embeddings, run, training, reviews_path=reviews, device="cuda",\n                  resume=resume if resume.exists() else None)\n            status("training_finished", checkpoint=str(run/"best.pt"), recognition_validated=False,\n                   elapsed_seconds=time.time()-started)\n        except BaseException as error:\n            status("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",\n                   error=type(error).__name__+": "+str(error))\n            raise\n', 'ascii_flux/progress.py': '"""Observe D1 output without modifying conversion or its checkpoint identity."""\nfrom contextlib import contextmanager\nimport json\nfrom pathlib import Path\nimport tarfile\nimport threading\nimport time\n\n\ndef dataset_progress_snapshot(output):\n    """Count only atomically published clips; ignore incomplete temporary files."""\n    output = Path(output)\n    manifest = output/"manifest.json"\n    try:\n        if manifest.is_file():\n            value = json.loads(manifest.read_text(encoding="utf-8"))\n            report = value["report"]\n            return {"complete": True, "saved": report.get("machine_accepted", len(value["clips"])),\n                    "attempted": report.get("attempted_full_clips"), "rejections": report.get("rejected_records", 0)}\n        stages = [p for p in output.parent.glob(output.name+".*") if p.is_dir()]\n        if not stages:\n            return {"complete": False, "saved": 0}\n        stage = max(stages, key=lambda p: p.stat().st_mtime)\n        count = 0\n        for shard in stage.glob("shard-*.tar"):\n            with tarfile.open(shard, "r") as archive:\n                count += sum(member.isfile() and member.name.endswith(".npz") for member in archive)\n        return {"complete": False, "saved": count}\n    except (OSError, ValueError, KeyError, tarfile.TarError):\n        # The staging directory may be renamed to the final path while observed.\n        return {"complete": False, "saved": None}\n\n\ndef _emit(message):\n    print(message, flush=True)\n\n\n@contextmanager\ndef dataset_progress(output, max_candidates, interval=2., heartbeat=20., emit=None):\n    """Notebook progress from saved batches, plus a heartbeat during each batch.\n\n    No model, converter function or input data is patched. In particular this\n    observer can be added without invalidating an existing converter\'s identity.\n    """\n    if max_candidates <= 0 or interval <= 0 or heartbeat <= 0:\n        raise ValueError("Invalid progress limits")\n    emit = emit or _emit\n    stopped = threading.Event()\n    started = time.monotonic()\n    previous_count, last_message = -1, started\n\n    def update(force=False):\n        nonlocal previous_count, last_message\n        status = dataset_progress_snapshot(output)\n        now = time.monotonic()\n        if status["saved"] is None:\n            return\n        if force or status["saved"] != previous_count or now-last_message >= heartbeat:\n            seconds = int(now-started)\n            duration = f"{seconds//60} min {seconds%60:02d} s"\n            if status["complete"]:\n                message = f"[D1] Terminé : {status[\'saved\']} clips sauvegardés"\n                if status["attempted"] is not None:\n                    message += f", {status[\'attempted\']} candidats traités"\n                message += f", {status[\'rejections\']} rejets. Durée : {duration}."\n            else:\n                message = (f"[D1] Conversion en cours : {status[\'saved\']} clips sauvegardés "\n                           f"(jusqu’à {max_candidates} candidats). Durée : {duration}.")\n            emit(message)\n            previous_count, last_message = status["saved"], now\n\n    def observe():\n        while not stopped.wait(interval):\n            update()\n\n    emit("[D1] Conversion CPU. Le compteur avance à chaque lot sauvegardé ; un lot peut prendre plusieurs minutes.")\n    update(force=True)\n    worker = threading.Thread(target=observe, name="ascii-flux-D1-progress", daemon=True)\n    worker.start()\n    succeeded = False\n    try:\n        yield\n        succeeded = True\n    finally:\n        stopped.set()\n        worker.join(timeout=2)\n        if succeeded:\n            update(force=True)\n        else:\n            emit("[D1] Conversion interrompue ou en erreur ; aucun résultat incomplet n\'est annoncé comme terminé.")\n', 'ascii_flux/public_source.py': '"""Public mirror profile for YouTube-VIS 2019, with exact source attribution.\n\nVIS labels/RLE masks come from a research repack on Kaggle. The corresponding\noriginal video frames come from a version-pinned YouTube-VOS archive on HF.\nNo new media rights are inferred from a mirror\'s availability or license label.\n"""\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport tarfile\nimport tempfile\nimport zipfile\nfrom .acquisition import download_file, _validate_annotations\nfrom .contract import DEFAULT, digest\nfrom .factory import file_sha256, validate_source\nfrom .storage import atomic_file, write_json\n\n\nTRANSFORM_VERSION = "kaggle-sequences-to-vis-1"\n\n\ndef normalize_repacked_annotations(raw, minimum_frames=24):\n    """Repackage tokens/IDs only; do not rewrite category labels or synthesize masks.\n\n    Videos with dropped/irregular listed frames are explicitly excluded because\n    the initial converter expects a uniform annotated-frame cadence.\n    """\n    if not isinstance(raw, dict) or not isinstance(raw.get("sequences"), list):\n        raise ValueError("Miroir d\'annotations : format meta+sequences attendu")\n    labels = raw.get("meta", {}).get("category_labels", {})\n    if not labels:\n        raise ValueError("Miroir d\'annotations sans labels de catégorie")\n    categories = [{"id": int(k), "name": str(v)} for k, v in labels.items()]\n    videos, annotations, excluded, identities = [], [], [], set()\n    for seq in raw["sequences"]:\n        paths = seq.get("image_paths", [])\n        paths = [PurePosixPath(p) for p in paths]\n        if not paths or any(p.is_absolute() or ".." in p.parts or len(p.parts) != 3 or p.parts[0] != "JPEGImages" or\n                            "\\\\" in str(p) or ":" in str(p) for p in paths):\n            raise ValueError("Chemins d\'images inattendus dans le miroir d\'annotations")\n        video_id = paths[0].parts[1]\n        if video_id in identities or any(p.parts[1] != video_id for p in paths):\n            raise ValueError("Identité de vidéo source incohérente ou dupliquée")\n        if any(c not in "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ-_" for c in video_id):\n            raise ValueError("Identifiant vidéo invalide")\n        identities.add(video_id)\n        try:\n            frame_ids = [int(p.stem) for p in paths]\n        except ValueError as exc:\n            raise ValueError("Noms des images sans indices temporels") from exc\n        reasons = []\n        if len(paths) < minimum_frames:\n            reasons.append("shorter_than_target_clip")\n        if any(b-a != 5 for a, b in zip(frame_ids, frame_ids[1:])):\n            reasons.append("irregular_annotated_frame_cadence")\n        if len(seq.get("segmentations", [])) != len(paths) or seq.get("length") != len(paths):\n            raise ValueError("Nombre d\'images et de masques incohérent dans le miroir")\n        if reasons:\n            excluded.append({"video_id": video_id, "original_sequence_id": seq.get("id"), "reasons": reasons})\n            continue\n        videos.append({"id": video_id, "file_names": [str(p) for p in paths], "height": int(seq["height"]),\n                       "width": int(seq["width"]), "length": len(paths), "original_sequence_id": seq.get("id"),\n                       "source_frame_indices_30fps": frame_ids})\n        for object_id, category in seq["categories"].items():\n            if int(category) not in {c["id"] for c in categories}:\n                raise ValueError("Catégorie d\'objet inconnue")\n            masks = []\n            for frame_masks in seq["segmentations"]:\n                counts = frame_masks.get(str(object_id))\n                if counts is not None and not isinstance(counts, str):\n                    raise ValueError("Masque COCO RLE compressé attendu")\n                masks.append(None if not counts else {"size": [int(seq["height"]), int(seq["width"])], "counts": counts})\n            annotations.append({"id": f"{video_id}-o{object_id}", "video_id": video_id,\n                                "category_id": int(category), "segmentations": masks,\n                                "source_object_id": str(object_id), "original_sequence_id": seq.get("id")})\n    result = {"videos": videos, "annotations": annotations, "categories": categories}\n    _validate_annotations(result)\n    return result, {"transform_version": TRANSFORM_VERSION, "input_videos": len(raw["sequences"]),\n                    "eligible_videos": len(videos), "eligible_instances": len(annotations),\n                    "minimum_annotated_frames": minimum_frames, "excluded": excluded,\n                    "labels": "preserved verbatim from the mirror; no action invented",\n                    "ids": "original video directory IDs and object IDs; numeric sequence IDs retained as metadata"}\n\n\ndef _tar_target(staging, member):\n    name = member.name\n    path = PurePosixPath(name)\n    if path.is_absolute() or ".." in path.parts or "\\\\" in name or ":" in name or any(part.rstrip(". ") != part for part in path.parts):\n        raise ValueError("Chemin dangereux ou ambigu dans le TAR")\n    if not (member.isfile() or member.isdir()):\n        raise ValueError("Liens et fichiers spéciaux interdits dans le TAR")\n    target = (staging/Path(*path.parts)).resolve()\n    if not target.is_relative_to(staging):\n        raise ValueError("Chemin hors du dossier d\'extraction")\n    return target\n\n\ndef extract_selected_tar(archive, destination, wanted, archive_sha256=None):\n    """Keep only images named by the selected VIS annotations; publish atomically."""\n    destination = Path(destination).resolve()\n    selection = set(wanted)\n    receipt = {"archive_sha256": archive_sha256 or file_sha256(archive), "selection_sha256": digest(sorted(selection))}\n    marker = destination/".extracted.json"\n    if destination.exists():\n        if marker.is_file() and json.loads(marker.read_text(encoding="utf-8")) == receipt:\n            return destination\n        raise FileExistsError("Extraction existante d\'une autre archive/sélection ; choisir un autre dossier")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    staging = Path(tempfile.mkdtemp(prefix=destination.name+".", dir=destination.parent)).resolve()\n    try:\n        count = 0\n        with tarfile.open(archive, "r:*") as tar:\n            for member in tar:\n                target = _tar_target(staging, member)\n                if not member.isfile() or member.name not in selection:\n                    continue\n                with tar.extractfile(member) as source, atomic_file(target) as stream:\n                    shutil.copyfileobj(source, stream, length=1024*1024)\n                count += 1\n                if count % 2000 == 0:\n                    print(f"Extraction : {count} images VIS…", flush=True)\n        if not count:\n            raise ValueError("Aucune image de l\'archive ne correspond aux annotations VIS")\n        write_json(staging/".extracted.json", receipt)\n        if destination.exists():\n            raise FileExistsError("Dossier apparu pendant l\'extraction")\n        os.replace(staging, destination)\n        return destination\n    except BaseException:\n        if staging.parent == destination.parent and staging.name.startswith(destination.name+"."):\n            shutil.rmtree(staging)\n        raise\n\n\ndef prepare_public_ytvis2019(data_root, source_template, profile, downloader=None):\n    """Public downloads with pinned hashes; never contact CodaLab or ask for login."""\n    root = Path(data_root).resolve()/"ytvis-2019-public"\n    source = dict(source_template)\n    source.update(snapshot="YouTube-VIS-2019/train; public mirrors profile "+profile["profile"],\n                  terms_checked_on=profile["checked_on"], local_media_provenance={\n                      "distribution": "Kaggle VIS annotations + matching original YouTube-VOS frames on Hugging Face",\n                      "annotations_mirror": profile["annotations"], "frames_mirror": profile["frames"]})\n    validate_source(source)  # Inherited source conditions checked before any data acquisition.\n    print("Miroirs publics : Kaggle (labels/masques VIS) + Hugging Face (images source).", flush=True)\n    print("Conditions originales conservées : recherche non commerciale ; annotations CC BY 4.0.", flush=True)\n    print("Conditions :", profile["terms_url"], flush=True)\n    ann = profile["annotations"]\n    ann_file = root/"downloads/annotations.zip"\n    ann_receipt = download_file(ann["url"], ann_file, "annotations", downloader, ann["sha256"])\n    with zipfile.ZipFile(ann_file) as zipped:\n        names = [n for n in zipped.namelist() if n.endswith(".json")]\n        if len(names) != 1:\n            raise ValueError("Une seule annotation JSON attendue dans la révision épinglée")\n        with zipped.open(names[0]) as stream:\n            raw = json.load(stream)\n    minimum_frames = math.ceil(DEFAULT.frames/DEFAULT.fps*profile["source_fps"])\n    data, conversion = normalize_repacked_annotations(raw, minimum_frames)\n    print(f"Annotations : {conversion[\'eligible_videos\']} vidéos admissibles ; exclusions temporelles journalisées.", flush=True)\n    media = profile["frames"]\n    print(f"Images source : archive TAR de {media[\'bytes\']/1e9:.2f} Go. Les fichiers acquis sont réutilisés.", flush=True)\n    media_file = root/"downloads/train.tar"\n    media_receipt = download_file(media["url"], media_file, "frames", downloader, media["sha256"])\n    wanted = {"train/"+name for video in data["videos"] for name in video["file_names"]}\n    extracted = extract_selected_tar(media_file, root/"frames", wanted, media_receipt["sha256"])\n    frames_root = extracted/"train"\n    complete, missing = [], []\n    for video in data["videos"]:\n        absent = [p for p in video["file_names"] if not (frames_root/p).is_file()]\n        if absent:\n            missing.append({"video_id": video["id"], "reasons": ["media_missing_in_vos_mirror"], "missing_frames": absent})\n        else:\n            complete.append(video)\n    if not complete:\n        raise ValueError("Aucune vidéo complète disponible dans le miroir d\'images")\n    ids = {v["id"] for v in complete}\n    data["videos"] = complete\n    data["annotations"] = [a for a in data["annotations"] if a["video_id"] in ids]\n    conversion["media_exclusions"] = missing\n    conversion["available_videos"] = len(complete)\n    conversion["available_instances"] = len(data["annotations"])\n    annotations_path = root/"annotations.normalized.json"\n    # Verify the dimensions of actual image files before publishing readiness.\n    from PIL import Image\n    for video in complete:\n        with Image.open(frames_root/video["file_names"][0]) as image:\n            if image.size != (video["width"], video["height"]):\n                raise ValueError(f"Dimensions incohérentes pour la vidéo {video[\'id\']}")\n    write_json(annotations_path, data)\n    write_json(root/"mirror-conversion-report.json", conversion)\n    source["local_media_provenance"].update(annotations_archive_sha256=ann_receipt["sha256"],\n        media_archive_sha256=media_receipt["sha256"], normalization=TRANSFORM_VERSION,\n        conversion_report_sha256=file_sha256(root/"mirror-conversion-report.json"))\n    source["cadence_evidence"] = {"url": profile["cadence_evidence"], "source_fps": profile["source_fps"],\n                                   "scope": "only sequences with constant filename stride 5 at original 30 fps"}\n    prepared = {"annotations": str(annotations_path), "annotations_sha256": file_sha256(annotations_path),\n                "frames_root": str(frames_root), "source_fps": profile["source_fps"], "source": source,\n                "videos": len(complete), "instances": len(data["annotations"]),\n                "listed_frames": sum(len(v["file_names"]) for v in complete), "mirror_profile": profile["profile"]}\n    write_json(root/"prepared.json", prepared)\n    print(f"Prêt pour D1 : {prepared[\'videos\']} vidéos, {prepared[\'instances\']} instances, sans compte CodaLab.", flush=True)\n    return prepared\n', 'ascii_flux/quality.py': '"""Bind sampled human converter review to an immutable training snapshot."""\nimport copy\nfrom pathlib import Path\nfrom .contract import digest\nfrom .offline import quality_rejections\nfrom .review import load_review_manifest\nfrom .storage import write_json\n\n\ndef calibrated_view(manifest_path, output, quality):\n    manifest = copy.deepcopy(load_review_manifest(manifest_path))\n    if Path(output).resolve().parent != Path(manifest_path).resolve().parent:\n        raise ValueError("Keep a calibrated view beside its immutable shards")\n    if not quality.get("calibrated"):\n        raise ValueError("An explicit calibrated policy is required")\n    original = manifest["fingerprint"]\n    removed = [r for r in manifest["clips"] if quality_rejections(r["quality"], quality)]\n    manifest["clips"] = [r for r in manifest["clips"] if not quality_rejections(r["quality"], quality)]\n    manifest["config"]["quality"] = quality\n    manifest["dataset_version"] += "-calibrated-view-1"\n    manifest["parent_dataset_fingerprint"] = original\n    manifest["report"].update(machine_accepted=len(manifest["clips"]), calibration_rejected=len(removed),\n                              human_review_scope="global converter sample review; not individual acceptance of every clip")\n    manifest.pop("fingerprint")\n    manifest["fingerprint"] = digest(manifest)\n    target = Path(output)\n    if target.exists() and load_review_manifest(target) != manifest:\n        raise FileExistsError("Calibrated views are immutable")\n    write_json(target, manifest)\n    return target\n\n\ndef bind_sample_review(manifest_path, sample_manifest_path, output, reviewer, human_statement):\n    manifest, sample = load_review_manifest(manifest_path), load_review_manifest(sample_manifest_path)\n    if manifest["converter"] != sample["converter"] or manifest["contract"] != sample["contract"]:\n        raise ValueError("Review sample must use the exact converter and representation")\n    quality = manifest["config"]["quality"]\n    if not quality.get("calibrated") or not reviewer.strip() or not human_statement.strip():\n        raise ValueError("Need calibrated policy and an actual identified human review")\n    if quality.get("calibration_sample_fingerprint") != sample["fingerprint"]:\n        raise ValueError("Quality policy was calibrated on a different human review sample")\n    value = {"scope": "converter-sample", "dataset_fingerprint": manifest["fingerprint"],\n             "converter_hash": manifest["converter"]["code_sha256"], "converter_identity": manifest["converter"],\n             "quality_policy": quality, "reviewer": reviewer, "human_statement": human_statement,\n             "sample_dataset_fingerprint": sample["fingerprint"], "sample_available_clips": len(sample["clips"]),\n             "number_of_clips_actually_inspected": None, "per_clip_human_acceptance": False,\n             "decisions": {}, "note": "Automatically filtered clips; sample review does not certify every clip"}\n    write_json(output, value)\n    return Path(output)\n', 'ascii_flux/review.py': '"""Compact standalone review player displaying the stored glyph/color tokens."""\nimport base64\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom .contract import Contract, digest\nfrom .storage import atomic_file, read_clip\n\n\ndef load_review_manifest(dataset):\n    """Recover a completed dataset from disk, independently of notebook globals."""\n    path = Path(dataset)\n    if path.is_dir() or path.suffix != ".json":\n        path = path/"manifest.json"\n    if not path.is_file():\n        raise FileNotFoundError(f"Aucun corpus terminé à cet emplacement : {path}")\n    value = json.loads(path.read_text(encoding="utf-8"))\n    content = dict(value)\n    fingerprint = content.pop("fingerprint", None)\n    if fingerprint != digest(content):\n        raise ValueError("Empreinte du manifest incorrecte ; revue non générée")\n    contract = Contract.from_dict(value["contract"])\n    if contract.hash != value["contract_hash"]:\n        raise ValueError("Contrat du corpus incohérent")\n    root = path.parent.resolve()\n    for name, expected in value["shards"].items():\n        shard = (root/name).resolve()\n        if not shard.is_relative_to(root):\n            raise ValueError("Chemin de shard en dehors du corpus")\n        h = hashlib.sha256()\n        with shard.open("rb") as stream:\n            for block in iter(lambda: stream.read(1024*1024), b""):\n                h.update(block)\n        if h.hexdigest() != expected:\n            raise ValueError(f"Shard corrompu : {name}")\n    return value\n\n\ndef review_page(manifest_path, output):\n    manifest_path = Path(manifest_path)\n    manifest = load_review_manifest(manifest_path)\n    contract = Contract.from_dict(manifest["contract"])\n    clips = []\n    for record in manifest["clips"]:\n        glyphs, colors, _, metadata = read_clip(manifest_path.parent, record, contract)\n        # Compress the exact uint8 tokens, without altering them. Decode only the\n        # selected clip in the browser instead of allocating 40M JSON numbers.\n        packed = gzip.compress(glyphs.tobytes(order="C")+colors.tobytes(order="C"), compresslevel=6, mtime=0)\n        clips.append({"id": record["clip_id"], "tokens_gzip": base64.b64encode(packed).decode("ascii"), "metadata": metadata})\n    data = {"contract": contract.to_dict(), "clips": clips, "fingerprint": manifest["fingerprint"],\n            "converter_hash": manifest["converter"]["code_sha256"], "codec": "gzip-u8-glyphs-colors-1"}\n    payload = json.dumps(data, ensure_ascii=False).replace("<", "\\\\u003c").replace("\\u2028", "\\\\u2028").replace("\\u2029", "\\\\u2029")\n    page = Path(__file__).with_name("review_player.html").read_text(encoding="utf-8").replace("__DATA__", payload)\n    with atomic_file(output) as stream:\n        stream.write(page.encode("utf-8"))\n    return str(Path(output).resolve())\n\n', 'ascii_flux/runtime.py': '"""Native recurrent inference, fixed step clock and independent text worker."""\nimport json\nfrom collections import OrderedDict\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom pathlib import Path\nimport threading\nimport time\nimport numpy as np\nimport torch\nfrom .contract import normalize_text\n\n\nclass FlowRuntime:\n    def __init__(self, model, encoder, fps=24., debounce=.25):\n        if fps <= 0 or debounce < 0:\n            raise ValueError("Invalid runtime clock")\n        self.model, self.encoder, self.fps, self.debounce = model.eval(), encoder, fps, debounce\n        self.device = next(model.parameters()).device\n        self.state = model.initial_state()\n        self.text = torch.zeros(1, model.config.text_dim, device=self.device)\n        self.lock = threading.Condition()\n        self.stopped = threading.Event()\n        self.revision, self.applied_revision = 0, 0\n        self.prompt, self.revised_at, self.reset_pending = "", 0., False\n        self.steps, self.dropped_ticks, self.step_ms, self.encode_ms = 0, 0, None, None\n        self.error = None\n        self.client_sequences = OrderedDict()\n        self.input_client, self.input_sequence = None, None\n        self.condition_pending = False\n        self.frame = {"glyphs": np.zeros((model.contract.rows, model.contract.cols), np.uint8).tolist(),\n                      "colors": np.full((model.contract.rows, model.contract.cols), model.contract.empty_color, np.uint8).tolist()}\n        self.threads = []\n\n    def set_prompt(self, prompt, client=None, sequence=None):\n        prompt = normalize_text(prompt)\n        if len(prompt) > 2000:\n            raise ValueError("Prompt too long")\n        if client is not None or sequence is not None:\n            if not isinstance(client, str) or not 1 <= len(client) <= 80:\n                raise ValueError("Invalid input session")\n            if not isinstance(sequence, int) or isinstance(sequence, bool) or sequence < 0:\n                raise ValueError("Invalid input sequence")\n        with self.lock:\n            if client is not None:\n                if sequence <= self.client_sequences.get(client, -1):\n                    return self.revision\n                self.client_sequences[client] = sequence\n                self.client_sequences.move_to_end(client)\n                if len(self.client_sequences) > 32:\n                    self.client_sequences.popitem(last=False)\n                self.input_client, self.input_sequence = client, sequence\n            if prompt != self.prompt:\n                self.prompt, self.revised_at = prompt, time.monotonic()\n                self.revision += 1\n                self.error = None\n                self.lock.notify_all()\n            return self.revision\n\n    def reset(self):\n        with self.lock:\n            self.reset_pending = True\n\n    def _encode_loop(self):\n        attempted = 0\n        while not self.stopped.is_set():\n            with self.lock:\n                if attempted == self.revision:\n                    self.lock.wait(.1)\n                    continue\n                delay = self.debounce - (time.monotonic()-self.revised_at)\n                if delay > 0:\n                    self.lock.wait(min(delay, .1))\n                    continue\n                revision, prompt = self.revision, self.prompt\n                attempted = revision\n            begin = time.perf_counter()\n            try:\n                vector = self.encoder.encode([prompt])\n                if vector.shape != (1, self.model.config.text_dim) or not np.isfinite(vector).all():\n                    raise ValueError("Invalid text embedding")\n                vector = torch.as_tensor(vector, dtype=torch.float32, device=self.device)\n                with self.lock:\n                    if revision == self.revision:\n                        self.text = vector\n                        self.condition_pending = True\n                        self.applied_revision = revision\n                        self.encode_ms = (time.perf_counter()-begin)*1000\n            except Exception as exc:\n                with self.lock:\n                    if revision == self.revision:\n                        self.error = f"Text encoding failed: {exc}"\n\n    @torch.inference_mode()\n    def tick(self):\n        with self.lock:\n            text = self.text\n            if self.reset_pending:\n                self.state = self.model.initial_state()\n                self.reset_pending = False\n            elif self.condition_pending and hasattr(self.model,"recondition"):\n                self.state = self.model.recondition(self.state)\n            self.condition_pending = False\n        begin = time.perf_counter()\n        state, _, _ = self.model.step(self.state, text)\n        if not torch.isfinite(state).all():\n            raise FloatingPointError("Non-finite recurrent state")\n        glyphs, colors = self.model.tokens(state)\n        self.state = state\n        frame = {"glyphs": glyphs[0].cpu().tolist(), "colors": colors[0].cpu().tolist()}\n        with self.lock:\n            self.frame = frame\n            self.steps += 1\n            self.step_ms = (time.perf_counter()-begin)*1000\n\n    def _step_loop(self):\n        deadline = time.monotonic()\n        try:\n            while not self.stopped.is_set():\n                self.tick()\n                deadline += 1/self.fps\n                now = time.monotonic()\n                if now > deadline:\n                    missed = int((now-deadline)*self.fps)+1\n                    with self.lock:\n                        self.dropped_ticks += missed\n                    deadline += missed/self.fps\n                self.stopped.wait(max(0., deadline-time.monotonic()))\n        except Exception as exc:\n            with self.lock:\n                self.error = f"Model stopped: {exc}"\n            self.stopped.set()\n\n    def snapshot(self):\n        with self.lock:\n            return {**self.frame, "steps": self.steps, "requested_fps": self.fps,\n                    "dropped_ticks": self.dropped_ticks, "step_ms": self.step_ms, "encode_ms": self.encode_ms,\n                    "revision": self.revision, "applied_revision": self.applied_revision, "error": self.error,\n                    "input_client": self.input_client, "input_sequence": self.input_sequence}\n\n    def start(self):\n        if self.threads:\n            raise RuntimeError("Runtime already started")\n        self.threads = [threading.Thread(target=self._encode_loop, daemon=True),\n                        threading.Thread(target=self._step_loop, daemon=True)]\n        for thread in self.threads:\n            thread.start()\n\n    def close(self):\n        self.stopped.set()\n        with self.lock:\n            self.lock.notify_all()\n        for thread in self.threads:\n            thread.join(timeout=2)\n\n\ndef make_server(runtime, provenance, port=8766):\n    page = Path(__file__).with_name("player.html").read_bytes()\n\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, *_):\n            pass\n\n        def send(self, status, data, content_type="application/json"):\n            body = data if isinstance(data, bytes) else json.dumps(data).encode("utf-8")\n            self.send_response(status)\n            self.send_header("Content-Type", content_type + "; charset=utf-8")\n            self.send_header("Content-Length", str(len(body)))\n            self.send_header("Cache-Control", "no-store")\n            self.send_header("X-Content-Type-Options", "nosniff")\n            self.end_headers()\n            try:\n                self.wfile.write(body)\n            except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):\n                pass  # The tab closed or canceled an obsolete request.\n\n        def do_GET(self):\n            if self.path == "/":\n                self.send(200, page, "text/html")\n            elif self.path == "/frame":\n                self.send(200, runtime.snapshot())\n            elif self.path == "/config":\n                self.send(200, {"contract": runtime.model.contract.to_dict(), "provenance": provenance,\n                                "application": "ascii-flux-native", "input_debounce_ms": runtime.debounce*1000,\n                                "execution": "local Python model; browser renders glyph tokens", "recognition_validated": False})\n            else:\n                self.send(404, {"error": "Not found"})\n\n        def do_POST(self):\n            try:\n                origin = self.headers.get("Origin")\n                if origin and origin != f"http://{self.headers.get(\'Host\')}":\n                    self.send(403, {"error": "Origin mismatch"})\n                    return\n                if not self.headers.get("Content-Type", "").startswith("application/json"):\n                    raise ValueError("JSON required")\n                length = int(self.headers.get("Content-Length", "0"))\n                if not 0 < length <= 16000:\n                    raise ValueError("Invalid request size")\n                value = json.loads(self.rfile.read(length))\n                if self.path == "/prompt":\n                    if not isinstance(value.get("prompt"), str):\n                        raise ValueError("Prompt must be text")\n                    self.send(200, {"revision": runtime.set_prompt(value["prompt"], value.get("client"), value.get("sequence"))})\n                elif self.path == "/reset":\n                    runtime.reset()\n                    self.send(200, {"reset": "scheduled"})\n                else:\n                    self.send(404, {"error": "Not found"})\n            except (ValueError, KeyError) as exc:\n                self.send(400, {"error": str(exc)})\n\n    return ThreadingHTTPServer(("127.0.0.1", port), Handler)\n\n\ndef serve(checkpoint, port=8766, fps=24., device="cpu", allow_development=False, debounce=0.):\n    from .text import TextEncoder\n    from .inference import load_inference\n    model, provenance = load_inference(checkpoint, device, allow_development)\n    if provenance.get("synthetic_encoder"):\n        raise ValueError("A fixture checkpoint has no SigLIP semantics and cannot drive the prompt demo")\n    encoder = TextEncoder(device)\n    runtime = FlowRuntime(model, encoder, fps, debounce=debounce)\n    server = make_server(runtime, provenance, port)\n    runtime.start()\n    print(f"ASCII Flux research prototype: http://127.0.0.1:{server.server_port}", flush=True)\n    try:\n        server.serve_forever()\n    except KeyboardInterrupt:\n        pass\n    finally:\n        server.server_close()\n        runtime.close()\n\n', 'ascii_flux/scale.py': '"""Resumable D2 conversion in bounded, independently committed shard jobs."""\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom PIL import Image\nfrom .catalogue import Catalogue, writer_lock\nfrom .contract import DEFAULT, digest, normalize_text\nfrom .factory import decode_mask, file_sha256, split_videos, validate_source\nfrom .offline import ConverterConfig, convert_clip, converter_identity, quality_rejections\nfrom .storage import write_json, write_shard\n\n\ndef inventory(annotations_path, source_fps, contract=DEFAULT):\n    if not source_fps or not math.isfinite(source_fps) or source_fps <= 0:\n        raise ValueError("Record and verify the annotated frame cadence before conversion")\n    data = json.loads(Path(annotations_path).read_text(encoding="utf-8"))\n    videos = {v["id"]: v for v in data["videos"]}\n    clips = sum(int((len(videos[a[\'video_id\']][\'file_names\'])/source_fps+1e-8)//(contract.frames/contract.fps))\n                for a in data["annotations"])\n    return {"videos": len(videos), "instances": len(data["annotations"]), "nonoverlapping_full_clips_upper_bound": clips,\n            "grid_slots_upper_bound": clips*contract.frames, "source_frame_fps": source_fps,\n            "note": "upper bound before missing masks, quality rejection and duplicate checks"}\n\n\ndef _convert_job(tasks, frames_root, source, config, source_fps, contract, annotation_hash, output):\n    output = Path(output)\n    if (output/"manifest.json").exists():\n        return output/"manifest.json"\n    output.mkdir(parents=True, exist_ok=True)\n    converter = ConverterConfig(**config["converter"])\n    identity = converter_identity(converter)\n    root = Path(frames_root).resolve()\n    clips, records, rejects = [], [], []\n    for ann, video, category, split, start in tasks:\n        clip_id = "clip-"+digest([source["source"], str(video["id"]), str(ann["id"]), start])[:32]\n        base = {"clip_id": clip_id, "video_id": str(video["id"]), "instance_id": str(ann["id"]),\n                "source": source["source"], "category": category, "split": split}\n        indices = np.floor((start+np.arange(contract.frames)/contract.fps)*source_fps+1e-7).astype(int)\n        try:\n            # Decode repeated source frames once; conversion still processes all\n            # time steps and retains the same stabilizer as the D1 converter.\n            decoded, hashes = {}, {}\n            for index in sorted(set(indices.tolist())):\n                path = (root/video["file_names"][index]).resolve()\n                if not path.is_relative_to(root):\n                    raise ValueError("media_path_escapes_root")\n                with Image.open(path) as image:\n                    frame = np.array(image.convert("RGB"))\n                if frame.shape[:2] != (video["height"], video["width"]):\n                    raise ValueError("source_dimensions_mismatch")\n                mask = decode_mask(ann["segmentations"][index], video["height"], video["width"])\n                decoded[index] = (frame, mask)\n                hashes[str(index)] = file_sha256(path)\n            glyphs, colors, boxes, metrics = convert_clip([decoded[i][0] for i in indices],\n                                                         [decoded[i][1] for i in indices], contract, converter)\n            reasons = quality_rejections(metrics, config["quality"])\n            if reasons:\n                rejects.append({**base, "reasons": reasons, "quality": metrics})\n                continue\n            label = normalize_text(category)\n            metadata = {**base, "provenance": source, "annotation_sha256": annotation_hash,\n                        "source_video_uid": f"{source[\'source\']}:{video[\'id\']}",\n                        "source_frame_sha256": hashes, "source_frame_indices": indices.tolist(),\n                        "source_frame_paths": [video["file_names"][i] for i in indices],\n                        "source_frame_fps": source_fps, "sampled_unique_source_frames": len(decoded),\n                        "source_frame_indices_30fps": [video["source_frame_indices_30fps"][i] for i in indices]\n                            if "source_frame_indices_30fps" in video else None,\n                        "source_object_id": ann.get("source_object_id"), "original_sequence_id": video.get("original_sequence_id"),\n                        "source_url": video.get("url", source["url"]), "start_seconds": start,\n                        "contract": contract.to_dict(), "contract_hash": contract.hash, "converter": identity,\n                        "quality": metrics, "mask_method": "source-instance-mask", "original_label": category,\n                        "original_caption": video.get("caption"), "normalized_label": label,\n                        "label_scope": "subject only; action not annotated", "rewrite": None,\n                        "synthetic": bool(source.get("synthetic", False)), "human_review": "pending",\n                        "box_format": "xyxy source pixels; outside-image padding permitted"}\n            clips.append((clip_id, glyphs, colors, boxes, metadata))\n            records.append({**base, "shard": "shard-00000.tar", "normalized_label": label,\n                            "synthetic": metadata["synthetic"], "quality": metrics})\n            if config.get("horizontal_flip", False):\n                if contract.glyphs != " .:+*#@":\n                    raise ValueError("Horizontal augmentation needs a verified mirror mapping for contour glyphs")\n                augmented_id = clip_id+"-flip"\n                augmentation = {"kind": "horizontal_flip", "parent_clip_id": clip_id,\n                                "new_source_observations": False,\n                                "box_coordinates": "original source trajectory; rendered crop is mirrored"}\n                augmented_meta = dict(metadata, clip_id=augmented_id, augmentation=augmentation)\n                clips.append((augmented_id, glyphs[..., ::-1].copy(), colors[..., ::-1].copy(), boxes, augmented_meta))\n                records.append(dict(records[-1], clip_id=augmented_id, augmentation=augmentation))\n        except (ValueError, OSError, IndexError) as error:\n            rejects.append({**base, "reasons": [type(error).__name__+":"+str(error)]})\n    shards = {}\n    if clips:\n        write_shard(output/"shard-00000.tar", clips, contract)\n        shards["shard-00000.tar"] = file_sha256(output/"shard-00000.tar")\n    report = {"attempted_full_clips": len(tasks), "machine_accepted": len(records), "rejected_records": len(rejects),\n              "reasons": dict(Counter(reason for r in rejects for reason in r["reasons"])),\n              "split_counts": dict(Counter(r["split"] for r in records)), "human_review_completed": False,\n              "D1_complete": False, "status": "D2 machine candidates; no human recognition claim"}\n    manifest = {"dataset_version": config["dataset_version"], "contract": contract.to_dict(),\n                "contract_hash": contract.hash, "source": source, "converter": identity,\n                "annotation_sha256": annotation_hash, "config": config, "clips": records, "shards": shards,\n                "report": report, "adapter_sha256": file_sha256(__file__)}\n    manifest["fingerprint"] = digest(manifest)\n    write_json(output/"rejections.json", rejects)\n    write_json(output/"manifest.json", manifest)  # commit marker, written last\n    return output/"manifest.json"\n\n\ndef build_scale(annotations_path, frames_root, source, config, output, source_fps, workers=2, max_jobs=None,\n                contract=DEFAULT):\n    if source["source"] == "YouTube-VIS":\n        validate_source(source)\n    elif source["source"] == "LV-VIS":\n        for key in ("snapshot", "url", "terms_url", "terms_checked_on", "annotations_license", "media_conditions", "cadence_evidence"):\n            if not source.get(key):\n                raise ValueError("Missing LV-VIS provenance: "+key)\n        if source["annotations_license"] != "CC-BY-NC-SA-4.0" or source.get("usage_scope") != "non-commercial-research":\n            raise ValueError("Retain LV-VIS original noncommercial research conditions")\n    else:\n        raise ValueError("No verified masked-video adapter for this source")\n    if workers < 1 or config["shard_size"] < 1 or (max_jobs is not None and max_jobs < 1):\n        raise ValueError("Positive worker and job limits required")\n    summary = inventory(annotations_path, source_fps, contract)\n    output = Path(output).resolve()\n    identity = {"annotation_sha256": file_sha256(annotations_path), "source": source, "config": config,\n                "contract": contract.to_dict(), "converter": converter_identity(ConverterConfig(**config["converter"])),\n                "adapter_sha256": file_sha256(__file__), "source_fps": source_fps,\n                "frames_root": str(Path(frames_root).resolve())}\n    plan_id = digest(identity)\n    with writer_lock(output), Catalogue(output, contract, config.get("target_grids", 1_000_000)) as catalogue:\n        plan = output/"plans"/plan_id\n        plan.mkdir(parents=True, exist_ok=True)\n        write_json(plan/"plan.json", {**identity, "inventory": summary})\n        data = json.loads(Path(annotations_path).read_text(encoding="utf-8"))\n        videos = {v["id"]: v for v in data["videos"]}\n        categories = {c["id"]: c["name"] for c in data["categories"]}\n        # YouTube-VIS preserves exactly the D1 partition rule across chunks.\n        splits = split_videos(videos, data["annotations"], categories, config["heldout_categories"], config["seed"])\n        tasks = []\n        seconds = contract.frames/contract.fps\n        for ann in sorted(data["annotations"], key=lambda a: str(a["id"])):\n            video = videos[ann["video_id"]]\n            count = int((len(video["file_names"])/source_fps+1e-8)//seconds)\n            for i in range(count):\n                tasks.append((ann, video, categories[ann["category_id"]], splits[video["id"]], i*seconds))\n        jobs = [tasks[i:i+config["shard_size"]] for i in range(0, len(tasks), config["shard_size"])]\n        pending = []\n        for number, batch in enumerate(jobs):\n            marker = plan/f"job-{number:06d}"/"manifest.json"\n            if marker.exists():\n                # Re-ingest after a crash between file publication and SQL commit.\n                catalogue.ingest(marker)\n            else:\n                pending.append((number, batch))\n        # A bounded map keeps at most `workers` full frame buffers in memory.\n        import cv2\n        cv2.setNumThreads(1)\n        start = time.monotonic()\n        def work(item):\n            number, batch = item\n            return _convert_job(batch, frames_root, source, config, source_fps, contract,\n                                identity["annotation_sha256"], plan/f"job-{number:06d}")\n        completed = 0\n        with ThreadPoolExecutor(max_workers=workers) as pool:\n            # Explicit bounded submission (Executor.map queues everything on\n            # Python 3.11/3.12). Finish each window before submitting the next.\n            for first in range(0, len(pending), workers):\n                window = pending[first:first+workers]\n                if max_jobs is not None:\n                    window = window[:max(0, max_jobs-completed)]\n                if not window:\n                    break\n                for manifest_path in pool.map(work, window):\n                    catalogue.ingest(manifest_path)\n                    completed += 1\n                    report = catalogue.report()\n                    report.update(jobs_completed_this_call=completed, jobs_total=len(jobs),\n                                  elapsed_seconds=round(time.monotonic()-start, 1), inventory=summary)\n                    write_json(output/"progress.json", report)\n                    print(json.dumps({k: report[k] for k in ("clips", "grid_slots", "distinct_token_grids", "jobs_completed_this_call", "jobs_total")}), flush=True)\n        report = catalogue.report()\n        write_json(output/"report.json", report)\n        return report\n\n', 'ascii_flux/shape_corpus.py': '"""Offline teacher shape corpus: words -> teacher images -> cut-outs -> glyph grids.\n\nTagged ``synthetic: teacher-t2i`` everywhere. Resumable by shard; each shard\nis an NPZ of tokens plus a JSON sidecar published last, atomically.\nWords are split by a hash of the word: reserved words never train.\n"""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport time\nimport numpy as np\nfrom .contract import canonical, normalize_text\nfrom .storage import atomic_file, write_json\nfrom .structure import STRUCTURE, to_glyphs\n\nVOCABULARY_VERSION = "lvvis-imagenet12k-1"\n\n\ndef _timm_info():\n    import timm.data\n    return Path(timm.data.__file__).parent/"_info"\n\n\ndef build_vocabulary(lvvis_annotations):\n    """Ordered, de-duplicated words with their origins; common concepts first."""\n    info = _timm_info()\n    lemmas = dict(line.rstrip("\\n").split("\\t", 1) for line in (info/"imagenet_synset_to_lemma.txt").read_text(encoding="utf-8").splitlines() if "\\t" in line)\n    order, entries = [], {}\n\n    def add(word, origin):\n        word = normalize_text(word.split(",")[0].replace("_", " "))\n        word = " ".join(word.split())\n        if not word or any(ch.isdigit() for ch in word) or len(word) > 40:\n            return\n        if word not in entries:\n            entries[word] = {"word": word, "origins": []}\n            order.append(word)\n        entries[word]["origins"].append(origin)\n\n    for synset in (info/"imagenet_synsets.txt").read_text(encoding="utf-8").split():\n        add(lemmas[synset], f"imagenet1k:{synset}")\n    categories = json.loads(Path(lvvis_annotations).read_text(encoding="utf-8"))["categories"]\n    for category in categories:\n        name = category["name"].split("_(")[0]\n        add(name, f"lvvis:{category[\'id\']}")\n    for synset in (info/"imagenet12k_synsets.txt").read_text(encoding="utf-8").split():\n        if synset in lemmas:\n            add(lemmas[synset], f"imagenet12k:{synset}")\n    return [entries[w] for w in order]\n\n\ndef split_of(word, reserved_percent=5):\n    value = int(hashlib.sha256(word.encode("utf-8")).hexdigest()[:8], 16) % 100\n    return "test_unseen" if value < reserved_percent else "val" if value < reserved_percent+3 else "train"\n\n\ndef build_shape_corpus(output, lvvis_annotations, keep=3, attempts=8, shard_words=100, max_words=None, device="cuda"):\n    from .teacher import TEMPLATES, Teacher, check_mask\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    vocabulary = build_vocabulary(lvvis_annotations)[:max_words]\n    write_json(output/"vocabulary.json", {"version": VOCABULARY_VERSION, "words": vocabulary})\n    done = set()\n    for sidecar in output.glob("shard-*.json"):\n        done.update(r["word"] for r in json.loads(sidecar.read_text(encoding="utf-8"))["words"])\n    pending = [entry for entry in vocabulary if entry["word"] not in done]\n    shard_index = len(list(output.glob("shard-*.json")))\n    teacher = Teacher(device)\n    provenance = teacher.provenance() | {"contract": STRUCTURE.to_dict(), "contract_hash": STRUCTURE.hash,\n                                         "vocabulary_version": VOCABULARY_VERSION, "keep": keep, "attempts": attempts}\n    write_json(output/"provenance.json", provenance)\n    started, processed = time.monotonic(), 0\n    for first in range(0, len(pending), shard_words):\n        glyphs, colors, words, reasons = [], [], [], {}\n        for entry in pending[first:first+shard_words]:\n            word, base = entry["word"], int(hashlib.sha256(entry["word"].encode("utf-8")).hexdigest()[:8], 16)\n            accepted = []\n            for batch in range(0, attempts, 4):\n                if len(accepted) >= keep:\n                    break\n                seeds = [base+batch+i for i in range(4)]\n                templates = [(batch+i) % len(TEMPLATES) for i in range(4)]\n                images = teacher.draw([TEMPLATES[t].format(word=word) for t in templates], seeds)\n                for image, mask, seed, template in zip(images, teacher.masks(images), seeds, templates):\n                    reason = check_mask(mask)\n                    if reason is None and len(accepted) < keep:\n                        g, c = to_glyphs(image, mask, STRUCTURE.palette)\n                        if (g != 0).mean() < .02:\n                            reason = "grid_nearly_empty"\n                        else:\n                            accepted.append((g, c, seed, template))\n                    if reason:\n                        reasons[reason] = reasons.get(reason, 0)+1\n            record = {"word": word, "origins": entry["origins"], "split": split_of(word), "samples": [],\n                      "synthetic": "teacher-t2i"}\n            for g, c, seed, template in accepted:\n                record["samples"].append({"index": len(glyphs), "seed": seed, "template": template})\n                glyphs.append(g)\n                colors.append(c)\n            words.append(record)\n            processed += 1\n        name = f"shard-{shard_index:05d}"\n        payload = io.BytesIO()\n        np.savez_compressed(payload, glyphs=np.asarray(glyphs, np.uint8).reshape(-1, STRUCTURE.rows, STRUCTURE.cols),\n                            colors=np.asarray(colors, np.uint8).reshape(-1, STRUCTURE.rows, STRUCTURE.cols))\n        with atomic_file(output/f"{name}.npz") as stream:\n            stream.write(payload.getvalue())\n        sha = hashlib.sha256(payload.getvalue()).hexdigest()\n        write_json(output/f"{name}.json", {"npz_sha256": sha, "contract_hash": STRUCTURE.hash, "words": words,\n                                           "rejections": reasons})\n        shard_index += 1\n        elapsed = time.monotonic()-started\n        write_json(output/"progress.json", {"words_done": len(done)+processed, "words_total": len(vocabulary),\n                                            "seconds_per_word": elapsed/max(processed, 1),\n                                            "last_shard": name, "synthetic": "teacher-t2i"})\n        print(canonical({"shard": name, "words_done": len(done)+processed, "of": len(vocabulary),\n                         "s_per_word": round(elapsed/max(processed, 1), 2), "rejections": reasons}), flush=True)\n', 'ascii_flux/storage.py': '"""Atomic immutable shards. No pickle is used for dataset material."""\nfrom contextlib import contextmanager\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport tarfile\nimport tempfile\nimport numpy as np\nfrom .contract import canonical\n\n\n@contextmanager\ndef atomic_file(path, mode="wb"):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=path.name + ".", suffix=".tmp", dir=path.parent)\n    try:\n        with os.fdopen(fd, mode) as stream:\n            yield stream\n            stream.flush()\n            os.fsync(stream.fileno())\n        os.replace(temporary, path)\n    finally:\n        if os.path.exists(temporary):\n            os.unlink(temporary)\n\n\ndef write_json(path, value):\n    with atomic_file(path) as stream:\n        stream.write((canonical(value) + "\\n").encode("utf-8"))\n\n\ndef validate_tokens(glyphs, colors, box, contract):\n    expected = (contract.frames, contract.rows, contract.cols)\n    if glyphs.shape != expected or colors.shape != expected or glyphs.dtype != np.uint8 or colors.dtype != np.uint8:\n        raise ValueError("Tokens must be uint8 [T,R,C] matching the contract")\n    if box.shape != (contract.frames, 4) or not np.isfinite(box).all():\n        raise ValueError("Boxes must be finite [T,4]")\n    if (box[:, 2:] <= box[:, :2]).any():\n        raise ValueError("Invalid xyxy boxes")\n    if glyphs.max() >= len(contract.glyphs):\n        raise ValueError("Glyph outside vocabulary")\n    occupied = glyphs != 0\n    if np.any(colors[~occupied] != contract.empty_color) or np.any(colors[occupied] >= len(contract.palette)):\n        raise ValueError("Invalid color, or a color on an empty cell")\n\n\ndef write_shard(path, clips, contract):\n    """Publish tokens and their metadata together, after fsync."""\n    with atomic_file(path) as stream:\n        with tarfile.open(fileobj=stream, mode="w") as tar:\n            for clip_id, glyphs, colors, boxes, metadata in clips:\n                if not clip_id or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for c in clip_id):\n                    raise ValueError("Unsafe clip ID")\n                validate_tokens(glyphs, colors, boxes, contract)\n                payload = io.BytesIO()\n                np.savez_compressed(payload, glyphs=glyphs, colors=colors, box=boxes)\n                for suffix, data in (("npz", payload.getvalue()), ("json", canonical(metadata).encode("utf-8"))):\n                    info = tarfile.TarInfo(f"{clip_id}.{suffix}")\n                    info.size = len(data)\n                    tar.addfile(info, io.BytesIO(data))\n\n\ndef read_clip(root, record, contract):\n    root = Path(root).resolve()\n    shard = (root / record["shard"]).resolve()\n    if not shard.is_relative_to(root):\n        raise ValueError("Shard path escapes dataset")\n    with tarfile.open(shard, "r") as tar:\n        clip_id = record["clip_id"]\n        with tar.extractfile(f"{clip_id}.npz") as stream:\n            with np.load(io.BytesIO(stream.read()), allow_pickle=False) as arrays:\n                glyphs, colors, box = (arrays[key] for key in ("glyphs", "colors", "box"))\n        metadata = json.loads(tar.extractfile(f"{clip_id}.json").read())\n    contract.require(metadata["contract"])\n    validate_tokens(glyphs, colors, box, contract)\n    return glyphs, colors, box, metadata\n\n', 'ascii_flux/structure.py': '"""Structure-aware glyph conversion for the offline data factory.\n\nDensity ramps fill a subject\'s interior and leave its outline to blurred\ngradients; recognizable ASCII art draws the outline with oriented strokes\n(see structure-based ASCII art, Xu, Zhang & Wong, SIGGRAPH 2010). Here the\nsilhouette boundary and strong interior edges receive oriented glyphs, while\nthe interior keeps a light tone ramp. Never used at inference.\n"""\nfrom dataclasses import replace\nimport cv2\nimport numpy as np\nfrom .contract import Contract\n\n# Ordered vocabulary: space first, density ramp, then oriented strokes.\nSTRUCTURE_GLYPHS = " .:+*#@-_|/\\\\()\'"\nSTRUCTURE = replace(Contract(), version="ascii-v5-structure-rgb64-1", glyphs=STRUCTURE_GLYPHS,\n                    glyph_version="ramp-strokes-1")\nRAMP = STRUCTURE_GLYPHS.index("."), STRUCTURE_GLYPHS.index("@")\n\n\ndef segment_plain_background(rgb, tolerance=28.):\n    """Subject mask for a teacher image prompted on a plain light background.\n\n    Background is estimated from the border; the mask is the largest connected\n    component that differs from it. Returns (mask, reason or None).\n    """\n    image = rgb.astype(np.float32)\n    border = np.concatenate([image[:4].reshape(-1, 3), image[-4:].reshape(-1, 3),\n                             image[:, :4].reshape(-1, 3), image[:, -4:].reshape(-1, 3)])\n    background = np.median(border, 0)\n    if np.mean(np.linalg.norm(border-background, axis=1) < 2*tolerance) < .6:\n        return None, "background_not_plain"\n    # Studio backdrops are smooth gradients rather than one color: model the\n    # background as a heavily blurred image of the border-filled frame.\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)\n    h, w = lab.shape[:2]\n    small = cv2.resize(lab, (16, 16), interpolation=cv2.INTER_AREA)\n    known = np.zeros((16, 16), bool); known[[0, -1]] = True; known[:, [0, -1]] = True\n    plane = np.where(known[..., None], small, np.median(small[known], 0))\n    for _ in range(30):\n        plane = np.where(known[..., None], small, cv2.blur(plane, (3, 3)))\n    bg_lab = cv2.resize(plane, (w, h), interpolation=cv2.INTER_CUBIC)\n    distance = np.linalg.norm(lab-bg_lab, axis=2)\n    mask = (distance > tolerance).astype(np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))\n    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if count < 2:\n        return None, "no_subject"\n    largest = 1+int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n    mask = labels == largest\n    # Fill holes enclosed by the subject (white belly, gaps between features).\n    flood = (~mask).astype(np.uint8)\n    cv2.floodFill(flood, None, (0, 0), 0)\n    mask = mask | flood.astype(bool)\n    h, w = mask.shape\n    if mask[:2].any() or mask[-2:].any() or mask[:, :2].any() or mask[:, -2:].any():\n        return mask, "subject_touches_border"\n    if mask.mean() < .03:\n        return mask, "subject_too_small"\n    return mask, None\n\n\ndef _fit(mask, contract, margin):\n    """Crop box with the subject\'s aspect ratio preserved inside the grid."""\n    ys, xs = np.nonzero(mask)\n    y0, y1, x0, x1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1\n    grid_aspect = contract.rows*contract.cell_height_width/contract.cols  # height/width in pixels\n    h, w = (y1-y0)*(1+2*margin), (x1-x0)*(1+2*margin)\n    if h/w > grid_aspect:\n        w = h/grid_aspect\n    else:\n        h = w*grid_aspect\n    cy, cx = (y0+y1)/2, (x0+x1)/2\n    return cy-h/2, cx-w/2, h, w\n\n\ndef to_glyphs(rgb, mask, palette, contract=STRUCTURE, margin=.06, sub=8, edge_threshold=.5):\n    """One RGB frame plus subject mask to (glyphs, colors) uint8 [R, C]."""\n    rows, cols = contract.rows, contract.cols\n    top, left, h, w = _fit(mask, contract, margin)\n    sx_cell = max(2, int(round(sub/contract.cell_height_width)))\n    hs, ws = rows*sub, cols*sx_cell\n    # Sample the crop at sub-cell resolution; outside the frame is background.\n    matrix = np.float32([[w/ws, 0, left], [0, h/hs, top]])\n    inverse = cv2.invertAffineTransform(matrix)\n    fine_mask = cv2.warpAffine(mask.astype(np.float32), inverse, (ws, hs), flags=cv2.INTER_AREA,\n                               borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n    fine_rgb = cv2.warpAffine(rgb, inverse, (ws, hs), flags=cv2.INTER_AREA, borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n    sy, sx = hs//rows, ws//cols\n    cell = lambda a: a.reshape(rows, sy, cols, sx, *a.shape[2:]).mean((1, 3))\n    coverage = cell(fine_mask)\n    # Silhouette boundary orientation from the mask gradient.\n    blurred = cv2.GaussianBlur(fine_mask, (0, 0), sy/3)\n    gx, gy = cv2.Sobel(blurred, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(blurred, cv2.CV_32F, 0, 1, ksize=3)\n    # Interior structure: strong luminance edges inside the subject only.\n    gray = cv2.cvtColor(fine_rgb, cv2.COLOR_RGB2GRAY)\n    canny = cv2.Canny(cv2.GaussianBlur(gray, (0, 0), 1.2), 60, 140).astype(np.float32)/255*(cv2.erode(fine_mask, np.ones((5, 5)))>.5)\n    ix, iy = cv2.Sobel(gray.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(gray.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)\n    boundary = cell(np.hypot(gx, gy) > 1e-3)*(coverage > .08)*(coverage < .92)\n    interior_edges = cell(canny)\n    gx_c, gy_c = cell(gx), cell(gy)\n    ix_c, iy_c = cell(ix*canny), cell(iy*canny)\n    luminance = cell(gray.astype(np.float32))/255\n    glyphs = np.zeros((rows, cols), np.uint8)\n    inside = coverage > .35\n    # Interior: light ramp from tone, kept sparse so outlines dominate.\n    tone = np.clip(1-luminance, 0, 1)*.55+.05\n    glyphs[inside] = np.round(RAMP[0]+tone[inside]*(RAMP[1]-RAMP[0])).astype(np.uint8)\n    index = {g: i for i, g in enumerate(contract.glyphs)}\n\n    def oriented(dx, dy, bottom):\n        # Gradient is normal to the edge; the stroke follows the tangent.\n        angle = (np.degrees(np.arctan2(dy, dx))+180) % 180\n        if angle < 22.5 or angle >= 157.5:\n            return index["|"]\n        if angle < 67.5:\n            return index["/"]\n        if angle < 112.5:\n            return index["_"] if bottom else index["-"]\n        return index["\\\\"]\n\n    edge = (interior_edges > .22) & inside\n    for r, c in zip(*np.nonzero(edge)):\n        glyphs[r, c] = oriented(ix_c[r, c], iy_c[r, c]*contract.cell_height_width, False)\n    outline = (boundary > edge_threshold*.2) & (coverage > .12)\n    for r, c in zip(*np.nonzero(outline)):\n        # gy < 0 means mask increases upward: this is the subject\'s lower edge.\n        glyphs[r, c] = oriented(gx_c[r, c], gy_c[r, c]*contract.cell_height_width, gy_c[r, c] < 0)\n    glyphs[coverage <= .12] = 0\n    # Colors: mean subject color per cell, quantized to the palette.\n    weights = fine_mask[..., None]\n    colors_sum = cell(fine_rgb.astype(np.float32)*weights)\n    mean = colors_sum/np.maximum(coverage[..., None], 1e-6)\n    pal = np.asarray(palette, np.float32)\n    # Brighten toward display on black, as in the v4.2 color rendering.\n    mean = np.clip(mean*1.35+45, 0, 255)\n    colors = np.argmin(((mean[..., None, :]-pal)**2).sum(-1), -1).astype(np.uint8)\n    colors[glyphs == 0] = contract.empty_color\n    return glyphs, colors\n', 'ascii_flux/teacher.py': '"""Offline image teacher for the shape corpus. Never imported at inference.\n\nA text-to-image model draws whole subjects for many words; a salient-object\nsegmenter cuts them out; ``structure.to_glyphs`` turns them into native grids.\nEvery record produced from it is tagged ``synthetic: teacher-t2i`` with the\nexact models, revisions, licences and prompt. The v5 model learns from the\nresulting glyph grids only.\n"""\nimport numpy as np\n\nGENERATOR = {"model": "stabilityai/sd-turbo", "decoder": "madebyollin/taesd",\n             "license": "Stability AI non-commercial research community license (sd-turbo); MIT (taesd)",\n             "steps": 2, "guidance": 0.0, "resolution": 512}\nSEGMENTER = {"model": "ZhengPeng7/BiRefNet_lite", "revision": "aa62cd87eafb9cc43056d08ef3615a14628b831d",\n             "license": "MIT", "resolution": 512, "threshold": .5}\nTEMPLATES = ("a toy figurine of a {word}, full body, standing on a white table, white background, product photo",\n             "a simple flat illustration of a {word}, full body, side view, white background")\n\n\nclass Teacher:\n    def __init__(self, device="cuda"):\n        import torch\n        from diffusers import AutoPipelineForText2Image, AutoencoderTiny\n        from huggingface_hub import model_info\n        from transformers import AutoModelForImageSegmentation\n        from .deform import install\n        install()\n        self.torch, self.device = torch, device\n        self.generator_revision = model_info(GENERATOR["model"]).sha\n        self.decoder_revision = model_info(GENERATOR["decoder"]).sha\n        pipe = AutoPipelineForText2Image.from_pretrained(GENERATOR["model"], revision=self.generator_revision,\n                                                         torch_dtype=torch.float16)\n        pipe.vae = AutoencoderTiny.from_pretrained(GENERATOR["decoder"], revision=self.decoder_revision,\n                                                   torch_dtype=torch.float16)\n        self.pipe = pipe.to(device)\n        self.pipe.set_progress_bar_config(disable=True)\n        self.segmenter = AutoModelForImageSegmentation.from_pretrained(\n            SEGMENTER["model"], revision=SEGMENTER["revision"], trust_remote_code=True).eval().to(device).half()\n        self.mean = torch.tensor([.485, .456, .406], device=device)[:, None, None]\n        self.std = torch.tensor([.229, .224, .225], device=device)[:, None, None]\n\n    def provenance(self):\n        return {"generator": GENERATOR | {"revision": self.generator_revision, "decoder_revision": self.decoder_revision},\n                "segmenter": SEGMENTER, "templates": list(TEMPLATES), "synthetic": "teacher-t2i"}\n\n    def draw(self, prompts, seeds):\n        torch = self.torch\n        generators = [torch.Generator(self.device).manual_seed(int(s)) for s in seeds]\n        images = self.pipe(list(prompts), num_inference_steps=GENERATOR["steps"], guidance_scale=GENERATOR["guidance"],\n                           generator=generators, height=GENERATOR["resolution"], width=GENERATOR["resolution"]).images\n        return [np.asarray(image.convert("RGB")) for image in images]\n\n    def masks(self, images):\n        torch = self.torch\n        with torch.inference_mode():\n            x = torch.stack([torch.from_numpy(i).to(self.device).permute(2, 0, 1).float()/255 for i in images])\n            x = torch.nn.functional.interpolate(x, size=(SEGMENTER["resolution"],)*2, mode="bilinear", align_corners=False)\n            probability = self.segmenter(((x-self.mean)/self.std).half())[-1].sigmoid().float()\n            probability = torch.nn.functional.interpolate(probability, size=images[0].shape[:2], mode="bilinear", align_corners=False)\n        return [p[0].cpu().numpy() > SEGMENTER["threshold"] for p in probability]\n\n\ndef check_mask(mask):\n    """Reason to reject a teacher cut-out, or None."""\n    import cv2\n    if mask.mean() < .02:\n        return "subject_too_small"\n    if mask[:3].any() or mask[-3:].any() or mask[:, :3].any() or mask[:, -3:].any():\n        return "subject_touches_border"\n    count, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)\n    areas = np.sort(stats[1:, cv2.CC_STAT_AREA])[::-1]\n    if count > 2 and areas[1] > .15*areas[0]:\n        return "several_subjects"\n    return None\n', 'ascii_flux/text.py': '"""Frozen text-only SigLIP 2 tower, pinned revision; no vision tower is loaded."""\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom .contract import TEXT_SPEC, canonical, normalize_text\nfrom .storage import atomic_file\n\n\nclass TextEncoder:\n    def __init__(self, device="cpu"):\n        import torch\n        from transformers import AutoConfig, AutoTokenizer, SiglipTextModel, Siglip2TextModel\n        self.device = device\n        self.tokenizer = AutoTokenizer.from_pretrained(TEXT_SPEC["model"], revision=TEXT_SPEC["revision"])\n        # The pinned SigLIP 2 base checkpoint declares the original Siglip\n        # architecture in its config. Select its actual text config instead of\n        # constructing a Siglip2 tower with a multimodal config (hidden_size is\n        # nested there). Neither branch instantiates a vision tower.\n        config = AutoConfig.from_pretrained(TEXT_SPEC["model"], revision=TEXT_SPEC["revision"])\n        text_config = config.text_config\n        towers = {"siglip_text_model": SiglipTextModel, "siglip2_text_model": Siglip2TextModel}\n        if text_config.model_type not in towers:\n            raise ValueError("Unexpected text architecture in pinned checkpoint")\n        tower = towers[text_config.model_type]\n        self.model, info = tower.from_pretrained(TEXT_SPEC["model"], revision=TEXT_SPEC["revision"],\n                                                config=text_config, output_loading_info=True)\n        if info.get("missing_keys") or info.get("mismatched_keys"):\n            raise ValueError("Frozen text tower has missing or incompatible weights")\n        self.model = self.model.to(device).eval()\n        self.model.requires_grad_(False)\n        self.torch = torch\n\n    def encode(self, texts):\n        texts = [normalize_text(t) for t in texts]\n        inputs = self.tokenizer(texts, padding="max_length", truncation=True,\n                                max_length=TEXT_SPEC["max_length"], return_tensors="pt").to(self.device)\n        with self.torch.inference_mode():\n            vectors = self.model(**inputs).pooler_output\n            return self.torch.nn.functional.normalize(vectors, dim=-1).cpu().numpy().astype(np.float32)\n\n    def encode_tokens(self, texts, count=16):\n        """Per-token final hidden states [B, count, 768] for cross-attention.\n\n        Same tokenizer settings as ``encode`` (padding to 64); the first\n        ``count`` positions are kept, padding included, identically at\n        training and inference.\n        """\n        texts = [normalize_text(t) for t in texts]\n        inputs = self.tokenizer(texts, padding="max_length", truncation=True,\n                                max_length=TEXT_SPEC["max_length"], return_tensors="pt").to(self.device)\n        with self.torch.inference_mode():\n            hidden = self.model(**inputs).last_hidden_state[:, :count]\n        return hidden.float().cpu().numpy().astype(np.float16)\n\n\nTOKEN_SPEC = TEXT_SPEC | {"embedding": "SiglipTextModel.last_hidden_state[:, :16]", "tokens": 16}\n\n\nPHRASE_VERSION = "label-phrases-1"\n\n\ndef label_phrases(label):\n    """Natural phrasings of one dataset label, generated by fixed generic rules.\n\n    Dataset names such as ``car_(automobile)`` are identifiers, not what a\n    person types. Training on several clean phrasings narrows the gap between\n    training labels and prompts. No synonym or translation table is involved;\n    the frozen encoder alone carries meaning to other words and languages.\n    """\n    label = normalize_text(label)\n    head, _, rest = label.partition("_(")\n    head = " ".join(head.replace("_", " ").split())\n    qualifier = " ".join(rest.rstrip(")").replace("_", " ").split())\n    if not head:\n        raise ValueError("Empty label")\n    phrases = [head, f"a {head}", f"{head} moving"]\n    if qualifier:\n        phrases.append(f"{head} ({qualifier})")\n    return list(dict.fromkeys(phrases))\n\n\ndef cache_phrase_embeddings(manifest_path, output, device="cpu"):\n    """Embed every phrasing of every label; the first phrasing is the canonical one."""\n    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))\n    labels = sorted({r["normalized_label"] for r in manifest["clips"]})\n    if not labels:\n        raise ValueError("Empty dataset")\n    phrases, owners = [], []\n    for label in labels:\n        for phrase in label_phrases(label):\n            phrases.append(phrase)\n            owners.append(label)\n    encoder = TextEncoder(device)\n    vectors = np.concatenate([encoder.encode(phrases[i:i+64]) for i in range(0, len(phrases), 64)])\n    if vectors.shape != (len(phrases), TEXT_SPEC["dimension"]) or not np.isfinite(vectors).all():\n        raise ValueError("Invalid phrase vectors")\n    metadata = {"text_spec": TEXT_SPEC, "dataset_fingerprint": manifest["fingerprint"], "synthetic_encoder": False,\n                "phrase_version": PHRASE_VERSION, "phrases": len(phrases), "labels": len(labels),\n                "loaded_text_tower": type(encoder.model).__name__, "vision_tower_loaded": False}\n    with atomic_file(output) as stream:\n        np.savez_compressed(stream, phrases=np.array(phrases), labels=np.array(owners), embeddings=vectors,\n                            metadata=np.array(canonical(metadata)))\n    return metadata\n\n\ndef load_phrase_embeddings(path, dataset_fingerprint):\n    with np.load(path, allow_pickle=False) as cache:\n        metadata = json.loads(str(cache["metadata"]))\n        if metadata["dataset_fingerprint"] != dataset_fingerprint or metadata["text_spec"] != TEXT_SPEC:\n            raise ValueError("Phrase cache belongs to another dataset or text encoder")\n        if metadata.get("synthetic_encoder") or metadata.get("phrase_version") != PHRASE_VERSION:\n            raise ValueError("Phrase cache is synthetic or from another phrasing version")\n        vectors = cache["embeddings"]\n        if not np.isfinite(vectors).all():\n            raise ValueError("Invalid phrase vectors")\n        result = {}\n        for label, vector in zip(cache["labels"].tolist(), vectors):\n            result.setdefault(label, []).append(vector)\n    return {label: np.stack(v) for label, v in result.items()}, metadata\n\n\ndef cache_embeddings(manifest_path, output, device="cpu", existing_cache=None):\n    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))\n    labels = sorted({r["normalized_label"] for r in manifest["clips"]})\n    if not labels:\n        raise ValueError("Empty dataset")\n    previous = {}\n    if existing_cache:\n        with np.load(existing_cache, allow_pickle=False) as cache:\n            old = json.loads(str(cache["metadata"]))\n            if old["text_spec"] != TEXT_SPEC or old.get("synthetic_encoder"):\n                raise ValueError("Only genuine embeddings from the same frozen text encoder can be reused")\n            previous = dict(zip(cache["labels"].tolist(), cache["embeddings"]))\n    missing = [label for label in labels if label not in previous]\n    encoder = TextEncoder(device) if missing else None\n    for first in range(0, len(missing), 32):\n        batch = missing[first:first+32]\n        previous.update(zip(batch, encoder.encode(batch)))\n    vectors = np.stack([previous[label] for label in labels])\n    if vectors.shape != (len(labels), TEXT_SPEC["dimension"]) or not np.isfinite(vectors).all():\n        raise ValueError("Invalid cached text vectors")\n    metadata = {"text_spec": TEXT_SPEC, "dataset_fingerprint": manifest["fingerprint"], "synthetic_encoder": False,\n                "loaded_text_tower": type(encoder.model).__name__ if encoder else "reused pinned checkpoint text embeddings",\n                "vision_tower_loaded": False}\n    with atomic_file(output) as stream:\n        np.savez_compressed(stream, labels=np.array(labels), embeddings=vectors, metadata=np.array(canonical(metadata)))\n    return metadata\n\n', 'ascii_flux/training.py': '"""Research baseline: emergence, dynamics, concept switch and long rollouts.\n\nValidation is token CE, never a claim of human recognition or semantic success.\n"""\nfrom dataclasses import asdict, dataclass\nfrom collections import OrderedDict, defaultdict\nfrom contextlib import nullcontext\nimport copy\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport random\nimport numpy as np\nimport torch\nfrom .contract import Contract, TEXT_SPEC, digest\nfrom .model import GlyphNCA, ModelConfig, token_loss\nfrom .storage import atomic_file, read_clip, write_json\n\n\n@dataclass(frozen=True)\nclass TrainConfig:\n    steps: int = 10000\n    batch_size: int = 2\n    lr: float = 0.0003\n    emergence_steps: int = 8\n    rollout_frames: int = 4\n    steps_per_frame: int = 2\n    transition_steps: int = 8\n    long_every: int = 100\n    long_steps: int = 512\n    long_grad_steps: int = 8\n    validate_every: int = 100\n    val_batches: int = 8\n    teacher_decay_steps: int = 5000\n    seed: int = 20261031\n    precision: str = "float32"\n    gradient_accumulation: int = 1\n    foreground_weight: float = 1.\n\n    def __post_init__(self):\n        values = asdict(self)\n        if any(v <= 0 for k, v in values.items() if k not in ("seed", "precision")):\n            raise ValueError("Training counts and learning rate must be positive")\n        if self.long_steps < 500 or self.long_grad_steps > self.long_steps:\n            raise ValueError("Long rollouts require >=500 steps and a bounded gradient tail")\n        if self.precision not in ("float32", "float16", "bfloat16"):\n            raise ValueError("Unknown training precision")\n\n\ndef load_manifest(path):\n    path = Path(path)\n    manifest = json.loads(path.read_text(encoding="utf-8"))\n    content = dict(manifest)\n    fingerprint = content.pop("fingerprint")\n    if digest(content) != fingerprint:\n        raise ValueError("Manifest fingerprint mismatch")\n    contract = Contract.from_dict(manifest["contract"])\n    if contract.hash != manifest["contract_hash"]:\n        raise ValueError("Representation fingerprint mismatch")\n    videos = {}\n    heldout = set(manifest["config"]["heldout_categories"])\n    for record in manifest["clips"]:\n        key = (record["source"], record["video_id"])\n        if key in videos and videos[key] != record["split"]:\n            raise ValueError("Source video leaks between partitions")\n        videos[key] = record["split"]\n        if record["category"] in heldout and record["split"] != "test_unseen":\n            raise ValueError("Reserved category leaks into development")\n    # Hash every shard once when opening a run; corrupted tokens must not train silently.\n    from .factory import file_sha256\n    for filename, expected in manifest["shards"].items():\n        shard = (path.parent / filename).resolve()\n        if not shard.is_relative_to(path.parent.resolve()) or file_sha256(shard) != expected:\n            raise ValueError(f"Invalid shard: {filename}")\n    return manifest, contract\n\n\nclass Corpus:\n    def __init__(self, manifest_path, embeddings_path, reviews_path=None, allow_unreviewed=False, allow_fixture=False):\n        self.root = Path(manifest_path).parent\n        self.manifest, self.contract = load_manifest(manifest_path)\n        self.allow_unreviewed, self.allow_fixture = allow_unreviewed, allow_fixture\n        if not self.manifest["config"]["quality"].get("calibrated") and not allow_unreviewed:\n            raise ValueError("Calibrate converter quality thresholds after D1 review, or explicitly use --allow-unreviewed for development")\n        decisions = {}\n        if reviews_path:\n            reviews = json.loads(Path(reviews_path).read_text(encoding="utf-8"))\n            if reviews["dataset_fingerprint"] != self.manifest["fingerprint"] or reviews["converter_hash"] != self.manifest["converter"]["code_sha256"]:\n                raise ValueError("Reviews belong to another dataset/converter")\n            if not reviews.get("reviewer", "").strip():\n                raise ValueError("Reviews need an identified reviewer")\n            decisions = reviews.get("decisions", {})\n            if reviews.get("scope") == "converter-sample":\n                if (reviews.get("converter_identity") != self.manifest["converter"] or\n                        reviews.get("quality_policy") != self.manifest["config"]["quality"] or\n                        not reviews.get("human_statement", "").strip() or not reviews.get("sample_dataset_fingerprint")):\n                    raise ValueError("Invalid converter sample review evidence")\n                sample_reviewed = True\n            else:\n                sample_reviewed = False\n        else:\n            sample_reviewed = False\n        self.review_scope = "converter-sample" if sample_reviewed else "per-clip" if reviews_path else "unreviewed"\n        with np.load(embeddings_path, allow_pickle=False) as cache:\n            metadata = json.loads(str(cache["metadata"]))\n            if metadata["dataset_fingerprint"] != self.manifest["fingerprint"]:\n                raise ValueError("Text cache belongs to another dataset")\n            if metadata.get("synthetic_encoder") and not allow_fixture:\n                raise ValueError("Synthetic text fixtures are not semantic embeddings")\n            if metadata["text_spec"] != TEXT_SPEC:\n                raise ValueError("Text encoder contract mismatch")\n            vectors = cache["embeddings"]\n            if vectors.shape != (len(cache["labels"]), TEXT_SPEC["dimension"]) or not np.isfinite(vectors).all():\n                raise ValueError("Invalid embedding cache")\n            self.embeddings = dict(zip(cache["labels"].tolist(), vectors))\n            self.embedding_metadata = metadata\n        self.groups = {name: [] for name in ("train", "val", "test_seen", "test_unseen")}\n        for record in self.manifest["clips"]:\n            decision = decisions.get(record["clip_id"], {}).get("decision", "pending")\n            if decision == "reject" or (decision != "accept" and not allow_unreviewed and not sample_reviewed):\n                continue\n            if record.get("synthetic") and not allow_fixture:\n                continue\n            if record["normalized_label"] not in self.embeddings:\n                raise ValueError("Missing label embedding")\n            self.groups[record["split"]].append(record)\n        if not self.groups["train"] or not self.groups["val"]:\n            raise ValueError("Need separate nonempty train and validation videos after review")\n        self._cache = OrderedDict()\n        self._train_labels = defaultdict(list)\n        for record in self.groups["train"]:\n            self._train_labels[record["normalized_label"]].append(record)\n\n    def read(self, record):\n        key = (record["shard"], record["clip_id"])\n        if key not in self._cache:\n            self._cache[key] = read_clip(self.root, record, self.contract)\n            if len(self._cache) > 64:\n                self._cache.popitem(last=False)\n        self._cache.move_to_end(key)\n        return self._cache[key]\n\n    def other_records(self, records, rng=None):\n        rng = rng or random\n        if len(self._train_labels) < 2:\n            return []\n        return [rng.choice(self._train_labels[rng.choice([label for label in self._train_labels\n                 if label != record["normalized_label"]])]) for record in records]\n\n    def batch(self, split, batch_size, device, rng=None):\n        rng = rng or random\n        records = [rng.choice(self.groups[split]) for _ in range(batch_size)]\n        clips = [self.read(r) for r in records]\n        glyphs = torch.as_tensor(np.stack([c[0] for c in clips]).astype(np.int64), device=device)\n        colors = torch.as_tensor(np.stack([c[1] for c in clips]).astype(np.int64), device=device)\n        text = torch.as_tensor(np.stack([self.embeddings[r["normalized_label"]] for r in records]), device=device)\n        return glyphs, colors, text, records\n\n\ndef rollout(model, state, text, steps):\n    for _ in range(steps):\n        state, glyph_logits, color_logits = model.step(state, text)\n    return state, glyph_logits, color_logits\n\n\ndef rng_state():\n    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),\n            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    np.random.set_state(state["numpy"])\n    torch.set_rng_state(state["torch"].cpu())\n    if state["cuda"] is not None and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])\n\n\ndef precision_context(config, device):\n    if config.precision == "float32":\n        return nullcontext()\n    if torch.device(device).type != "cuda":\n        raise ValueError("Mixed precision training requires a tested CUDA/ROCm GPU")\n    dtype = torch.float16 if config.precision == "float16" else torch.bfloat16\n    return torch.autocast("cuda", dtype=dtype)\n\n\n@torch.no_grad()\ndef validate(model, corpus, config, device):\n    # Preserve training RNG so validation cadence does not alter optimization.\n    saved = rng_state()\n    model.eval()\n    try:\n        random.seed(config.seed + 1)\n        torch.manual_seed(config.seed + 1)\n        values = []\n        for _ in range(config.val_batches):\n            glyphs, colors, text, _ = corpus.batch("val", config.batch_size, device)\n            state, gl, cl = rollout(model, model.initial_state(config.batch_size), text, config.emergence_steps)\n            values.append(float(token_loss(gl, cl, glyphs[:, 0], colors[:, 0], config.foreground_weight)))\n            for t in range(1, min(config.rollout_frames+1, corpus.contract.frames)):\n                state, gl, cl = rollout(model, state, text, config.steps_per_frame)\n                values.append(float(token_loss(gl, cl, glyphs[:, t], colors[:, t], config.foreground_weight)))\n        return float(np.mean(values))\n    finally:\n        restore_rng(saved)\n        model.train()\n\ndef training_losses(model, corpus, config, model_config, step, device):\n    glyphs, colors, text, records = corpus.batch("train", config.batch_size, device)\n    conditioned = text * (torch.rand(config.batch_size, 1, device=device) >= model_config.text_dropout)\n    t = random.randrange(corpus.contract.frames-config.rollout_frames)\n    state, gl, cl = rollout(model, model.initial_state(config.batch_size, noisy=bool(step % 2)), conditioned, config.emergence_steps)\n    emergence = token_loss(gl, cl, glyphs[:, t], colors[:, t], config.foreground_weight)\n    # Teacher forcing gradually gives way to recurrent predictions.\n    teacher_probability = max(.1, 1-step/config.teacher_decay_steps)\n    dynamics = emergence*0\n    for k in range(1, config.rollout_frames+1):\n        if random.random() < teacher_probability:\n            state = model.lift(glyphs[:, t+k-1], colors[:, t+k-1])\n        state, gl, cl = rollout(model, state, conditioned, config.steps_per_frame)\n        dynamics = dynamics + token_loss(gl, cl, glyphs[:, t+k], colors[:, t+k], config.foreground_weight) / config.rollout_frames\n    # Select a genuinely different label for B; skip if the corpus has one label.\n    chosen = corpus.other_records(records)\n    transition = emergence*0\n    if chosen:\n        clips = [corpus.read(r) for r in chosen]\n        b_glyphs = torch.as_tensor(np.stack([c[0][0] for c in clips]).astype(np.int64), device=device)\n        b_colors = torch.as_tensor(np.stack([c[1][0] for c in clips]).astype(np.int64), device=device)\n        b_text = torch.as_tensor(np.stack([corpus.embeddings[r["normalized_label"]] for r in chosen]), device=device)\n        b_text = b_text * (torch.rand(config.batch_size, 1, device=device) >= model_config.text_dropout)\n        _, gl, cl = rollout(model, state.detach(), b_text, config.transition_steps)\n        transition = token_loss(gl, cl, b_glyphs, b_colors, config.foreground_weight)\n    long_loss = emergence*0\n    if (step+1) % config.long_every == 0:\n        with torch.no_grad():\n            long_state, _, _ = rollout(model, model.initial_state(config.batch_size, noisy=True), conditioned,\n                                       config.long_steps-config.long_grad_steps)\n        _, gl, cl = rollout(model, long_state, conditioned, config.long_grad_steps)\n        long_loss = token_loss(gl, cl, glyphs[:, t], colors[:, t], config.foreground_weight)\n    loss = emergence + dynamics + transition + .25*long_loss\n    return loss, teacher_probability, {"emergence_ce": emergence, "dynamics_ce": dynamics, "transition_ce": transition, "long_ce": long_loss}\n\n\ndef train(manifest_path, embeddings_path, output, config=None, model_config=None, reviews_path=None,\n          device="cpu", resume=None, allow_unreviewed=False, allow_fixture=False):\n    config, model_config = config or TrainConfig(), model_config or ModelConfig()\n    corpus = Corpus(manifest_path, embeddings_path, reviews_path, allow_unreviewed, allow_fixture)\n    if config.rollout_frames >= corpus.contract.frames:\n        raise ValueError("rollout_frames must be shorter than a clip")\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    if (output/"latest.pt").exists() and not resume:\n        raise FileExistsError("Existing run: supply --resume or choose another output")\n    random.seed(config.seed)\n    np.random.seed(config.seed)\n    torch.manual_seed(config.seed)\n    model = GlyphNCA(corpus.contract, model_config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr)\n    precision_context(config, device)  # fail before starting an incompatible run\n    scaler = torch.amp.GradScaler("cuda", enabled=config.precision == "float16", init_scale=256.)\n    skipped_updates = 0\n    best, best_model, best_step, start = math.inf, None, 0, 0\n    reviews_hash = None\n    if reviews_path:\n        from .factory import file_sha256\n        reviews_hash = file_sha256(reviews_path)\n    provenance = {"dataset_fingerprint": corpus.manifest["fingerprint"], "text_spec": TEXT_SPEC,\n                  "reviews_sha256": reviews_hash, "review_scope": corpus.review_scope, "allow_unreviewed": allow_unreviewed,\n                  "allow_fixture": allow_fixture, "synthetic_encoder": corpus.embedding_metadata.get("synthetic_encoder", False),\n                  "recognition_validated": False, "training_device": str(device), "torch_version": str(torch.__version__)}\n    provenance["source_conditions"] = corpus.manifest["source"]\n    provenance["converter"] = corpus.manifest["converter"]\n    code_hash = hashlib.sha256()\n    for name in ("training.py", "model.py", "text.py", "contract.py"):\n        code_hash.update(name.encode())\n        code_hash.update(Path(__file__).with_name(name).read_bytes())\n    provenance["training_code_sha256"] = code_hash.hexdigest()\n    history = []\n    if resume:\n        # Full resume contains optimizer + Python/NumPy RNG; only load our trusted local checkpoints.\n        checkpoint = torch.load(resume, map_location="cpu", weights_only=False)\n        corpus.contract.require(checkpoint["contract"])\n        if checkpoint["model_config"] != asdict(model_config) or checkpoint["provenance"] != provenance:\n            raise ValueError("Resume provenance/model mismatch")\n        old_cfg, new_cfg = dict(checkpoint["train_config"]), asdict(config)\n        old_cfg.pop("steps"); new_cfg.pop("steps")\n        if old_cfg != new_cfg:\n            raise ValueError("Resume training configuration mismatch (only total steps may change)")\n        model.load_state_dict(checkpoint["model"])\n        optimizer.load_state_dict(checkpoint["optimizer"])\n        scaler.load_state_dict(checkpoint.get("scaler", {}))\n        skipped_updates = checkpoint.get("skipped_amp_updates", 0)\n        best, best_model, start = checkpoint["best_val"], checkpoint["best_model"], checkpoint["step"]\n        best_step = checkpoint.get("best_step", start)\n        if config.steps < start:\n            raise ValueError("Total steps cannot precede the saved training step")\n        history = checkpoint["history"]\n        restore_rng(checkpoint["rng"])\n    for step in range(start, config.steps):\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        metrics = defaultdict(float)\n        total_loss = 0.\n        for micro in range(config.gradient_accumulation):\n            with precision_context(config, device):\n                loss, teacher_probability, components = training_losses(model, corpus, config, model_config, step, device)\n                scaled_loss = loss / config.gradient_accumulation\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite loss; latest atomic checkpoint remains available")\n            scaler.scale(scaled_loss).backward()\n            total_loss += float(loss.detach()) / config.gradient_accumulation\n            for key, value in components.items():\n                metrics[key] += float(value.detach()) / config.gradient_accumulation\n        scaler.unscale_(optimizer)\n        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=not scaler.is_enabled())\n        if not torch.isfinite(norm):\n            # GradScaler detected these infinities during unscale, skips the\n            # optimizer update, and backs off its scale. Never count it as an\n            # optimizer update or poison the weights with infinite gradients.\n            skipped_updates += 1\n        scaler.step(optimizer)\n        scaler.update()\n        if (step+1) % config.validate_every == 0 or step+1 == config.steps:\n            with precision_context(config, device):\n                value = validate(model, corpus, config, device)\n            if not math.isfinite(value):\n                raise FloatingPointError("Non-finite validation; keep the preceding atomic checkpoint")\n            if value < best:\n                best, best_model = value, copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})\n                best_step = step+1\n            row = {"step": step+1, "train_ce": total_loss, "val_token_ce": value,\n                   "best_val_token_ce": best, "teacher_probability": teacher_probability,\n                   "optimizer_updates": step+1-skipped_updates, "skipped_amp_updates": skipped_updates, **metrics}\n            history.append(row)\n            checkpoint = {"format": "ascii-flux-v5-1", "contract": corpus.contract.to_dict(),\n                          "model_config": asdict(model_config), "train_config": asdict(config), "provenance": provenance,\n                          "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(), "best_model": best_model,\n                          "best_val": best, "best_step": best_step, "step": step+1, "rng": rng_state(), "history": history,\n                          "skipped_amp_updates": skipped_updates}\n            with atomic_file(output/"latest.pt") as stream:\n                torch.save(checkpoint, stream)\n            # Tensor-only inference artifact: safe weights_only loading, no optimizer/RNG pickle.\n            with atomic_file(output/"best.pt") as stream:\n                torch.save({k: checkpoint[k] for k in ("format", "contract", "model_config", "provenance", "step")} |\n                           {"model": best_model, "step": best_step}, stream)\n            write_json(output/"history.json", history)\n            print(json.dumps(row), flush=True)\n    return model\n\n\ndef load_inference(path, device="cpu", allow_development=False):\n    checkpoint = torch.load(path, map_location="cpu", weights_only=True)\n    if checkpoint["format"] != "ascii-flux-v5-1":\n        raise ValueError("Unknown checkpoint format")\n    provenance = checkpoint["provenance"]\n    if provenance["text_spec"] != TEXT_SPEC:\n        raise ValueError("Text representation mismatch")\n    if (provenance["allow_fixture"] or provenance["allow_unreviewed"]) and not allow_development:\n        raise ValueError("Development checkpoint: explicitly enable development mode")\n    model = GlyphNCA(Contract.from_dict(checkpoint["contract"]), ModelConfig(**checkpoint["model_config"])).to(device).eval()\n    model.load_state_dict(checkpoint["model"])\n    return model, provenance\n', 'ascii_flux/live.html': '<!doctype html>\n<html lang="fr">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>ASCII Flux</title>\n<style>\n  html, body { margin: 0; height: 100%; background: #000; overflow: hidden; }\n  canvas { position: fixed; inset: 0; width: 100vw; height: 100vh; display: block; }\n  input {\n    position: fixed; left: 50%; bottom: 4vh; transform: translateX(-50%);\n    width: min(560px, 80vw); padding: 10px 0; border: 0; border-bottom: 1px solid #333;\n    background: transparent; color: #ddd; outline: none; text-align: center;\n    font: 20px ui-monospace, Consolas, monospace; caret-color: #ddd;\n  }\n  input:focus { border-bottom-color: #666; }\n</style>\n</head>\n<body>\n<canvas id="c"></canvas>\n<input id="t" autocomplete="off" spellcheck="false" autofocus aria-label="Consigne">\n<script>\nconst canvas = document.getElementById(\'c\'), ctx = canvas.getContext(\'2d\'), input = document.getElementById(\'t\');\nlet cfg = null, frame = null, dirty = true;\n\nfunction resize() {\n  const dpr = window.devicePixelRatio || 1;\n  canvas.width = innerWidth * dpr; canvas.height = innerHeight * dpr;\n  ctx.setTransform(dpr, 0, 0, dpr, 0, 0); dirty = true;\n}\n\nfunction draw() {\n  requestAnimationFrame(draw);\n  if (!dirty || !cfg || !frame) return;\n  dirty = false;\n  const W = innerWidth, H = innerHeight * 0.88;\n  // Grid aspect from the contract: cell height = cell ratio x cell width.\n  const cw = Math.min(W / cfg.cols, H / (cfg.rows * cfg.cell)), ch = cw * cfg.cell;\n  const x0 = (W - cw * cfg.cols) / 2, y0 = (H - ch * cfg.rows) / 2;\n  ctx.fillStyle = \'#000\'; ctx.fillRect(0, 0, W, innerHeight);\n  ctx.font = `${Math.floor(ch * 0.92)}px Consolas, ui-monospace, monospace`;\n  ctx.textAlign = \'center\'; ctx.textBaseline = \'middle\';\n  ctx.shadowBlur = ch * 0.35;\n  for (let r = 0; r < cfg.rows; r++) for (let c = 0; c < cfg.cols; c++) {\n    const i = r * cfg.cols + c, g = frame.g[i];\n    if (g === \' \') continue;\n    const p = cfg.palette[frame.c[i]] || [200, 200, 200];\n    const color = `rgb(${p[0]},${p[1]},${p[2]})`;\n    ctx.fillStyle = color; ctx.shadowColor = color;\n    ctx.fillText(g, x0 + (c + 0.5) * cw, y0 + (r + 0.5) * ch);\n  }\n}\n\nfunction connect() {\n  const source = new EventSource(\'/stream\');\n  source.onmessage = e => {\n    const m = JSON.parse(e.data);\n    frame = { g: m.g, c: Uint8Array.from(atob(m.c), ch => ch.charCodeAt(0)) };\n    dirty = true;\n  };\n  source.onerror = () => { source.close(); setTimeout(connect, 1000); };\n}\n\ninput.addEventListener(\'input\', () => {\n  fetch(\'/prompt\', { method: \'POST\', headers: { \'Content-Type\': \'application/json\' },\n                     body: JSON.stringify({ prompt: input.value }) }).catch(() => {});\n});\ndocument.addEventListener(\'keydown\', () => input.focus());\n\nfetch(\'/config\').then(r => r.json()).then(c => { cfg = c; resize(); connect(); draw(); });\naddEventListener(\'resize\', resize);\n</script>\n</body>\n</html>\n', 'ascii_flux/player.html': '<!doctype html>\n<html lang="fr">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n  <meta name="theme-color" content="#050607">\n  <title>ASCII Flux</title>\n  <link rel="icon" href="data:image/svg+xml,%3Csvg xmlns=\'http://www.w3.org/2000/svg\' viewBox=\'0 0 32 32\'%3E%3Crect width=\'32\' height=\'32\' rx=\'8\' fill=\'%23050607\'/%3E%3Ctext x=\'16\' y=\'24\' text-anchor=\'middle\' fill=\'%23a7e6cc\' font-family=\'monospace\' font-size=\'25\'%3E%40%3C/text%3E%3C/svg%3E">\n  <style>\n    :root { color-scheme: dark; font-family: system-ui, sans-serif; background: #000; color: #e8edea; }\n    * { box-sizing: border-box; }\n    body { margin: 0; height: 100svh; overflow: hidden; display: flex; flex-direction: column; }\n    header { position: absolute; inset: 25px 30px auto; z-index: 1; display: flex; align-items: center; gap: 12px; pointer-events: none; }\n    .brand { font-size: 12px; font-weight: 600; letter-spacing: .18em; color: #b0b9b4; }\n    .badge { padding: 4px 7px; border: 1px solid #242b27; border-radius: 5px; color: #6c7870; font-size: 10px; }\n    main { flex: 1; min-height: 0; position: relative; }\n    canvas { display: block; width: 100%; height: 100%; }\n    footer { padding: 8px 24px max(24px, env(safe-area-inset-bottom)); }\n    .bar { display: flex; align-items: center; gap: 12px; max-width: 880px; margin: auto; padding: 9px 10px 9px 19px; border: 1px solid #2a332e; border-radius: 17px; background: #101411; box-shadow: 0 9px 40px #0006; transition: border-color .15s; }\n    .bar:focus-within { border-color: #5e8170; }\n    .prefix { font: 20px ui-monospace, monospace; color: #6ba98a; user-select: none; }\n    input { flex: 1; min-width: 0; height: 38px; padding: 0; border: 0; outline: 0; background: transparent; color: #edf5ef; font: 16px system-ui, sans-serif; }\n    input::placeholder { color: #737e77; }\n    button { flex-shrink: 0; width: 38px; height: 38px; display: grid; place-items: center; border: 0; border-radius: 10px; background: transparent; color: #7d8c82; cursor: pointer; }\n    button:hover { background: #202b24; color: #d0eadb; }\n    button:focus-visible { outline: 2px solid #7fb899; outline-offset: 2px; }\n    button svg { width: 18px; height: 18px; }\n    .state { max-width: 850px; min-height: 25px; margin: 9px auto 0; display: flex; align-items: center; gap: 7px; color: #717c75; font-size: 11px; }\n    .dot { width: 5px; height: 5px; border-radius: 50%; background: #61766a; }\n    body[data-state="live"] .dot { background: #8dbb9e; }\n    body[data-state="updating"] .dot { background: #d0bd86; }\n    body[data-state="offline"] .dot { background: #c98077; }\n    @media (max-width: 600px) { header { inset: 18px 20px auto; } footer { padding-left: 14px; padding-right: 14px; } .bar { padding-left: 14px; gap: 9px; } input { font-size: 16px; } }\n  </style>\n</head>\n<body data-state="connecting">\n  <header><span class="brand">ASCII FLUX</span><span class="badge">Expérimental</span></header>\n  <main><canvas id="grid" aria-label="Animation ASCII générée en direct"></canvas></main>\n  <footer>\n    <div class="bar">\n      <span class="prefix" aria-hidden="true">›</span>\n      <input id="prompt" type="text" aria-label="Décris ce que tu veux voir" placeholder="Décris ce que tu veux voir…" maxlength="2000" autocomplete="off" autocapitalize="off" spellcheck="false" autofocus>\n      <button id="reset" type="button" aria-label="Recommencer l’animation" title="Recommencer l’animation">\n        <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true"><path d="M4 10a8 8 0 1 1 .9 6M4 4v6h6"/></svg>\n      </button>\n    </div>\n    <div class="state" role="status" aria-live="polite"><span class="dot" aria-hidden="true"></span><span id="status">Connexion au modèle…</span></div>\n  </footer>\n  <script>\n    const canvas = document.getElementById(\'grid\');\n    const context = canvas.getContext(\'2d\');\n    const input = document.getElementById(\'prompt\');\n    const status = document.getElementById(\'status\');\n    const client = crypto.randomUUID();\n    let contract, palette, frame, seen = -1, sequence = 0, sentSequence = -1;\n    let sending = false, connected = false, dirty = true, width = 0, height = 0, ratio = 1;\n\n    function indicate(state, message) {\n      document.body.dataset.state = state;\n      if (status.textContent !== message) status.textContent = message;\n    }\n\n    async function request(path, body) {\n      const controller = new AbortController();\n      const timeout = setTimeout(() => controller.abort(), 4000);\n      try {\n        const response = await fetch(path, body === undefined\n          ? {cache: \'no-store\', signal: controller.signal}\n          : {method: \'POST\', headers: {\'Content-Type\': \'application/json\'}, body: JSON.stringify(body), signal: controller.signal});\n        const value = await response.json();\n        if (!response.ok) throw Error(value.error || \'Connexion impossible\');\n        return value;\n      } finally { clearTimeout(timeout); }\n    }\n\n    // Send immediately. Serialize requests, keeping the most recent keystroke\n    // if typing outpaces the network; never let an old request replace it.\n    async function sendInput() {\n      if (sending || !connected) return;\n      sending = true;\n      try {\n        while (sentSequence < sequence) {\n          const current = sequence;\n          await request(\'/prompt\', {prompt: input.value, client, sequence: current});\n          sentSequence = current;\n        }\n      } catch (_) {\n        connected = false;\n        indicate(\'offline\', \'Reconnexion au modèle…\');\n      } finally { sending = false; }\n    }\n\n    input.addEventListener(\'input\', () => {\n      sequence++;\n      indicate(\'updating\', \'Mise à jour…\');\n      sendInput();\n    });\n    document.getElementById(\'reset\').addEventListener(\'click\', async () => {\n      try { await request(\'/reset\', {}); }\n      catch (_) { indicate(\'offline\', \'Reconnexion au modèle…\'); }\n      input.focus();\n    });\n\n    function resize() {\n      const bounds = canvas.getBoundingClientRect();\n      width = bounds.width; height = bounds.height;\n      ratio = window.devicePixelRatio || 1;\n      canvas.width = Math.round(width * ratio);\n      canvas.height = Math.round(height * ratio);\n      context.setTransform(ratio, 0, 0, ratio, 0, 0);\n      dirty = true;\n    }\n    new ResizeObserver(resize).observe(canvas);\n    window.addEventListener(\'resize\', resize);\n\n    function paint() {\n      if (dirty) {\n        dirty = false;\n        context.fillStyle = \'#000\';\n        context.fillRect(0, 0, width, height);\n        if (frame && contract) {\n          const cell = Math.min((width - 32) / contract.cols, Math.max(1, height - 68) / (contract.rows * contract.cell_height_width));\n          const row = cell * contract.cell_height_width;\n          const left = (width - cell * contract.cols) / 2;\n          const top = (height - row * contract.rows) / 2 + 15;\n          context.font = `${row * .92}px "Cascadia Mono", Consolas, ui-monospace, monospace`;\n          context.textAlign = \'center\'; context.textBaseline = \'middle\';\n          for (let r = 0; r < contract.rows; r++) for (let c = 0; c < contract.cols; c++) {\n            const glyph = frame.glyphs[r][c];\n            if (glyph === 0) continue;\n            context.fillStyle = palette[frame.colors[r][c]];\n            context.fillText(contract.glyphs[glyph], left + (c + .5) * cell, top + (r + .5) * row, cell * .98);\n          }\n        }\n      }\n      requestAnimationFrame(paint);\n    }\n\n    async function frames() {\n      const started = performance.now();\n      try {\n        if (!connected) {\n          const info = await request(\'/config\');\n          contract = info.contract;\n          palette = contract.palette.map(rgb => `rgb(${rgb.join(\',\')})`);\n          connected = true; seen = -1;\n          sentSequence = -1;\n          sendInput();\n        }\n        const next = await request(\'/frame\');\n        if (next.steps !== seen) { frame = next; seen = next.steps; dirty = true; }\n        if (next.error) indicate(\'offline\', \'Le modèle a rencontré une erreur.\');\n        else if (next.input_client !== client || next.input_sequence !== sequence || next.applied_revision !== next.revision) {\n          indicate(\'updating\', \'Mise à jour…\');\n        } else indicate(\'live\', input.value ? \'En direct\' : \'Écris pour commencer\');\n      } catch (_) {\n        connected = false;\n        indicate(\'offline\', \'Reconnexion au modèle…\');\n      }\n      setTimeout(frames, connected ? Math.max(0, 1000 / 24 - (performance.now() - started)) : 750);\n    }\n    resize(); paint(); frames();\n  </script>\n</body>\n</html>\n', 'ascii_flux/review_player.html': '<!doctype html>\n<html lang="fr">\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>ASCII Flux v5 — revue D1</title>\n<style>\nbody{background:#101318;color:#eee;font:16px system-ui;margin:24px;max-width:1250px}\nbutton,input,select{font:inherit;padding:7px;margin:4px;background:#202936;color:#eee;border:1px solid #56647a;border-radius:4px}\ncanvas{background:black;max-width:100%;display:block;margin:12px 0}\npre{white-space:pre-wrap;overflow-wrap:anywhere;font-size:12px}.warning{color:#ffc66d}\n#status,#counts,#frame-position{color:#a9c9df}label{display:inline-block}details{margin-top:16px}\n</style>\n<h1>Revue humaine D1</h1>\n<p class="warning">Clips du convertisseur hors ligne. Ce lecteur ne démontre aucune génération par un modèle.</p>\n<label>Relecteur <input id="reviewer" aria-label="Relecteur"></label>\n<button id="prev">Précédent</button><button id="next">Suivant</button><button id="pause">Pause</button>\n<label>Clip <select id="clip-select" aria-label="Clip"></select></label>\n<p id="position"></p>\n<p id="status">Chargement… Si aucune animation n\'apparaît, ouvrir ce fichier HTML directement dans un navigateur.</p>\n<canvas id="grid" aria-label="Animation ASCII du clip sélectionné"></canvas><span id="frame-position"></span>\n<div>\n<label>Décision <select id="decision"><option value="pending">À revoir</option><option value="accept">Accepter</option><option value="reject">Rejeter</option></select></label>\n<label>Motif <input id="reason" size="42" placeholder="Lisibilité, scintillement, masque…"></label>\n</div>\n<p id="counts"></p>\n<button id="download">Exporter reviews.json</button><label>Reprendre une revue <input id="import" type="file" accept="application/json"></label>\n<p>Examiner la forme, le masque et le mouvement. Exporter les décisions avant de quitter le lecteur.\nLa calibration des seuils suit cette revue ; aucune décision n\'est préremplie comme acceptée.</p>\n<details><summary>Métadonnées, provenance et métriques du clip</summary><pre id="meta"></pre></details>\n<noscript>Ce lecteur nécessite JavaScript. Ouvrir le fichier HTML dans un navigateur.</noscript>\n<script>\nconst data=__DATA__;\nconst $=id=>document.getElementById(id),ctx=$(\'grid\').getContext(\'2d\');\nconst decisions=Object.fromEntries(data.clips.map(c=>[c.id,{decision:\'pending\',reason:\'\'}]));\nconst storageKey=\'ascii-flux-review-\'+data.fingerprint;\nconst cells=data.contract.rows*data.contract.cols,tokenCount=data.contract.frames*cells;\nconst cw=10,ch=cw*data.contract.cell_height_width;\nlet index=0,frame=0,playing=true,currentTokens=null,loadVersion=0;\nconst cache=new Map();\n$(\'grid\').width=cw*data.contract.cols;$(\'grid\').height=ch*data.contract.rows;\n\nfunction reviewValue(){return {dataset_fingerprint:data.fingerprint,converter_hash:data.converter_hash,reviewer:$(\'reviewer\').value.trim(),decisions};}\nfunction restoreReview(value){\n  if(value.dataset_fingerprint!==data.fingerprint||value.converter_hash!==data.converter_hash)throw Error(\'Cette revue appartient à un autre corpus ou convertisseur.\');\n  for(const [key,v] of Object.entries(value.decisions||{})){\n    if(Object.hasOwn(decisions,key)&&v&&[\'pending\',\'accept\',\'reject\'].includes(v.decision)&&typeof v.reason===\'string\')decisions[key]={decision:v.decision,reason:v.reason};\n  }\n  $(\'reviewer\').value=typeof value.reviewer===\'string\'?value.reviewer:\'\';\n}\nfunction save(){\n  try{localStorage.setItem(storageKey,JSON.stringify(reviewValue()));}catch(error){/* Export remains available when browser storage is disabled. */}\n  const values=Object.values(decisions);\n  $(\'counts\').textContent=`${values.filter(v=>v.decision===\'accept\').length} acceptés · ${values.filter(v=>v.decision===\'reject\').length} rejetés · ${values.filter(v=>v.decision===\'pending\').length} à revoir`;\n}\nasync function unpackClip(clip){\n  if(data.codec!==\'gzip-u8-glyphs-colors-1\')throw Error(\'Format de lecteur inconnu.\');\n  if(typeof DecompressionStream===\'undefined\')throw Error(\'Ce navigateur ne prend pas en charge la décompression. Ouvrir ce fichier dans une version récente de Chrome, Edge ou Firefox.\');\n  const packed=Uint8Array.from(atob(clip.tokens_gzip),c=>c.charCodeAt(0));\n  const stream=new Blob([packed]).stream().pipeThrough(new DecompressionStream(\'gzip\'));\n  const tokens=new Uint8Array(await new Response(stream).arrayBuffer());\n  if(tokens.length!==tokenCount*2)throw Error(\'Longueur des jetons du clip incorrecte.\');\n  return tokens;\n}\nasync function show(){\n  const version=++loadVersion;currentTokens=null;\n  ctx.fillStyle=\'black\';ctx.fillRect(0,0,$(\'grid\').width,$(\'grid\').height);\n  if(!data.clips.length){$(\'status\').textContent=\'Aucun clip retenu par les contrôles automatiques.\';return;}\n  const clip=data.clips[index];\n  $(\'position\').textContent=`${index+1}/${data.clips.length} — ${clip.metadata.original_label||clip.metadata.normalized_label||clip.id}`;\n  $(\'clip-select\').value=String(index);\n  $(\'decision\').value=decisions[clip.id].decision;$(\'reason\').value=decisions[clip.id].reason;\n  $(\'meta\').textContent=JSON.stringify(clip.metadata,null,2);$(\'status\').textContent=\'Chargement du clip…\';\n  try{\n    const tokens=cache.get(clip.id)||await unpackClip(clip);\n    if(version!==loadVersion)return;\n    cache.set(clip.id,tokens);\n    while(cache.size>3)cache.delete(cache.keys().next().value);\n    currentTokens=tokens;$(\'status\').textContent=\'Prêt pour la revue. Les décisions restent en attente tant que tu ne les renseignes pas.\';draw();\n  }catch(error){if(version===loadVersion)$(\'status\').textContent=error.message;}\n}\nfunction draw(){\n  if(!currentTokens)return;\n  ctx.fillStyle=\'black\';ctx.fillRect(0,0,$(\'grid\').width,$(\'grid\').height);ctx.font=`${ch-2}px monospace`;ctx.textBaseline=\'top\';\n  const offset=frame*cells;\n  for(let r=0;r<data.contract.rows;r++)for(let c=0;c<data.contract.cols;c++){\n    const at=offset+r*data.contract.cols+c,glyph=currentTokens[at];if(glyph===0)continue;\n    const color=data.contract.palette[currentTokens[tokenCount+at]];\n    ctx.fillStyle=`rgb(${color.join(\',\')})`;ctx.fillText(data.contract.glyphs[glyph],c*cw,r*ch,cw);\n  }\n  $(\'frame-position\').textContent=`Image ${frame+1}/${data.contract.frames}`;\n}\nfunction selectClip(next){if(!data.clips.length)return;index=(next+data.clips.length)%data.clips.length;frame=0;show();}\n$(\'prev\').onclick=()=>selectClip(index-1);$(\'next\').onclick=()=>selectClip(index+1);\n$(\'clip-select\').onchange=()=>selectClip(Number($(\'clip-select\').value));\n$(\'pause\').onclick=()=>{playing=!playing;$(\'pause\').textContent=playing?\'Pause\':\'Lecture\';};\nfor(const id of [\'decision\',\'reason\'])$(id).oninput=()=>{\n  if(data.clips.length)decisions[data.clips[index].id]={decision:$(\'decision\').value,reason:$(\'reason\').value};save();\n};\n$(\'reviewer\').oninput=save;\n$(\'download\').onclick=()=>{\n  if(!$(\'reviewer\').value.trim()){alert(\'Renseigner le nom du relecteur avant l’export.\');return;}\n  const url=URL.createObjectURL(new Blob([JSON.stringify(reviewValue(),null,2)],{type:\'application/json\'}));\n  const link=document.createElement(\'a\');link.href=url;link.download=\'reviews.json\';link.click();\n  setTimeout(()=>URL.revokeObjectURL(url),1000);\n};\n$(\'import\').onchange=async event=>{\n  try{restoreReview(JSON.parse(await event.target.files[0].text()));save();show();}catch(error){alert(error.message);}\n};\nfor(let i=0;i<data.clips.length;i++){\n  const clip=data.clips[i],option=document.createElement(\'option\');option.value=String(i);\n  option.textContent=`${i+1} — ${clip.metadata.original_label||clip.metadata.normalized_label||clip.id}`;$(\'clip-select\').appendChild(option);\n}\ntry{const saved=localStorage.getItem(storageKey);if(saved)restoreReview(JSON.parse(saved));}catch(error){/* Start with pending decisions. */}\nsave();show();\nsetInterval(()=>{if(playing&&currentTokens){frame=(frame+1)%data.contract.frames;draw();}},1000/data.contract.fps);\n</script>\n</html>\n', 'configs/d1.json': '{\n  "dataset_version": "ytvis-d1-ramp-1",\n  "max_clips": 200,\n  "shard_size": 25,\n  "seed": 20261031,\n  "heldout_categories": ["giraffe", "airplane"],\n  "converter": {\n    "margin": 0.15,\n    "smoothing": 0.25,\n    "gamma": 0.8,\n    "hysteresis": 0.7,\n    "saturation": 1.15,\n    "min_coverage": 0.12\n  },\n  "quality": {\n    "calibrated": false,\n    "min_occupancy": null,\n    "max_occupancy": null,\n    "min_subject_cells": null,\n    "max_compensated_change": null,\n    "min_mask_fraction": null,\n    "max_residual_camera_motion": null\n  }\n}\n\n', 'configs/d1_visual_review.json': '{\n  "reviewer": "Utilisateur (confirmation dans le chat du 2026-10-31)",\n  "human_statement": "Oui, globalement reconnaissables",\n  "sample_dataset_fingerprint": "b0789e95db9b6c560b26f0b68ec8057b03758097cc18f1d7e4e59f12ac861188",\n  "scope": "Global visual acceptance of the D1 converter sample, not acceptance of every clip or validation of trained model recognition",\n  "inspected_clip_count": null,\n  "initial_quality_policy": "configs/d2.json"\n}\n', 'configs/d2.json': '{\n  "dataset_version": "ytvis-d2-ramp-1",\n  "target_grids": 1000000,\n  "horizontal_flip": true,\n  "shard_size": 25,\n  "seed": 20261031,\n  "heldout_categories": ["giraffe", "airplane"],\n  "converter": {\n    "margin": 0.15, "smoothing": 0.25, "gamma": 0.8,\n    "hysteresis": 0.7, "saturation": 1.15, "min_coverage": 0.12\n  },\n  "quality": {\n    "calibrated": true,\n    "calibration_scope": "initial cutoffs from D1 distribution after user\'s global visual acceptance; not a per-clip human acceptance",\n    "calibration_sample_fingerprint": "b0789e95db9b6c560b26f0b68ec8057b03758097cc18f1d7e4e59f12ac861188",\n    "min_occupancy": 0.02, "max_occupancy": 0.65,\n    "min_subject_cells": 8, "max_compensated_change": 0.6,\n    "min_mask_fraction": 0.95, "max_residual_camera_motion": null\n  }\n}\n', 'configs/d2_visual_review.json': '{\n  "reviewer": "user",\n  "reviewed_on": "2026-10-31",\n  "scope": "eight horse frames from the D2 corpus, not a per-clip acceptance or a model recognition evaluation",\n  "human_statement": "Oui, la majorité est lisible",\n  "dataset_fingerprint": "66b3dd8ab55cc13f577a194c5837dfb1059df94a9a772f39eda6186bbb919c8c",\n  "gallery": "outputs/subject-audit-001/horse.png",\n  "frame_index": 12,\n  "gallery_clip_ids": [\n    "clip-e9132320cd3a11550419c19a9d4cd0b7", "clip-83f88c71f7ec83fbc69ed9ad0bcb446a",\n    "clip-3fc46f1249a5e6634e089a6abae01b6e", "clip-78b4fa0a98a961591cf4b155dc2ec72d",\n    "clip-5f80f053649c96259cf7b48b1482b9e7", "clip-7735cff29b9832369b14fbd592b157ee",\n    "clip-0dd7420ffdd07e53f88dab4da6b92716", "clip-9415841a948fb5062a9df55d6a59ef1f"\n  ],\n  "per_clip_decisions": {},\n  "model_recognition_validated": false\n}\n', 'configs/generation_visual_review.json': '{\n  "reviewer": "user",\n  "reviewed_on": "2026-10-31",\n  "scope": "qualitative recognition of six generated stills, labels visible; not a forced-choice study or an animation review",\n  "human_statement": "aucun",\n  "interpreted_result": "no subject recognized on the presented sheet",\n  "checkpoint": "outputs/discrete-six-best/checkpoint.pt",\n  "checkpoint_step": 1000,\n  "inference_snapshot_sha256": "ef5ee99851f803e769462257e5c817e3e00eca4dd55bc22de1d8606388a1016c",\n  "tokens_sha256": "950299e071fce2620487cb6931a702d22585537877c5243c06b51f8a9b8ca740",\n  "gallery_sha256": "6c679bba963b5bf934427a3d68d191b3a1fc6f3567f5f9968f7181368c97a55e",\n  "gallery": "outputs/discrete-six-best/concepts.png",\n  "source_run": "data/runs/discrete-six-001",\n  "seed": 42,\n  "sampling_step": 32,\n  "subjects": ["cat", "dog", "horse", "bird", "fish", "person"],\n  "recognition_validated": false,\n  "formal_accuracy": null\n}\n', 'configs/lvvis_public.json': '{\n  "annotations_url": "https://drive.google.com/file/d/1k-o8gBMD7m1-fghw-a1iNDZCi2ZZgV9g/view?usp=sharing",\n  "annotations_sha256": "17ff02672040ee83f17cb9f9e1c94707769ed6c07ea730421a1185202d7731d5",\n  "frames_url": "https://drive.google.com/file/d/1er2lBQLF75TI5O4wzGyur0YYoohMK6C3/view?usp=sharing",\n  "frames_sha256": "4134387d491df5c9b9637d2b42fa7240ab93a176d79d1f7d38ed803682878324",\n  "source_fps": 5.0,\n  "source": {\n    "source": "LV-VIS",\n    "snapshot": "official current training annotations 17ff02672040 + frames 4134387d491d, retrieved 2026-10-31",\n    "url": "https://github.com/haochenheheda/LVVIS",\n    "terms_url": "https://github.com/haochenheheda/LVVIS/blob/main/README.md",\n    "terms_checked_on": "2026-10-31",\n    "annotations_license": "CC-BY-NC-SA-4.0",\n    "media_conditions": "CC BY-NC-SA 4.0; released for non-commercial research only; retain original attribution and conditions",\n    "usage_scope": "non-commercial-research",\n    "cadence_evidence": {\n      "url": "https://arxiv.org/abs/2304.01715",\n      "section": "LV-VIS dataset collection: extract 5 frames per second from videos of 2 to 10 seconds",\n      "source_fps": 5.0\n    }\n  }\n}\n', 'configs/train.json': '{\n  "steps": 10000,\n  "batch_size": 2,\n  "lr": 0.0003,\n  "emergence_steps": 8,\n  "rollout_frames": 4,\n  "steps_per_frame": 2,\n  "transition_steps": 8,\n  "long_every": 100,\n  "long_steps": 512,\n  "long_grad_steps": 8,\n  "validate_every": 100,\n  "val_batches": 8,\n  "teacher_decay_steps": 5000,\n  "seed": 20261031\n}\n\n', 'configs/train_local_8gb.json': '{\n  "steps": 20000,\n  "batch_size": 8,\n  "gradient_accumulation": 1,\n  "precision": "float16",\n  "foreground_weight": 4.0,\n  "lr": 0.0003,\n  "emergence_steps": 8,\n  "rollout_frames": 4,\n  "steps_per_frame": 2,\n  "transition_steps": 8,\n  "long_every": 100,\n  "long_steps": 512,\n  "long_grad_steps": 8,\n  "validate_every": 100,\n  "val_batches": 8,\n  "teacher_decay_steps": 5000,\n  "seed": 20261031\n}\n', 'configs/ytvis2019_public.json': '{\n  "profile": "ytvis-2019-public-1",\n  "checked_on": "2026-10-31",\n  "annotations": {\n    "url": "https://www.kaggle.com/api/v1/datasets/download/farnoosharefi/youtube-vis-train2019?datasetVersionNumber=1",\n    "page": "https://www.kaggle.com/datasets/farnoosharefi/youtube-vis-train2019",\n    "version": 1,\n    "sha256": "78a0cc867a1861be800c80fc55ccacb9008ca77af2a38016250faefd40a6f2ef",\n    "bytes": 64197762,\n    "format": "meta+sequences; per-object COCO RLE masks; 2238 sequences",\n    "host_license_declaration": "Unknown; does not replace the original YouTube-VIS conditions"\n  },\n  "frames": {\n    "url": "https://huggingface.co/datasets/yinloonga/YouTube_VOS_2019/resolve/7a0eb80c729b0dbbceaea15950e86ed606ab21ed/train.tar?download=true",\n    "page": "https://huggingface.co/datasets/yinloonga/YouTube_VOS_2019",\n    "revision": "7a0eb80c729b0dbbceaea15950e86ed606ab21ed",\n    "sha256": "5a81246005cfc2245bcd1f7e8fff41868963dfc3873e92bb810ea5f35a7cf8c0",\n    "bytes": 9258444800,\n    "format": "uncompressed TAR; original YouTube-VOS source frames",\n    "host_license_declaration": "Not declared; original YouTube-VOS/VIS research conditions retained"\n  },\n  "source_fps": 6.0,\n  "frame_stride_at_30fps": 5,\n  "terms_url": "https://youtube-vos.org/dataset/term/",\n  "annotations_license": "CC-BY-4.0",\n  "usage_scope": "non-commercial-research",\n  "cadence_evidence": "https://arxiv.org/html/1905.04804v2#S4"\n}\n', 'configs/ytvis_source.json': '{\n  "source": "YouTube-VIS",\n  "synthetic": false,\n  "snapshot": "REPLACE_WITH_EXACT_DATASET_RELEASE",\n  "url": "https://youtube-vos.org/dataset/vis/",\n  "terms_url": "https://youtube-vos.org/dataset/term/",\n  "terms_checked_on": "2026-10-31",\n  "annotations_license": "CC-BY-4.0",\n  "media_conditions": "Non-commercial research only; videos retain their underlying rights. Consult the complete source terms.",\n  "usage_scope": "non-commercial-research",\n  "local_media_provenance": "REPLACE_WITH_DOWNLOAD_OR_ARCHIVE_ORIGIN"\n}\n'}
SOURCE_HASH = 'be82a396a207f4904b49b954ec930f283ea785aad9824eaf96664bb12cd38de5'
WORK = Path('/content/ascii_flux_v5') if Path('/content').exists() else Path.cwd()/'ascii_flux_v5_colab'
WORK.mkdir(parents=True, exist_ok=True)
for name, content in SOURCES.items():
    target = WORK/name
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix+'.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as stream:
        stream.write(content); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, target)
marker = WORK/'.dependencies-installed'
if not marker.exists() or marker.read_text() != SOURCE_HASH:
    already = [name for name in ('numpy','PIL','cv2','torch','transformers','huggingface_hub') if name in sys.modules]
    if already:
        raise RuntimeError('Redémarrer le noyau avant installation : dépendances déjà importées '+str(already))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(WORK)+'[train,vis,export]'])
    marker.write_text(SOURCE_HASH)
    print('Installation terminée. Redémarrage requis avant la cellule suivante.', flush=True)
    if 'google.colab' in sys.modules:
        os.kill(os.getpid(), signal.SIGKILL)
    raise SystemExit('Redémarrer le noyau, puis reprendre à la cellule suivante.')
print('Dépendances déjà installées ; aucune installation dans ce noyau.')


In [ ]:
# Après redémarrage : aucune installation dans les cellules suivantes.
import json, sys
from pathlib import Path
WORK = Path('/content/ascii_flux_v5') if Path('/content').exists() else Path.cwd()/'ascii_flux_v5_colab'
sys.path.insert(0, str(WORK))
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Périphérique :', DEVICE)
# Google Drive est utilisé seulement s'il est déjà monté.
from ascii_flux.acquisition import default_data_root
DATA_ROOT = default_data_root(WORK.parent)
ANNOTATIONS = None
FRAMES_ROOT = None
DATASET = DATA_ROOT/'v5/d1-public-001'
RUN = DATA_ROOT/'runs/v5-001'
DOWNLOAD_DATA = True  # False : préparer les chemins sans télécharger les archives.
print('Données :', DATA_ROOT)
print('Profil : YouTube-VIS 2019 TRAIN via miroirs publics Kaggle + Hugging Face, sans connexion CodaLab.')


## Télécharger YouTube-VIS avant D1

Le profil par défaut utilise deux **miroirs publics vérifiés le 31 octobre 2026** :

- [Kaggle : annotations YouTube-VIS train 2019, version 1](https://www.kaggle.com/datasets/farnoosharefi/youtube-vis-train2019)
  fournit les catégories et les masques COCO RLE, dans un format réorganisé.
- [Hugging Face : images source YouTube-VOS 2019](https://huggingface.co/datasets/yinloonga/YouTube_VOS_2019)
  fournit les images originales correspondantes. La révision et le SHA-256 du TAR
  sont épinglés. Seules les images des vidéos VIS admissibles sont extraites.

**Aucune connexion CodaLab, aucun token et aucune URL à renseigner.**
L'archive d'annotations fait environ 64 Mo et celle des images **9,26 Go** ; prévoir
aussi la place pour les images extraites. Les téléchargements terminés sont réutilisés.
Pour préparer seulement les chemins, régler `DOWNLOAD_DATA = False` dans la cellule précédente.

Le notebook adapte le format d'annotations sans inventer de masque ou de label.
Les vidéos dont les images annotées ont une cadence irrégulière et les vidéos trop
courtes sont exclues avec un motif dans `mirror-conversion-report.json`. La couverture
du miroir d'images est vérifiée et les absences sont également publiées.
Les annotations CC BY 4.0 et les [conditions originales de recherche non commerciale](https://youtube-vos.org/dataset/term/)
sont consignées avant l'acquisition. Un miroir n'accorde pas de nouveaux droits.
Le profil lit les images annotées à 6 Hz (30 Hz / intervalle 5,
[article §4](https://arxiv.org/html/1905.04804v2#S4)).

Si un lien renvoie une page de connexion, un quota Google Drive ou du HTML,
l'acquisition s'arrête avec un message ; ce contenu n'est jamais traité comme un dataset.


In [ ]:
from ascii_flux.acquisition import load_prepared
from ascii_flux.public_source import prepare_public_ytvis2019
PREPARED = None
try:
    PREPARED = load_prepared(DATA_ROOT, profile='ytvis-2019-public')
    if PREPARED is None:
        if globals().get('DOWNLOAD_DATA', True):
            profile = json.loads((WORK/'configs/ytvis2019_public.json').read_text(encoding='utf-8'))
            template = json.loads((WORK/'configs/ytvis_source.json').read_text(encoding='utf-8'))
            PREPARED = prepare_public_ytvis2019(DATA_ROOT, template, profile)
        else:
            print('Acquisition en attente : régler DOWNLOAD_DATA = True pour télécharger les miroirs publics.')
    if PREPARED is not None:
        ANNOTATIONS = Path(PREPARED['annotations'])
        FRAMES_ROOT = Path(PREPARED['frames_root'])
        SOURCE_FPS = PREPARED['source_fps']
        print('Annotations :', ANNOTATIONS)
        print('Images :', FRAMES_ROOT)
        print('Cadence des images annotées :', SOURCE_FPS, 'Hz ; origine et hash enregistrés.')
except (ValueError, OSError, ImportError) as error:
    print('Acquisition non terminée :', error)
    print('Vérifier le message et la connexion réseau, puis relancer cette cellule. D1 reste en attente.')


## D1 — convertir des clips réels

La configuration produit au plus 200 candidats. Elle laisse les seuils non calibrés.
La rampe initiale est ` .:+*#@`, la grille 64×32, la palette 64 couleurs, la
cadence 12 Hz et la longueur 48 images. Vérifier les noms des catégories réservées
dans le snapshot choisi. Les vidéos trop courtes sont rejetées ; les répétitions
dues au rééchantillonnage restent tracées.
La conversion CPU peut prendre plusieurs minutes. Un message revient toutes les
20 secondes et le compteur avance à chaque lot sauvegardé (25 clips par défaut).
Une cellule occupée sans exception n'est pas nécessairement bloquée.


In [ ]:
from ascii_flux.factory import build_dataset, file_sha256
manifest = None
if globals().get('PREPARED') is None:
    print('D1 en attente : exécuter d’abord la cellule « Télécharger YouTube-VIS avant D1 ».')
else:
    source = PREPARED['source']
    config = json.loads((WORK/'configs/d1.json').read_text(encoding='utf-8'))
    try:
        if (DATASET/'manifest.json').exists():
            from ascii_flux.training import load_manifest
            from ascii_flux.offline import ConverterConfig, converter_identity
            previous, _ = load_manifest(DATASET/'manifest.json')
            if (previous['annotation_sha256'] != file_sha256(ANNOTATIONS) or
                previous['source'] != source or previous['config'] != config or
                previous['converter'] != converter_identity(ConverterConfig(**config['converter']))):
                raise ValueError('Ce run correspond à une autre source/configuration. Choisir un nouveau chemin DATASET dans la cellule des chemins.')
            manifest = previous
            print('Run D1 existant vérifié et réutilisé :', DATASET)
        else:
            from ascii_flux.progress import dataset_progress
            with dataset_progress(DATASET, config['max_clips']):
                manifest = build_dataset(ANNOTATIONS, FRAMES_ROOT, source, config, DATASET, SOURCE_FPS)
        summary = {key: value for key, value in manifest['report'].items() if key != 'source'}
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print('Provenance complète conservée dans :', DATASET/'manifest.json')
    except (ValueError, OSError) as error:
        print('D1 non lancé :', error)


In [ ]:
# Cette cellule peut être relancée seule : elle recharge le corpus depuis le disque.
import importlib, sys
from pathlib import Path
import ascii_flux.review as review_tools
from ascii_flux.acquisition import default_data_root
from IPython.display import FileLink, display
review_tools = importlib.reload(review_tools)
if 'DATASET' not in globals():
    DATASET = default_data_root(Path.cwd())/'v5/d1-public-001'
if not (Path(DATASET)/'manifest.json').is_file():
    print('Aucun corpus terminé trouvé dans', DATASET)
    print('Terminer D1 ou corriger DATASET. La revue lit le disque et ne dépend pas d’une variable manifest en mémoire.')
else:
    DATASET = Path(DATASET)
    manifest = review_tools.load_review_manifest(DATASET)
    REVIEW_HTML = DATASET.parent/(DATASET.name+'-review.html')
    review_tools.review_page(DATASET/'manifest.json', REVIEW_HTML)
    print(f"Lecteur prêt : {len(manifest['clips'])} clips. Taille : {REVIEW_HTML.stat().st_size/1e6:.2f} Mo.")
    print('Ouvrir ce fichier HTML directement dans un navigateur :', REVIEW_HTML.resolve())
    try:
        link_path = REVIEW_HTML.resolve().relative_to(Path.cwd().resolve())
    except ValueError:
        link_path = REVIEW_HTML.resolve()
    display(FileLink(str(link_path)))
    if 'google.colab' in sys.modules:
        from google.colab import files
        files.download(str(REVIEW_HTML))
    print('Le lecteur exporte reviews.json. Le conserver dans le répertoire du corpus correspondant.')


## Revue et calibration humaines

Observer forme, masque, couleurs et scintillement ; renseigner le relecteur et les
décisions, puis exporter `reviews.json`. Fixer les seuils de qualité à partir de
cette revue, consigner la méthode et le compromis fidélité/scintillement. Créer une
nouvelle version de configuration et un nouveau dataset, puis revoir les candidats.
Ne pas modifier le manifest d'un corpus existant.

Pour les cellules suivantes, `TRAIN_DATASET` doit désigner ce nouveau corpus.
L'entraînement refuse les seuils non calibrés ou l'absence de train/val acceptés.
Le corpus D1 n'établit pas à lui seul un corpus d'entraînement suffisant.


In [ ]:
TRAIN_DATASET = DATA_ROOT/'v5/d1-public-002'
REVIEWS = TRAIN_DATASET/'reviews.json'
from ascii_flux.text import cache_embeddings
EMBEDDINGS_READY = False
if not (TRAIN_DATASET/'manifest.json').is_file() or not REVIEWS.is_file():
    print('F1 en attente : terminer D1, la calibration et la revue du corpus TRAIN_DATASET avant l’encodage.')
else:
    training_manifest = json.loads((TRAIN_DATASET/'manifest.json').read_text(encoding='utf-8'))
    if not training_manifest['config']['quality'].get('calibrated'):
        print('F1 en attente : les seuils du corpus choisi doivent être calibrés après revue D1.')
    else:
        cache_embeddings(TRAIN_DATASET/'manifest.json', TRAIN_DATASET/'text.npz', DEVICE)
        EMBEDDINGS_READY = True


In [ ]:
from ascii_flux.training import TrainConfig, train
settings = TrainConfig(**json.loads((WORK/'configs/train.json').read_text()))
# Pour reprendre : RESUME = RUN/'latest.pt'. Même configuration, corpus et revue.
RESUME = None
if not globals().get('EMBEDDINGS_READY'):
    print('Entraînement en attente : préparer le corpus validé et ses embeddings dans la cellule précédente.')
else:
    model = train(TRAIN_DATASET/'manifest.json', TRAIN_DATASET/'text.npz', RUN,
                  config=settings, reviews_path=REVIEWS, device=DEVICE, resume=RESUME)


In [ ]:
from ascii_flux.training import load_inference
from ascii_flux.text import TextEncoder
from ascii_flux.evaluation import measure_rollout, export_onnx
from ascii_flux.storage import write_json
if not (RUN/'best.pt').is_file():
    print('Mesures en attente : aucun checkpoint entraîné dans', RUN)
else:
    model, provenance = load_inference(RUN/'best.pt', DEVICE)
    encoder = TextEncoder(DEVICE)
    vectors = encoder.encode(['cat', 'dog'])
    diagnostics = measure_rollout(model, vectors[0], steps=600, switch_embedding=vectors[1])
    diagnostics['checkpoint_provenance'] = provenance
    write_json(RUN/'rollout.json', diagnostics)
    print(json.dumps(diagnostics, ensure_ascii=False, indent=2))
    export_onnx(model, RUN/'step.onnx')


## Lecture des résultats et suite

La validation utilise l'entropie croisée sur jetons, pas une reconnaissance humaine.
Un état fini et figé est un échec de dynamique. Les diagnostics ne mesurent pas le
temps d'apparition d'une forme lisible, la transition sémantique, les langues ou
la stabilité visuelle sur dix minutes. Organiser ces évaluations sur vidéos
réservées et concepts réservés, puis comparer les autres architectures.

L'export ONNX est contrôlé numériquement ; il ne prouve pas les performances dans
un navigateur. Le prototype local `python -m ascii_flux serve --checkpoint …/best.pt`
utilise Python pour le calcul et un canvas pour afficher les jetons. Le portage
WebGPU/WASM, le corpus réel validé et tous les jalons restent à réaliser.
